In [ ]:
# --- Path fix (added 2026-09-29 repo reorganization) ---
# This notebook was originally built assuming its working directory was the
# repo root. It now lives under 01_data_engineering/notebooks/, and every raw
# file it reads (railways-master/, processed_data/, Railway_Network_.*) still
# lives one level up, in 01_data_engineering/. Run this cell first so every
# relative path below resolves exactly as it did before the move.
import os
from pathlib import Path

_target = Path.cwd().parent  # .../01_data_engineering
if _target.name == "01_data_engineering":
    os.chdir(_target)
print("Working directory:", Path.cwd())


In [1]:
import pandas as pd
import numpy as np
import json
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Environment ready.")

Environment ready.


In [2]:
STATIONS_PATH = "railways-master/stations.json"

with open(STATIONS_PATH, "r", encoding="utf-8") as f:
    stations_raw = json.load(f)

print("Object type:", type(stations_raw))
print("Top-level keys:", stations_raw.keys())
print("Number of features:", len(stations_raw["features"]))

Object type: <class 'dict'>
Top-level keys: dict_keys(['type', 'features'])
Number of features: 8990


In [3]:
# Inspect the first station feature

first_station = stations_raw["features"][0]

print("Feature keys:")
print(first_station.keys())

print("\nGeometry:")
print(first_station["geometry"])

print("\nProperties:")
print(first_station["properties"])

Feature keys:
dict_keys(['geometry', 'type', 'properties'])

Geometry:
{'type': 'Point', 'coordinates': [75.4516454, 27.2520587]}

Properties:
{'state': 'Rajasthan', 'code': 'BDHL', 'name': 'Badhal', 'zone': 'NWR', 'address': 'Kishangarh Renwal, Rajasthan'}


In [4]:
# Inspect station property structure

property_keys = set()

for feature in stations_raw["features"]:
    property_keys.update(feature.get("properties", {}).keys())

print("Number of unique property fields:", len(property_keys))
print("\nProperty fields:")
for key in sorted(property_keys):
    print("-", key)

Number of unique property fields: 5

Property fields:
- address
- code
- name
- state
- zone


In [5]:
# Check for missing or malformed geometry

geometry_issues = []

for i, feature in enumerate(stations_raw["features"]):
    geometry = feature.get("geometry")

    if geometry is None:
        geometry_issues.append({
            "index": i,
            "issue": "missing_geometry",
            "geometry": geometry
        })
    elif not isinstance(geometry, dict):
        geometry_issues.append({
            "index": i,
            "issue": "invalid_geometry_type",
            "geometry": geometry
        })
    elif geometry.get("type") != "Point":
        geometry_issues.append({
            "index": i,
            "issue": "not_a_point",
            "geometry": geometry
        })
    elif not isinstance(geometry.get("coordinates"), list):
        geometry_issues.append({
            "index": i,
            "issue": "invalid_coordinates",
            "geometry": geometry
        })
    elif len(geometry.get("coordinates", [])) != 2:
        geometry_issues.append({
            "index": i,
            "issue": "incorrect_coordinate_length",
            "geometry": geometry
        })

print("Total geometry issues:", len(geometry_issues))

geometry_issues[:10]

Total geometry issues: 293


[{'index': 1, 'issue': 'missing_geometry', 'geometry': None},
 {'index': 2, 'issue': 'missing_geometry', 'geometry': None},
 {'index': 3, 'issue': 'missing_geometry', 'geometry': None},
 {'index': 25, 'issue': 'missing_geometry', 'geometry': None},
 {'index': 29, 'issue': 'missing_geometry', 'geometry': None},
 {'index': 30, 'issue': 'missing_geometry', 'geometry': None},
 {'index': 34, 'issue': 'missing_geometry', 'geometry': None},
 {'index': 47, 'issue': 'missing_geometry', 'geometry': None},
 {'index': 49, 'issue': 'missing_geometry', 'geometry': None},
 {'index': 55, 'issue': 'missing_geometry', 'geometry': None}]

In [6]:
# Inspect the station records associated with geometry problems

for issue in geometry_issues[:10]:
    feature = stations_raw["features"][issue["index"]]

    print("\nIndex:", issue["index"])
    print("Issue:", issue["issue"])
    print("Properties:", feature.get("properties"))
    print("Geometry:", feature.get("geometry"))


Index: 1
Issue: missing_geometry
Properties: {'state': None, 'code': 'XX-BECE', 'name': 'XX-BECE', 'zone': None, 'address': None}
Geometry: None

Index: 2
Issue: missing_geometry
Properties: {'state': None, 'code': 'XX-BSPY', 'name': 'XX-BSPY', 'zone': None, 'address': None}
Geometry: None

Index: 3
Issue: missing_geometry
Properties: {'state': None, 'code': 'YY-BPLC', 'name': 'YY-BPLC', 'zone': None, 'address': None}
Geometry: None

Index: 25
Issue: missing_geometry
Properties: {'state': None, 'code': 'BJUP', 'name': 'BJUP', 'zone': None, 'address': None}
Geometry: None

Index: 29
Issue: missing_geometry
Properties: {'state': None, 'code': 'CIKA', 'name': 'CIKA', 'zone': None, 'address': None}
Geometry: None

Index: 30
Issue: missing_geometry
Properties: {'state': None, 'code': 'XX-CPHO', 'name': 'XX-CPHO', 'zone': None, 'address': None}
Geometry: None

Index: 34
Issue: missing_geometry
Properties: {'state': None, 'code': 'LLHW', 'name': 'LLHW', 'zone': None, 'address': None}
Geometr

In [7]:
# Comprehensive station data-quality audit

audit = {
    "total_records": len(stations_raw["features"]),
    "missing_geometry": 0,
    "non_point_geometry": 0,
    "invalid_coordinates": 0,
    "missing_code": 0,
    "missing_name": 0,
    "missing_state": 0,
    "missing_zone": 0,
    "missing_address": 0,
}

for feature in stations_raw["features"]:

    props = feature.get("properties") or {}
    geometry = feature.get("geometry")

    # Geometry
    if geometry is None:
        audit["missing_geometry"] += 1
    elif geometry.get("type") != "Point":
        audit["non_point_geometry"] += 1
    else:
        coords = geometry.get("coordinates")

        if (
            not isinstance(coords, list)
            or len(coords) != 2
            or any(c is None for c in coords)
        ):
            audit["invalid_coordinates"] += 1

    # Attributes
    if props.get("code") is None:
        audit["missing_code"] += 1

    if props.get("name") is None:
        audit["missing_name"] += 1

    if props.get("state") is None:
        audit["missing_state"] += 1

    if props.get("zone") is None:
        audit["missing_zone"] += 1

    if props.get("address") is None:
        audit["missing_address"] += 1


audit_df = pd.DataFrame(
    list(audit.items()),
    columns=["metric", "count"]
)

audit_df["percentage"] = (
    audit_df["count"] / audit["total_records"] * 100
)

audit_df

,metric,count,percentage
0,total_records,8990,100.000000
1,missing_geometry,293,3.259177
2,non_point_geometry,0,0.000000
3,invalid_coordinates,0,0.000000
4,missing_code,0,0.000000
5,missing_name,0,0.000000
6,missing_state,4532,50.411568
7,missing_zone,4532,50.411568
8,missing_address,4532,50.411568


In [8]:
# Step 4A — Safely construct the station DataFrame

station_records = []

for feature in stations_raw["features"]:

    properties = feature.get("properties") or {}
    geometry = feature.get("geometry")

    if geometry is not None:
        coordinates = geometry.get("coordinates")

        if isinstance(coordinates, list) and len(coordinates) == 2:
            longitude = coordinates[0]
            latitude = coordinates[1]
        else:
            longitude = None
            latitude = None
    else:
        longitude = None
        latitude = None

    station_records.append({
        "station_code": properties.get("code"),
        "station_name": properties.get("name"),
        "state": properties.get("state"),
        "zone": properties.get("zone"),
        "address": properties.get("address"),
        "longitude": longitude,
        "latitude": latitude
    })

stations_df = pd.DataFrame(station_records)

print("Station DataFrame created successfully.")
print("Shape:", stations_df.shape)

stations_df.head()

Station DataFrame created successfully.
Shape: (8990, 7)


,station_code,station_name,state,zone,address,longitude,latitude
0,BDHL,Badhal,Rajasthan,NWR,"Kishangarh Renwal, Rajasthan",75.451645,27.252059
1,XX-BECE,XX-BECE,None,None,None,NaN,NaN
2,XX-BSPY,XX-BSPY,None,None,None,NaN,NaN
3,YY-BPLC,YY-BPLC,None,None,None,NaN,NaN
4,KHH,KICHHA,Uttar Pradesh,NER,"Kichha, Uttar Pradesh",79.519746,28.913427


In [9]:
# Build station DataFrame while preserving missing geometry

station_records = []

for feature in stations_raw["features"]:

    properties = feature.get("properties") or {}
    geometry = feature.get("geometry") or {}

    coordinates = geometry.get("coordinates")

    if (
        isinstance(coordinates, list)
        and len(coordinates) == 2
    ):
        longitude = coordinates[0]
        latitude = coordinates[1]
    else:
        longitude = None
        latitude = None

    station_records.append({
        "station_code": properties.get("code"),
        "station_name": properties.get("name"),
        "state": properties.get("state"),
        "zone": properties.get("zone"),
        "address": properties.get("address"),
        "longitude": longitude,
        "latitude": latitude
    })

stations_df = pd.DataFrame(station_records)

print("Shape:", stations_df.shape)

Shape: (8990, 7)


In [10]:
# Step 4B — Determine overlap between missing geometry and missing attributes

stations_df["missing_geometry"] = (
    stations_df["longitude"].isna() |
    stations_df["latitude"].isna()
)

stations_df["missing_metadata"] = (
    stations_df["state"].isna() &
    stations_df["zone"].isna() &
    stations_df["address"].isna()
)

print("Total records:", len(stations_df))

print(
    "Missing geometry:",
    stations_df["missing_geometry"].sum()
)

print(
    "Missing metadata:",
    stations_df["missing_metadata"].sum()
)

print(
    "Both missing geometry AND metadata:",
    (
        stations_df["missing_geometry"] &
        stations_df["missing_metadata"]
    ).sum()
)

print(
    "Geometry missing but metadata available:",
    (
        stations_df["missing_geometry"] &
        ~stations_df["missing_metadata"]
    ).sum()
)

print(
    "Geometry available but metadata missing:",
    (
        ~stations_df["missing_geometry"] &
        stations_df["missing_metadata"]
    ).sum()
)

Total records: 8990
Missing geometry: 293
Missing metadata: 4532
Both missing geometry AND metadata: 105
Geometry missing but metadata available: 188
Geometry available but metadata missing: 4427


In [11]:
# Inspect records with missing geometry but available metadata

geometry_missing_metadata_available = stations_df[
    stations_df["missing_geometry"] &
    ~stations_df["missing_metadata"]
]

print(
    "Count:",
    len(geometry_missing_metadata_available)
)

geometry_missing_metadata_available.head(20)

Count: 188


,station_code,station_name,state,zone,address,longitude,latitude,missing_geometry,missing_metadata
337,AAV,Ambav,Gujarat,WR,"Ambav, Gujarat",NaN,NaN,True,False
981,HLG,Hillgrove,Tamil Nadu,SR,"Nilgiris, Tamil Nadu",NaN,NaN,True,False
1412,DNGD,Dungarda,Gujarat,WR,"Gujarat, Gujarat",NaN,NaN,True,False
1619,ARVI,Arvi,Maharashtra,CR,"Dist. Wardha, Maharashtra",NaN,NaN,True,False
1620,ARX,Areli,Uttar Pradesh,NER,"Areli, Uttar Pradesh",NaN,NaN,True,False
1621,ASM,Asarma,Gujarat,WR,"SH 166, Gujarat",NaN,NaN,True,False
1622,ATB,Alattambadi,Tamil Nadu,SR,"Mazavarayanallu Rd, Tamil Nadu",NaN,NaN,True,False
1623,ATDA,Atladara,Gujarat,WR,"Gujarat, Gujarat",NaN,NaN,True,False
1624,ATX,Alinagar Tola,Bihar,ECR,"Purnea, Bihar",NaN,NaN,True,False
1626,AYI,Ayingudi,Tamil Nadu,SR,"Peravurani - Aranthangi Rd, Tamil Nadu",NaN,NaN,True,False


In [12]:
# Check duplicate station codes

duplicate_codes = stations_df[
    stations_df["station_code"].duplicated(keep=False)
].sort_values("station_code")

print("Duplicate-code records:", len(duplicate_codes))
print("Unique duplicated codes:", duplicate_codes["station_code"].nunique())

duplicate_codes.head(30)

Duplicate-code records: 0
Unique duplicated codes: 0


,station_code,station_name,state,zone,address,longitude,latitude,missing_geometry,missing_metadata


In [13]:
# Step 5 — Create station data-quality flags

stations_df["has_geometry"] = (
    stations_df["longitude"].notna() &
    stations_df["latitude"].notna()
)

stations_df["has_metadata"] = (
    stations_df["state"].notna() &
    stations_df["zone"].notna() &
    stations_df["address"].notna()
)

stations_df["spatially_usable"] = stations_df["has_geometry"]

stations_df["metadata_complete"] = stations_df["has_metadata"]

stations_df["data_quality_status"] = np.select(
    [
        stations_df["has_geometry"] & stations_df["has_metadata"],
        stations_df["has_geometry"] & ~stations_df["has_metadata"],
        ~stations_df["has_geometry"] & stations_df["has_metadata"],
        ~stations_df["has_geometry"] & ~stations_df["has_metadata"]
    ],
    [
        "complete",
        "missing_metadata",
        "missing_geometry",
        "missing_geometry_and_metadata"
    ],
    default="unknown"
)

stations_df["data_quality_status"].value_counts()

data_quality_status
missing_metadata                 4427
complete                         4270
missing_geometry                  188
missing_geometry_and_metadata     105
Name: count, dtype: int64

In [14]:
# Percentage distribution

quality_summary = (
    stations_df["data_quality_status"]
    .value_counts()
    .rename_axis("data_quality_status")
    .reset_index(name="record_count")
)

quality_summary["percentage"] = (
    quality_summary["record_count"]
    / len(stations_df)
    * 100
)

quality_summary

,data_quality_status,record_count,percentage
0,missing_metadata,4427,49.243604
1,complete,4270,47.497219
2,missing_geometry,188,2.091212
3,missing_geometry_and_metadata,105,1.167964


In [15]:
# Step 6 — Coordinate range validation

spatial_stations = stations_df[stations_df["has_geometry"]].copy()

print("Spatially usable stations:", len(spatial_stations))

print("\nLongitude range:")
print(
    spatial_stations["longitude"].min(),
    "to",
    spatial_stations["longitude"].max()
)

print("\nLatitude range:")
print(
    spatial_stations["latitude"].min(),
    "to",
    spatial_stations["latitude"].max()
)

Spatially usable stations: 8697

Longitude range:
68.9678414502 to 95.725374

Latitude range:
8.088028 to 33.738534244499846


In [16]:
# Check for impossible geographic coordinates

invalid_longitude = spatial_stations[
    (spatial_stations["longitude"] < -180) |
    (spatial_stations["longitude"] > 180)
]

invalid_latitude = spatial_stations[
    (spatial_stations["latitude"] < -90) |
    (spatial_stations["latitude"] > 90)
]

print("Invalid longitude records:", len(invalid_longitude))
print("Invalid latitude records:", len(invalid_latitude))

Invalid longitude records: 0
Invalid latitude records: 0


In [17]:
# Broad geographic plausibility check for India

outside_india_bounds = spatial_stations[
    (spatial_stations["longitude"] < 68) |
    (spatial_stations["longitude"] > 98) |
    (spatial_stations["latitude"] < 6) |
    (spatial_stations["latitude"] > 38)
]

print(
    "Stations outside broad India bounds:",
    len(outside_india_bounds)
)

outside_india_bounds[
    [
        "station_code",
        "station_name",
        "state",
        "zone",
        "longitude",
        "latitude"
    ]
].head(30)

Stations outside broad India bounds: 0


,station_code,station_name,state,zone,longitude,latitude


In [18]:
# Step 7 — Exact coordinate duplicate analysis

coordinate_duplicates = (
    spatial_stations[
        spatial_stations.duplicated(
            subset=["longitude", "latitude"],
            keep=False
        )
    ]
    .sort_values(["longitude", "latitude"])
)

print(
    "Records sharing an exact coordinate:",
    len(coordinate_duplicates)
)

print(
    "Unique duplicated coordinate pairs:",
    coordinate_duplicates[
        ["longitude", "latitude"]
    ].drop_duplicates().shape[0]
)

coordinate_duplicates[
    [
        "station_code",
        "station_name",
        "state",
        "zone",
        "longitude",
        "latitude"
    ]
].head(30)

Records sharing an exact coordinate: 11
Unique duplicated coordinate pairs: 4


,station_code,station_name,state,zone,longitude,latitude
4643,BWC,Banwali,Rajasthan,NWR,74.029894,29.919979
4719,FSW,Fateh Singhwala,Rajasthan,NWR,74.029894,29.919979
1273,RRP,Rairangpur,Orissa,SER,86.164207,22.440956
3291,OND,Aunlajori Junction,Orissa,SER,86.164207,22.440956
958,ABW,Abutara Halt,West Bengal,NFR,89.586875,26.070776
2264,DHH,Dinhata,West Bengal,NFR,89.586875,26.070776
4061,FLM,Falimari,West Bengal,NFR,89.586875,26.070776
5026,NGTG,New Gitaldaha,West Bengal,NFR,89.586875,26.070776
4425,VTG,Bhetaguri,West Bengal,NFR,89.665310,26.302968
4667,COB,Cooch Behar,West Bengal,NFR,89.665310,26.302968


In [19]:
# Coordinate uniqueness

total_spatial = len(spatial_stations)

unique_coordinates = spatial_stations[
    ["longitude", "latitude"]
].drop_duplicates().shape[0]

print("Spatial records:", total_spatial)
print("Unique coordinate pairs:", unique_coordinates)
print(
    "Duplicate coordinate records:",
    total_spatial - unique_coordinates
)

Spatial records: 8697
Unique coordinate pairs: 8690
Duplicate coordinate records: 7


In [20]:
# Step 8 — Station identifier quality

print("Station code data type:")
print(stations_df["station_code"].map(type).value_counts())

print("\nStation name data type:")
print(stations_df["station_name"].map(type).value_counts())

print("\nEmpty station codes:")
print((stations_df["station_code"].astype(str).str.strip() == "").sum())

print("\nEmpty station names:")
print((stations_df["station_name"].astype(str).str.strip() == "").sum())

print("\nCodes with leading/trailing whitespace:")
print(
    (
        stations_df["station_code"]
        != stations_df["station_code"].astype(str).str.strip()
    ).sum()
)

print("\nNames with leading/trailing whitespace:")
print(
    (
        stations_df["station_name"]
        != stations_df["station_name"].astype(str).str.strip()
    ).sum()
)

Station code data type:
station_code
<class 'str'>    8990
Name: count, dtype: int64

Station name data type:
station_name
<class 'str'>    8990
Name: count, dtype: int64

Empty station codes:
0

Empty station names:
1

Codes with leading/trailing whitespace:
0

Names with leading/trailing whitespace:
0


In [21]:
# Check station-code casing

code_case_summary = {
    "uppercase": stations_df["station_code"].astype(str).str.isupper().sum(),
    "lowercase": stations_df["station_code"].astype(str).str.islower().sum(),
    "mixed_or_other": (
        len(stations_df)
        - stations_df["station_code"].astype(str).str.isupper().sum()
        - stations_df["station_code"].astype(str).str.islower().sum()
    )
}

code_case_summary

{'uppercase': np.int64(8990),
 'lowercase': np.int64(0),
 'mixed_or_other': np.int64(0)}

In [22]:
# Identify the station with an empty name

empty_name_stations = stations_df[
    stations_df["station_name"].astype(str).str.strip() == ""
]

empty_name_stations[
    [
        "station_code",
        "station_name",
        "state",
        "zone",
        "address",
        "longitude",
        "latitude"
    ]
]

,station_code,station_name,state,zone,address,longitude,latitude
3235,P,,,?,,NaN,NaN


In [23]:
# Inspect the raw source record for the empty-name station

empty_indices = empty_name_stations.index.tolist()

for idx in empty_indices:
    print("Raw feature index:", idx)
    print(json.dumps(
        stations_raw["features"][idx],
        indent=2
    ))

Raw feature index: 3235
{
  "geometry": null,
  "type": "Feature",
  "properties": {
    "state": "",
    "code": "P",
    "name": "",
    "zone": "?",
    "address": ""
  }
}


In [24]:
# Step 10 — Finalize cleaned station dataset

cleaned_stations = stations_df.copy()

# Explicit missing-value flags
cleaned_stations["missing_geometry"] = (
    cleaned_stations["longitude"].isna() |
    cleaned_stations["latitude"].isna()
)

cleaned_stations["missing_name"] = (
    cleaned_stations["station_name"].astype(str).str.strip() == ""
)

cleaned_stations["missing_metadata"] = (
    cleaned_stations["state"].isna() &
    cleaned_stations["zone"].isna() &
    cleaned_stations["address"].isna()
)

# Spatial usability
cleaned_stations["spatially_usable"] = (
    cleaned_stations["longitude"].notna() &
    cleaned_stations["latitude"].notna()
)

# Overall quality classification
cleaned_stations["data_quality_status"] = np.select(
    [
        (
            cleaned_stations["spatially_usable"] &
            ~cleaned_stations["missing_metadata"] &
            ~cleaned_stations["missing_name"]
        ),
        (
            cleaned_stations["spatially_usable"] &
            (
                cleaned_stations["missing_metadata"] |
                cleaned_stations["missing_name"]
            )
        ),
        (
            ~cleaned_stations["spatially_usable"] &
            ~cleaned_stations["missing_metadata"]
        ),
        (
            ~cleaned_stations["spatially_usable"] &
            cleaned_stations["missing_metadata"]
        )
    ],
    [
        "complete",
        "missing_metadata",
        "missing_geometry",
        "missing_geometry_and_metadata"
    ],
    default="review"
)

print("Final station dataset shape:", cleaned_stations.shape)

cleaned_stations.head()

Final station dataset shape: (8990, 15)


,station_code,station_name,state,zone,address,longitude,latitude,missing_geometry,missing_metadata,has_geometry,has_metadata,spatially_usable,metadata_complete,data_quality_status,missing_name
0,BDHL,Badhal,Rajasthan,NWR,"Kishangarh Renwal, Rajasthan",75.451645,27.252059,False,False,True,True,True,True,complete,False
1,XX-BECE,XX-BECE,None,None,None,NaN,NaN,True,True,False,False,False,False,missing_geometry_and_metadata,False
2,XX-BSPY,XX-BSPY,None,None,None,NaN,NaN,True,True,False,False,False,False,missing_geometry_and_metadata,False
3,YY-BPLC,YY-BPLC,None,None,None,NaN,NaN,True,True,False,False,False,False,missing_geometry_and_metadata,False
4,KHH,KICHHA,Uttar Pradesh,NER,"Kichha, Uttar Pradesh",79.519746,28.913427,False,False,True,True,True,True,complete,False


In [25]:
# Final validation

print("Total records:", len(cleaned_stations))
print("Unique station codes:", cleaned_stations["station_code"].nunique())
print("Spatially usable:", cleaned_stations["spatially_usable"].sum())
print("\nQuality distribution:")
print(cleaned_stations["data_quality_status"].value_counts())

Total records: 8990
Unique station codes: 8990
Spatially usable: 8697

Quality distribution:
data_quality_status
missing_metadata                 4427
complete                         4270
missing_geometry                  188
missing_geometry_and_metadata     105
Name: count, dtype: int64


In [26]:
# Step 11 — Final station handoff dataset

final_stations = cleaned_stations[
    [
        "station_code",
        "station_name",
        "state",
        "zone",
        "address",
        "longitude",
        "latitude",
        "spatially_usable",
        "missing_geometry",
        "missing_metadata",
        "missing_name",
        "data_quality_status"
    ]
].copy()

print("Final station columns:")
print(final_stations.columns.tolist())

print("\nShape:", final_stations.shape)

final_stations.head()

Final station columns:
['station_code', 'station_name', 'state', 'zone', 'address', 'longitude', 'latitude', 'spatially_usable', 'missing_geometry', 'missing_metadata', 'missing_name', 'data_quality_status']

Shape: (8990, 12)


,station_code,station_name,state,zone,address,longitude,latitude,spatially_usable,missing_geometry,missing_metadata,missing_name,data_quality_status
0,BDHL,Badhal,Rajasthan,NWR,"Kishangarh Renwal, Rajasthan",75.451645,27.252059,True,False,False,False,complete
1,XX-BECE,XX-BECE,None,None,None,NaN,NaN,False,True,True,False,missing_geometry_and_metadata
2,XX-BSPY,XX-BSPY,None,None,None,NaN,NaN,False,True,True,False,missing_geometry_and_metadata
3,YY-BPLC,YY-BPLC,None,None,None,NaN,NaN,False,True,True,False,missing_geometry_and_metadata
4,KHH,KICHHA,Uttar Pradesh,NER,"Kichha, Uttar Pradesh",79.519746,28.913427,True,False,False,False,complete


In [27]:
from pathlib import Path

# Create project output directory
OUTPUT_DIR = Path("processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "cleaned_stations.csv"

final_stations.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved:", OUTPUT_PATH.resolve())

Saved: C:\Users\vinja\Desktop\Rail-Intell\processed_data\cleaned_stations.csv


In [28]:
# Verify exported dataset

verification = pd.read_csv(OUTPUT_PATH)

print("Reloaded shape:", verification.shape)
print("Unique station codes:", verification["station_code"].nunique())
print("Spatially usable:", verification["spatially_usable"].sum())

Reloaded shape: (8990, 12)
Unique station codes: 8990
Spatially usable: 8697


In [29]:
from pathlib import Path

# Search from the current project directory
current_dir = Path.cwd()

print("Current notebook directory:")
print(current_dir)

print("\nSearching for trains.json...")

matches = list(current_dir.rglob("trains.json"))

for path in matches:
    print(path.resolve())

Current notebook directory:
C:\Users\vinja\Desktop\Rail-Intell

Searching for trains.json...
C:\Users\vinja\Desktop\Rail-Intell\railways-master\trains.json


In [30]:
from pathlib import Path

# Path resolved relative to the notebook working directory
TRAINS_PATH = Path.cwd() / "railways-master" / "trains.json"

print("Resolved TRAINS_PATH:", TRAINS_PATH.resolve())
assert TRAINS_PATH.exists(), f"trains.json not found at {TRAINS_PATH.resolve()}"

Resolved TRAINS_PATH: C:\Users\vinja\Desktop\Rail-Intell\railways-master\trains.json


In [31]:
with open(TRAINS_PATH, "r", encoding="utf-8") as f:
    trains_raw = json.load(f)

print("Object type:", type(trains_raw))
print("Top-level keys:", trains_raw.keys())
print("Number of features:", len(trains_raw["features"]))

Object type: <class 'dict'>
Top-level keys: dict_keys(['type', 'features'])
Number of features: 5208


In [32]:
# Inspect the first train feature

first_train = trains_raw["features"][0]

print("Feature keys:")
print(first_train.keys())

print("\nGeometry:")
print(first_train["geometry"])

print("\nProperties:")
print(first_train["properties"])

Feature keys:
dict_keys(['geometry', 'type', 'properties'])

Geometry:
{'type': 'LineString', 'coordinates': [[74.880117, 32.706975], [74.953339, 32.762368], [75.044078, 32.80617], [75.13222499999999, 32.769105], [75.14542599999999, 32.863528], [75.154881, 32.92664]]}

Properties:
{'third_ac': 0, 'arrival': '12:15:00', 'from_station_code': 'JAT', 'name': 'Jammu Tawi Udhampur Special', 'zone': 'NR', 'chair_car': 0, 'first_class': 0, 'duration_m': 35, 'sleeper': 0, 'from_station_name': 'JAMMU TAWI', 'number': '04601', 'departure': '10:40:00', 'return_train': '04602', 'to_station_code': 'UHP', 'second_ac': 0, 'classes': '', 'to_station_name': 'UDHAMPUR', 'duration_h': 1, 'type': 'DEMU', 'first_ac': 0, 'distance': 53}


In [33]:
# Inspect train property structure

train_property_keys = set()

for feature in trains_raw["features"]:
    train_property_keys.update(
        (feature.get("properties") or {}).keys()
    )

print("Number of unique property fields:", len(train_property_keys))

print("\nProperty fields:")
for key in sorted(train_property_keys):
    print("-", key)

Number of unique property fields: 21

Property fields:
- arrival
- chair_car
- classes
- departure
- distance
- duration_h
- duration_m
- first_ac
- first_class
- from_station_code
- from_station_name
- name
- number
- return_train
- second_ac
- sleeper
- third_ac
- to_station_code
- to_station_name
- type
- zone


In [34]:
# Convert train GeoJSON into a tabular dataset

train_records = []

for feature in trains_raw["features"]:

    properties = feature.get("properties") or {}
    geometry = feature.get("geometry") or {}

    train_records.append({
        "train_number": properties.get("number"),
        "train_name": properties.get("name"),
        "from_station_code": properties.get("from_station_code"),
        "from_station_name": properties.get("from_station_name"),
        "to_station_code": properties.get("to_station_code"),
        "to_station_name": properties.get("to_station_name"),
        "zone": properties.get("zone"),
        "type": properties.get("type"),
        "distance": properties.get("distance"),
        "duration_h": properties.get("duration_h"),
        "duration_m": properties.get("duration_m"),
        "arrival": properties.get("arrival"),
        "departure": properties.get("departure"),
        "return_train": properties.get("return_train"),
        "classes": properties.get("classes"),
        "first_ac": properties.get("first_ac"),
        "second_ac": properties.get("second_ac"),
        "third_ac": properties.get("third_ac"),
        "sleeper": properties.get("sleeper"),
        "first_class": properties.get("first_class"),
        "chair_car": properties.get("chair_car"),
        "geometry": geometry
    })

trains_df = pd.DataFrame(train_records)

print("Shape:", trains_df.shape)
print("\nColumns:")
print(trains_df.columns.tolist())

trains_df.head()

Shape: (5208, 22)

Columns:
['train_number', 'train_name', 'from_station_code', 'from_station_name', 'to_station_code', 'to_station_name', 'zone', 'type', 'distance', 'duration_h', 'duration_m', 'arrival', 'departure', 'return_train', 'classes', 'first_ac', 'second_ac', 'third_ac', 'sleeper', 'first_class', 'chair_car', 'geometry']


,train_number,train_name,from_station_code,from_station_name,to_station_code,to_station_name,zone,type,distance,duration_h,duration_m,arrival,departure,return_train,classes,first_ac,second_ac,third_ac,sleeper,first_class,chair_car,geometry
0,04601,Jammu Tawi Udhampur Special,JAT,JAMMU TAWI,UHP,UDHAMPUR,NR,DEMU,53.0,1.0,35.0,12:15:00,10:40:00,04602,,0,0,0,0,0,0,"{'type': 'LineString', 'coordinates': [[74.880..."
1,04602,UDHAMPUR JAMMUTAWI DMU,UHP,UDHAMPUR,JAT,JAMMU TAWI,NR,DEMU,53.0,1.0,50.0,08:35:00,06:45:00,04601,,0,0,0,0,0,0,"{'type': 'LineString', 'coordinates': [[75.154..."
2,04603,JAT UDAHMPUR DMU,JAT,JAMMU TAWI,UHP,UDHAMPUR,NR,DEMU,53.0,1.0,35.0,17:50:00,16:15:00,04604,,0,0,0,0,0,0,"{'type': 'LineString', 'coordinates': [[74.880..."
3,04604,UDHAMPUR JAMMUTAWI DMU,UHP,UDHAMPUR,JAT,JAMMU TAWI,NR,DEMU,53.0,1.0,30.0,19:50:00,18:20:00,04603,,0,0,0,0,0,0,"{'type': 'LineString', 'coordinates': [[75.154..."
4,04728,Mumbai BandraT-Bikaner SF Special,BDTS,MUMBAI BANDRA TERMINUS,BKN,BIKANER JN,NWR,SF,1212.0,21.0,55.0,12:30:00,14:35:00,04727,,0,1,1,1,0,0,"{'type': 'LineString', 'coordinates': [[72.840..."


In [35]:
# Basic train attribute completeness

train_missing = pd.DataFrame({
    "missing_count": trains_df.isna().sum(),
    "missing_percentage": trains_df.isna().mean() * 100
})

train_missing

,missing_count,missing_percentage
train_number,0,0.000000
train_name,0,0.000000
from_station_code,0,0.000000
from_station_name,0,0.000000
to_station_code,0,0.000000
to_station_name,0,0.000000
zone,0,0.000000
type,0,0.000000
distance,15,0.288018
duration_h,15,0.288018


In [36]:
# Step 14 — Train identifier quality

print("Total train records:", len(trains_df))

print(
    "Unique train numbers:",
    trains_df["train_number"].nunique()
)

print(
    "Duplicate train numbers:",
    trains_df["train_number"].duplicated().sum()
)

print(
    "\nTrain number data types:"
)

print(
    trains_df["train_number"]
    .map(type)
    .value_counts()
)

Total train records: 5208
Unique train numbers: 5208
Duplicate train numbers: 0

Train number data types:
train_number
<class 'str'>    5208
Name: count, dtype: int64


In [37]:
# Identify the 15 records with missing operational metrics

missing_operational = trains_df[
    trains_df[
        ["distance", "duration_h", "duration_m", "classes"]
    ].isna().any(axis=1)
].copy()

print("Records with missing operational metrics:", len(missing_operational))

missing_operational[
    [
        "train_number",
        "train_name",
        "from_station_code",
        "to_station_code",
        "zone",
        "type",
        "distance",
        "duration_h",
        "duration_m",
        "classes"
    ]
]

Records with missing operational metrics: 15


,train_number,train_name,from_station_code,to_station_code,zone,type,distance,duration_h,duration_m,classes
94,24041,DELHI SARAI ROHILLA - KOTDWAR Mussoorie Exp,NBD,KTW,,,NaN,NaN,NaN,None
4723,12374,RAMPURHAT - SEALDAH Intercity SF Exp,RPH,SDAH,,,NaN,NaN,NaN,None
4724,12373,SEALDAH - RAMPURHAT Intercity SF Exp,SDAH,RPH,,,NaN,NaN,NaN,None
4725,13258,ANAND VIHAR DELHI - DANAPUR JanSadharan Exp,ANVT,DNR,,,NaN,NaN,NaN,None
4726,13257,DANAPUR - DELHI ANAND VIHAR Jan Sadharan Express,DNR,ANVT,,,NaN,NaN,NaN,None
4727,57326,GUNTUR - KACHEGUDA PASS,GNT,KCG,,,NaN,NaN,NaN,None
4728,57325,KACHEGUDA - GUNTUR PASS,KCG,GNT,,,NaN,NaN,NaN,None
4731,12163,DADAR - CHENNAI EGMORE SF Exp,DR,MS,,,NaN,NaN,NaN,None
4732,12164,DADAR - CHENNAI EGMORE SF Exp,MS,DR,,,NaN,NaN,NaN,None
4742,12735,SECUNDERABAD - YESVANTPUR Garib Rath Exp,SC,YPR,,,NaN,NaN,NaN,None


In [38]:
# Missingness pattern for operational fields

operational_fields = [
    "distance",
    "duration_h",
    "duration_m",
    "classes"
]

missing_pattern = (
    trains_df[operational_fields]
    .isna()
    .astype(int)
    .astype(str)
    .agg("".join, axis=1)
    .value_counts()
)

missing_pattern

0000    5193
1111      15
Name: count, dtype: int64

In [39]:
# Step 15 — Train geometry validation

geometry_stats = {
    "missing_geometry": 0,
    "non_linestring": 0,
    "missing_coordinates": 0,
    "too_few_coordinates": 0
}

for feature in trains_raw["features"]:

    geometry = feature.get("geometry")

    if geometry is None:
        geometry_stats["missing_geometry"] += 1
        continue

    if geometry.get("type") != "LineString":
        geometry_stats["non_linestring"] += 1
        continue

    coordinates = geometry.get("coordinates")

    if not isinstance(coordinates, list):
        geometry_stats["missing_coordinates"] += 1
        continue

    if len(coordinates) < 2:
        geometry_stats["too_few_coordinates"] += 1

print(geometry_stats)

{'missing_geometry': 0, 'non_linestring': 0, 'missing_coordinates': 0, 'too_few_coordinates': 9}


In [40]:
# Number of route coordinates per train

trains_df["coordinate_count"] = trains_df["geometry"].apply(
    lambda g: len(g.get("coordinates", []))
    if isinstance(g, dict)
    else 0
)

print(trains_df["coordinate_count"].describe())

print(
    "\nTrains with fewer than 2 route coordinates:",
    (trains_df["coordinate_count"] < 2).sum()
)

print(
    "Trains with zero route coordinates:",
    (trains_df["coordinate_count"] == 0).sum()
)

count    5208.000000
mean       79.573733
std       100.055483
min         1.000000
25%        16.000000
50%        34.000000
75%       100.000000
max       684.000000
Name: coordinate_count, dtype: float64

Trains with fewer than 2 route coordinates: 9
Trains with zero route coordinates: 0


In [41]:
# Identify trains with fewer than 2 route coordinates

degenerate_trains = trains_df[
    trains_df["coordinate_count"] < 2
].copy()

degenerate_trains[
    [
        "train_number",
        "train_name",
        "from_station_code",
        "from_station_name",
        "to_station_code",
        "to_station_name",
        "zone",
        "type",
        "distance",
        "coordinate_count"
    ]
]

,train_number,train_name,from_station_code,from_station_name,to_station_code,to_station_name,zone,type,distance,coordinate_count
2713,52021,Moti Koral Miyagam Mix Passenger,MKRL,Moti Koral,MYG,MIYAGAM KARJAN,WR,Pass,28.0,1
2714,52022,Miyagam Moti Koral Mix Passenger,MYG,MIYAGAM KARJAN,MKRL,Moti Koral,WR,Pass,28.0,1
2764,52143,Pulgaon Arvi Passenger,PLO,PULGAON JN,ARVI,Arvi,CR,Pass,39.0,1
2765,52144,Arvi Pulgaon Passenger,ARVI,Arvi,PLO,PULGAON JN,CR,Pass,39.0,1
3208,53647,Dildarnagar Tarighat Passenger,DLN,DILDARNAGAR JN,TRG,Tarighat,NCR,Pass,19.0,1
3233,53648,Tarighat Dildarnagar Passenger,TRG,Tarighat,DLN,DILDARNAGAR JN,NCR,Pass,19.0,1
3600,53474,JAMALPUR MONGHYAR PASS,JMP,JAMALPUR JN,MGR,Munger,ER,Pass,9.0,1
4078,56890,Tiruturaipdi Tiruvarur Passenger,TTP,Tiruturaipudi Junction,TVR,THIRUVARUR JN,SR,Pass,26.0,1
5141,56889,Tiruvarur Tiruturaipudi Passenger,TVR,THIRUVARUR JN,TTP,Tiruturaipudi Junction,SR,Pass,26.0,1


In [42]:
# Inspect the raw geometry of degenerate routes

for _, row in degenerate_trains.iterrows():
    print(
        row["train_number"],
        "|",
        row["from_station_code"],
        "→",
        row["to_station_code"],
        "| coordinates:",
        row["geometry"].get("coordinates")
    )

52021 | MKRL → MYG | coordinates: [[73.120353, 22.050722]]
52022 | MYG → MKRL | coordinates: [[73.120353, 22.050722]]
52143 | PLO → ARVI | coordinates: [[78.318111, 20.726734]]
52144 | ARVI → PLO | coordinates: [[78.318111, 20.726734]]
53647 | DLN → TRG | coordinates: [[83.668324, 25.419424]]
53648 | TRG → DLN | coordinates: [[83.60210496000946, 25.571721452321867]]
53474 | JMP → MGR | coordinates: [[86.492285, 25.313906]]
56890 | TTP → TVR | coordinates: [[79.63389799999999, 10.764347]]
56889 | TVR → TTP | coordinates: [[79.63389799999999, 10.764347]]


In [43]:
# Step 17 — Train data-quality flags

trains_df["geometry_usable"] = (
    trains_df["coordinate_count"] >= 2
)

trains_df["missing_distance"] = (
    trains_df["distance"].isna()
)

trains_df["missing_duration"] = (
    trains_df["duration_h"].isna() |
    trains_df["duration_m"].isna()
)

trains_df["missing_classes"] = (
    trains_df["classes"].isna()
)

trains_df["operational_data_complete"] = (
    ~trains_df["missing_distance"] &
    ~trains_df["missing_duration"] &
    ~trains_df["missing_classes"]
)

trains_df["geometry_quality_status"] = np.where(
    trains_df["geometry_usable"],
    "usable",
    "insufficient_coordinates"
)

print(
    trains_df["geometry_quality_status"].value_counts()
)

print(
    "\nOperational completeness:"
)

print(
    trains_df["operational_data_complete"].value_counts()
)

geometry_quality_status
usable                      5199
insufficient_coordinates       9
Name: count, dtype: int64

Operational completeness:
operational_data_complete
True     5193
False      15
Name: count, dtype: int64


In [44]:
# Step 18 — Validate train endpoint station codes
# against the finalized station dataset

station_codes = set(
    final_stations["station_code"]
)

trains_df["from_station_exists"] = (
    trains_df["from_station_code"].isin(station_codes)
)

trains_df["to_station_exists"] = (
    trains_df["to_station_code"].isin(station_codes)
)

print(
    "Origin codes found in station dataset:",
    trains_df["from_station_exists"].sum(),
    "/",
    len(trains_df)
)

print(
    "Destination codes found in station dataset:",
    trains_df["to_station_exists"].sum(),
    "/",
    len(trains_df)
)

print(
    "\nOrigin codes missing:",
    (~trains_df["from_station_exists"]).sum()
)

print(
    "Destination codes missing:",
    (~trains_df["to_station_exists"]).sum()
)

Origin codes found in station dataset: 5208 / 5208
Destination codes found in station dataset: 5208 / 5208

Origin codes missing: 0
Destination codes missing: 0


In [45]:
# Step 19 — Validate train endpoint names against station master

station_lookup = (
    final_stations[
        ["station_code", "station_name"]
    ]
    .drop_duplicates("station_code")
    .set_index("station_code")["station_name"]
)

trains_df["station_master_from_name"] = (
    trains_df["from_station_code"].map(station_lookup)
)

trains_df["station_master_to_name"] = (
    trains_df["to_station_code"].map(station_lookup)
)

# Normalize only for comparison; do not modify source values
def normalize_name(value):
    if pd.isna(value):
        return ""
    
    return (
        str(value)
        .strip()
        .upper()
        .replace(".", "")
        .replace(",", "")
    )

trains_df["from_name_match"] = (
    trains_df["from_station_name"].map(normalize_name)
    ==
    trains_df["station_master_from_name"].map(normalize_name)
)

trains_df["to_name_match"] = (
    trains_df["to_station_name"].map(normalize_name)
    ==
    trains_df["station_master_to_name"].map(normalize_name)
)

print(
    "Origin name matches:",
    trains_df["from_name_match"].sum(),
    "/",
    len(trains_df)
)

print(
    "Destination name matches:",
    trains_df["to_name_match"].sum(),
    "/",
    len(trains_df)
)

Origin name matches: 5208 / 5208
Destination name matches: 5208 / 5208


In [46]:
# Inspect endpoint-name mismatches

name_mismatches = trains_df[
    ~trains_df["from_name_match"] |
    ~trains_df["to_name_match"]
]

print("Trains with at least one name mismatch:", len(name_mismatches))

name_mismatches[
    [
        "train_number",
        "from_station_code",
        "from_station_name",
        "station_master_from_name",
        "to_station_code",
        "to_station_name",
        "station_master_to_name"
    ]
].head(30)

Trains with at least one name mismatch: 0


,train_number,from_station_code,from_station_name,station_master_from_name,to_station_code,to_station_name,station_master_to_name


In [47]:
# Step 20 — Distance quality audit

distance_summary = trains_df["distance"].describe()

print(distance_summary)

count    5193.000000
mean      544.769305
std       696.791114
min         0.000000
25%        88.000000
50%       227.000000
75%       693.000000
max      4279.000000
Name: distance, dtype: float64


In [48]:
# Check for zero and negative distances

print(
    "Missing distances:",
    trains_df["distance"].isna().sum()
)

print(
    "Zero distances:",
    (trains_df["distance"] == 0).sum()
)

print(
    "Negative distances:",
    (trains_df["distance"] < 0).sum()
)

print(
    "Positive distances:",
    (trains_df["distance"] > 0).sum()
)

Missing distances: 15
Zero distances: 27
Negative distances: 0
Positive distances: 5166


In [49]:
# Inspect unusually small reported distances

trains_df[
    trains_df["distance"].notna()
].sort_values("distance")[
    [
        "train_number",
        "train_name",
        "from_station_code",
        "to_station_code",
        "distance",
        "type"
    ]
].head(20)

,train_number,train_name,from_station_code,to_station_code,distance,type
940,55104,Barhaj Bazar Bhatni Passenger,BHJ,BTT,0.0,Pass
2338,41064,VLCY MSB LOCAL,VLCY,MSB,0.0,Pass
2339,41066,Velachery Chennai Local,VLCY,MSB,0.0,Pass
2340,41112,Velachery Chennai Beach Passenger,VLCY,MSB,0.0,Pass
4958,55542,Phulwaria Hajipur Passenger,BHBR,HJP,0.0,Pass
3626,55146,Barhaj Bazar Salempur Passenger,BHJ,SRU,0.0,Pass
657,53446,Bhagalpur Mandar Hill Passenger,BGP,MDLE,0.0,Pass
3175,53442,Bhagalpur Mandar Hill Passenger,BGP,MDLE,0.0,Pass
3625,55145,Salempur Barhaj Passenger,SRU,BHJ,0.0,Pass
2711,52015,Bharuch Kavi Passenger,BH,KAVI,0.0,Pass


In [50]:
# Step 21 — Flag distance quality

trains_df["distance_quality_status"] = "valid"

trains_df.loc[
    trains_df["distance"].isna(),
    "distance_quality_status"
] = "missing"

trains_df.loc[
    trains_df["distance"] == 0,
    "distance_quality_status"
] = "zero"

trains_df.loc[
    trains_df["distance"] < 0,
    "distance_quality_status"
] = "negative"

print(trains_df["distance_quality_status"].value_counts())

distance_quality_status
valid      5166
zero         27
missing      15
Name: count, dtype: int64


In [51]:
# Inspect all zero-distance trains

zero_distance_trains = trains_df[
    trains_df["distance"] == 0
].sort_values("train_number")

zero_distance_trains[
    [
        "train_number",
        "train_name",
        "from_station_code",
        "from_station_name",
        "to_station_code",
        "to_station_name",
        "type",
        "zone",
        "distance",
        "geometry_usable"
    ]
]

,train_number,train_name,from_station_code,from_station_name,to_station_code,to_station_name,type,zone,distance,geometry_usable
2167,33512,Hasanabad Sealdah EMU,HNB,Hasanabad,SDAH,KOLKATA SEALDAH,MEMU,ER,0.0,True
2338,41064,VLCY MSB LOCAL,VLCY,Velachery,MSB,CHENNAI BEACH,Pass,?,0.0,True
2339,41066,Velachery Chennai Local,VLCY,Velachery,MSB,CHENNAI BEACH,Pass,?,0.0,True
2340,41112,Velachery Chennai Beach Passenger,VLCY,Velachery,MSB,CHENNAI BEACH,Pass,?,0.0,True
2711,52015,Bharuch Kavi Passenger,BH,BHARUCH JN,KAVI,Kavi,Pass,?,0.0,True
2885,52501,BhikhnaThori Narkatiaganj Passenger,BKF,Bhikhna Thori,NKE,NARKATIAGANJ JN,Pass,ECR,0.0,True
2886,52502,Narkatiaganj Bhikhna Thori Passenger,NKE,NARKATIAGANJ JN,BKF,Bhikhna Thori,Pass,ECR,0.0,True
2887,52503,BhikhnaThori Narkatiaganj Passenger,BKF,Bhikhna Thori,NKE,NARKATIAGANJ JN,Pass,ECR,0.0,True
2888,52504,Narkatiaganj Bhikhna Thori Passenger,NKE,NARKATIAGANJ JN,BKF,Bhikhna Thori,Pass,ECR,0.0,True
2889,52505,BhikhnaThori Narkatiaganj Passenger,BKF,Bhikhna Thori,NKE,NARKATIAGANJ JN,Pass,ECR,0.0,True


In [52]:
# Step 22 — Compare reported distance with geometry endpoint displacement

from math import radians, sin, cos, sqrt, atan2

def haversine_km(coord1, coord2):
    lon1, lat1 = coord1
    lon2, lat2 = coord2

    R = 6371.0

    dlon = radians(lon2 - lon1)
    dlat = radians(lat2 - lat1)

    a = (
        sin(dlat / 2) ** 2
        + cos(radians(lat1))
        * cos(radians(lat2))
        * sin(dlon / 2) ** 2
    )

    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

In [53]:
def endpoint_displacement(geometry):
    if not isinstance(geometry, dict):
        return None

    coords = geometry.get("coordinates")

    if not coords or len(coords) < 2:
        return None

    return haversine_km(coords[0], coords[-1])


trains_df["endpoint_displacement_km"] = trains_df["geometry"].apply(
    endpoint_displacement
)

trains_df[
    ["train_number", "distance", "endpoint_displacement_km"]
].head()

,train_number,distance,endpoint_displacement_km
0,04601,53.0,35.438523
1,04602,53.0,35.438523
2,04603,53.0,35.438523
3,04604,53.0,35.438523
4,04728,1212.0,996.620891


In [54]:
# Step 23 — Distance vs endpoint displacement sanity check

trains_df["distance_displacement_ratio"] = (
    trains_df["distance"] / trains_df["endpoint_displacement_km"]
)

comparison = trains_df[
    trains_df["distance"].notna()
    & trains_df["endpoint_displacement_km"].notna()
    & (trains_df["endpoint_displacement_km"] > 0)
].copy()

print("Comparable trains:", len(comparison))

comparison["distance_displacement_ratio"].describe()

Comparable trains: 5184


count    5184.000000
mean        1.359379
std         0.958591
min         0.000000
25%         1.115245
50%         1.241517
75%         1.412655
max        23.996409
Name: distance_displacement_ratio, dtype: float64

In [55]:
# Inspect unusually high route-distance / straight-line-distance ratios

comparison.sort_values(
    "distance_displacement_ratio",
    ascending=False
)[
    [
        "train_number",
        "train_name",
        "from_station_code",
        "to_station_code",
        "distance",
        "endpoint_displacement_km",
        "distance_displacement_ratio"
    ]
].head(20)

,train_number,train_name,from_station_code,to_station_code,distance,endpoint_displacement_km,distance_displacement_ratio
4715,64092,Nizamuddin Parikrama EMU,NZM,LPNR,32.0,1.333533,23.996409
4713,64090,Nizamuddin Parikrama EMU,NZM,LPNR,32.0,1.333533,23.996409
2750,52102,Matheran Neral Passenger,MAE,NRL,126.0,6.575960,19.160701
2751,52103,Neral Matheran Passenger,NRL,MAE,126.0,6.575960,19.160701
2757,52110,Matheran Neral Passenger,MAE,NRL,126.0,6.575960,19.160701
2756,52109,Neral Matheran Passenger,NRL,MAE,126.0,6.575960,19.160701
2754,52106,Matheran Neral Passenger,MAE,NRL,126.0,6.575960,19.160701
2753,52105,Neral Matheran Passenger,NRL,MAE,126.0,6.575960,19.160701
4935,06346,PALGHAT TOWN - POLLACHI Passenger SPL,PGTN,POY,638.0,40.501353,15.752560
4918,06345,,POY,PGTN,638.0,40.501353,15.752560


In [56]:
# Inspect cases where reported distance is close to endpoint displacement

comparison.sort_values(
    "distance_displacement_ratio"
)[
    [
        "train_number",
        "train_name",
        "from_station_code",
        "to_station_code",
        "distance",
        "endpoint_displacement_km",
        "distance_displacement_ratio"
    ]
].head(20)

,train_number,train_name,from_station_code,to_station_code,distance,endpoint_displacement_km,distance_displacement_ratio
892,55102,Barhaj Bazar Bhatni Passenger,BHJ,BTT,0.0,22.903479,0.0
893,55103,Bhatani Barhaj Passenger,BTT,BHJ,0.0,22.903479,0.0
4958,55542,Phulwaria Hajipur Passenger,BHBR,HJP,0.0,132.039622,0.0
2711,52015,Bharuch Kavi Passenger,BH,KAVI,0.0,34.389980,0.0
3590,55101,Bhatani Barhaj Passenger,BTT,BHJ,0.0,22.903479,0.0
657,53446,Bhagalpur Mandar Hill Passenger,BGP,MDLE,0.0,48.716812,0.0
3626,55146,Barhaj Bazar Salempur Passenger,BHJ,SRU,0.0,19.182935,0.0
3625,55145,Salempur Barhaj Passenger,SRU,BHJ,0.0,19.182935,0.0
4889,53441,Mandar Parvat Bhagalpur Passenger,MDLE,BGP,0.0,48.716812,0.0
940,55104,Barhaj Bazar Bhatni Passenger,BHJ,BTT,0.0,22.903479,0.0


In [57]:
# Step 24 — Final train dataset

final_train_columns = [
    "train_number",
    "train_name",
    "from_station_code",
    "from_station_name",
    "to_station_code",
    "to_station_name",
    "zone",
    "type",
    "distance",
    "duration_h",
    "duration_m",
    "arrival",
    "departure",
    "return_train",
    "classes",
    "first_ac",
    "second_ac",
    "third_ac",
    "sleeper",
    "first_class",
    "chair_car",
    "geometry",
    "coordinate_count",
    "geometry_usable",
    "geometry_quality_status",
    "missing_distance",
    "distance_quality_status",
    "missing_duration",
    "missing_classes",
    "operational_data_complete",
    "from_station_exists",
    "to_station_exists",
    "from_name_match",
    "to_name_match"
]

final_trains = trains_df[final_train_columns].copy()

print("Final train dataset shape:", final_trains.shape)
print("Unique train numbers:", final_trains["train_number"].nunique())
print("Duplicate train numbers:", final_trains["train_number"].duplicated().sum())

Final train dataset shape: (5208, 34)
Unique train numbers: 5208
Duplicate train numbers: 0


In [58]:
# Step 25 — Final train integrity check

print("=== FINAL TRAIN DATASET QA ===")

print("Rows:", len(final_trains))
print("Unique train numbers:", final_trains["train_number"].nunique())
print("Duplicate train numbers:",
      final_trains["train_number"].duplicated().sum())

print("\nGeometry status:")
print(final_trains["geometry_quality_status"].value_counts())

print("\nDistance status:")
print(final_trains["distance_quality_status"].value_counts())

print("\nOperational completeness:")
print(final_trains["operational_data_complete"].value_counts())

print("\nMissing origin station:",
      (~final_trains["from_station_exists"]).sum())

print("Missing destination station:",
      (~final_trains["to_station_exists"]).sum())

print("Origin name mismatches:",
      (~final_trains["from_name_match"]).sum())

print("Destination name mismatches:",
      (~final_trains["to_name_match"]).sum())

=== FINAL TRAIN DATASET QA ===
Rows: 5208
Unique train numbers: 5208
Duplicate train numbers: 0

Geometry status:
geometry_quality_status
usable                      5199
insufficient_coordinates       9
Name: count, dtype: int64

Distance status:
distance_quality_status
valid      5166
zero         27
missing      15
Name: count, dtype: int64

Operational completeness:
operational_data_complete
True     5193
False      15
Name: count, dtype: int64

Missing origin station: 0
Missing destination station: 0
Origin name mismatches: 0
Destination name mismatches: 0


In [59]:
# Step 26 — Export final train dataset

output_path = Path("processed_data") / "cleaned_trains.csv"

final_trains.to_csv(
    output_path,
    index=False
)

print("Exported:", output_path.resolve())
print("Shape:", final_trains.shape)

Exported: C:\Users\vinja\Desktop\Rail-Intell\processed_data\cleaned_trains.csv
Shape: (5208, 34)


In [60]:
# Step 27 — Verify exported train dataset

check_trains = pd.read_csv(output_path)

print("Reloaded shape:", check_trains.shape)
print("Unique train numbers:", check_trains["train_number"].nunique())
print("Duplicate train numbers:",
      check_trains["train_number"].duplicated().sum())

Reloaded shape: (5208, 34)
Unique train numbers: 5208
Duplicate train numbers: 0


In [61]:
from pathlib import Path

current_dir = Path.cwd()

matches = list(current_dir.rglob("schedules.json"))

print("Current working directory:")
print(current_dir.resolve())

print("\nMatches found:")
for path in matches:
    print(path.resolve())

Current working directory:
C:\Users\vinja\Desktop\Rail-Intell

Matches found:
C:\Users\vinja\Desktop\Rail-Intell\railways-master\schedules.json


In [62]:
schedule_path = matches[0]

with open(schedule_path, "r", encoding="utf-8") as f:
    schedules_raw = json.load(f)

print(type(schedules_raw))
print("Number of schedule records:", len(schedules_raw))
print(schedules_raw[0])

<class 'list'>
Number of schedule records: 417080
{'arrival': 'None', 'day': 1, 'train_name': 'Falaknuma Lingampalli MMTS', 'station_name': 'KACHEGUDA FALAKNUMA', 'station_code': 'FM', 'id': 302214, 'train_number': '47154', 'departure': '07:55:00'}


In [63]:
# Step 29 — Schedule DataFrame

schedules_df = pd.DataFrame(schedules_raw)

print("Shape:", schedules_df.shape)

print("\nColumns:")
print(schedules_df.columns.tolist())

print("\nData types:")
print(schedules_df.dtypes)

Shape: (417080, 8)

Columns:
['arrival', 'day', 'train_name', 'station_name', 'station_code', 'id', 'train_number', 'departure']

Data types:
arrival          object
day             float64
train_name       object
station_name     object
station_code     object
id                int64
train_number     object
departure        object
dtype: object


In [64]:
# Step 30 — Schedule missingness audit

print("=== SCHEDULE MISSINGNESS ===")

schedule_missing = schedules_df.isna().sum()

print(schedule_missing)

print("\nColumns with missing values:")
print(
    schedule_missing[schedule_missing > 0]
    .sort_values(ascending=False)
)

=== SCHEDULE MISSINGNESS ===
arrival             0
day             22561
train_name          0
station_name        0
station_code        0
id                  0
train_number        0
departure           0
dtype: int64

Columns with missing values:
day    22561
dtype: int64


In [65]:
# Check source-encoded missing values

for col in ["arrival", "departure", "day", "station_code",
            "station_name", "train_number"]:
    
    print(f"\n{col}:")
    print("NaN:", schedules_df[col].isna().sum())
    print("'None':", (schedules_df[col] == "None").sum())
    print("Empty:", (schedules_df[col] == "").sum())


arrival:
NaN: 0
'None': 27835
Empty: 0

departure:
NaN: 0
'None': 27827
Empty: 0

day:
NaN: 22561
'None': 0
Empty: 0

station_code:
NaN: 0
'None': 0
Empty: 0

station_name:
NaN: 0
'None': 0
Empty: 2

train_number:
NaN: 0
'None': 0
Empty: 0


In [66]:
# Step 31 — Schedule identifier audit

print("=== SCHEDULE IDENTIFIERS ===")

print("Total rows:", len(schedules_df))

print("Unique train numbers:",
      schedules_df["train_number"].nunique())

print("Unique station codes:",
      schedules_df["station_code"].nunique())

print("Duplicate complete rows:",
      schedules_df.duplicated().sum())

print("\nTrain number type:")
print(schedules_df["train_number"].dtype)

print("\nStation code type:")
print(schedules_df["station_code"].dtype)

=== SCHEDULE IDENTIFIERS ===
Total rows: 417080
Unique train numbers: 5208
Unique station codes: 8539
Duplicate complete rows: 0

Train number type:
object

Station code type:
object


In [67]:
# Step 32 — Schedule ordering inspection

print(schedules_df.head(20).to_string())

     arrival  day                             train_name         station_name station_code      id train_number departure
0       None  1.0             Falaknuma Lingampalli MMTS  KACHEGUDA FALAKNUMA           FM  302214        47154  07:55:00
1       None  1.0           Thrissur Guruvayur Passenger              THRISUR          TCR  281458        56044  18:55:00
2       None  1.0          Porbandar Muzaffarpur Express            PORBANDAR          PBR  309335        19269  15:05:00
3       None  1.0                     RAIPUR ITWARI PASS            RAIPUR JN            R  283774        58205  13:30:00
4       None  1.0                     Gomoh-Asansol MEMU             GOMOH JN          GMO  319937        63542  07:20:00
5       None  1.0    Rai Bareli-Mumbai LTT Saket Express        RAE BARELI JN          RBL  520605        21068  16:15:00
6       None  1.0    Subramanya Road Mangalore Passenger     Subrahmanya Road         SBHR  335930        56646  13:25:00
7       None  1.0       

In [68]:
# Inspect one complete train schedule

sample_train = schedules_df["train_number"].iloc[0]

print("Sample train:", sample_train)

schedules_df[
    schedules_df["train_number"] == sample_train
][
    [
        "train_number",
        "train_name",
        "station_code",
        "station_name",
        "arrival",
        "departure",
        "day",
        "id"
    ]
].to_string(index=False)

Sample train: 47154


'train_number                 train_name station_code            station_name  arrival departure  day     id\n       47154 Falaknuma Lingampalli MMTS           FM     KACHEGUDA FALAKNUMA     None  07:55:00  1.0 302214\n       47154 Falaknuma Lingampalli MMTS          HPG               HUPPUGUDA 07:57:00  07:58:00  1.0 302215\n       47154 Falaknuma Lingampalli MMTS          YKA     YAKUTPURA KACHEGUDA 08:01:00  08:02:00  1.0 302216\n       47154 Falaknuma Lingampalli MMTS          DQR           KACHEGUDA DBR 08:03:00  08:04:00  1.0 302217\n       47154 Falaknuma Lingampalli MMTS          MXT      MALAKPET KACHEGUDA 08:05:00  08:06:00  1.0 302218\n       47154 Falaknuma Lingampalli MMTS          KCG               KACHEGUDA 08:09:00  08:10:00  1.0 302219\n       47154 Falaknuma Lingampalli MMTS          VAR              VIDYANAGAR 08:13:00  08:14:00  1.0 302220\n       47154 Falaknuma Lingampalli MMTS          JOO KACHEGUDA JAMIA OSMANIA 08:15:00  08:16:00  1.0 302221\n       47154 Falak

In [69]:
# Step 30 — Schedule missingness audit

print("=== SCHEDULE MISSINGNESS ===")

for col in schedules_df.columns:
    print(
        f"{col:15} | "
        f"NaN: {schedules_df[col].isna().sum():6} | "
        f"'None': {(schedules_df[col] == 'None').sum():6} | "
        f"Empty: {(schedules_df[col] == '').sum():6}"
    )

=== SCHEDULE MISSINGNESS ===
arrival         | NaN:      0 | 'None':  27835 | Empty:      0
day             | NaN:  22561 | 'None':      0 | Empty:      0
train_name      | NaN:      0 | 'None':      0 | Empty:      8
station_name    | NaN:      0 | 'None':      0 | Empty:      2
station_code    | NaN:      0 | 'None':      0 | Empty:      0
id              | NaN:      0 | 'None':      0 | Empty:      0
train_number    | NaN:      0 | 'None':      0 | Empty:      0
departure       | NaN:      0 | 'None':  27827 | Empty:      0


In [70]:
# Step 31 — Schedule identifier audit

print("=== SCHEDULE IDENTIFIERS ===")

print("Total rows:", len(schedules_df))
print("Unique train numbers:", schedules_df["train_number"].nunique())
print("Unique station codes:", schedules_df["station_code"].nunique())

print("Duplicate complete rows:", schedules_df.duplicated().sum())

print(
    "Missing train numbers:",
    schedules_df["train_number"].isna().sum()
)

print(
    "Missing station codes:",
    schedules_df["station_code"].isna().sum()
)

print(
    "Empty train numbers:",
    (schedules_df["train_number"] == "").sum()
)

print(
    "Empty station codes:",
    (schedules_df["station_code"] == "").sum()
)

=== SCHEDULE IDENTIFIERS ===
Total rows: 417080
Unique train numbers: 5208
Unique station codes: 8539
Duplicate complete rows: 0
Missing train numbers: 0
Missing station codes: 0
Empty train numbers: 0
Empty station codes: 0


In [71]:
# Step 32 — Schedule train coverage against cleaned train master

schedule_train_numbers = set(schedules_df["train_number"].dropna())
master_train_numbers = set(final_trains["train_number"].dropna())

missing_trains = schedule_train_numbers - master_train_numbers

print("Unique trains in schedules:", len(schedule_train_numbers))
print("Trains in cleaned master:", len(master_train_numbers))
print("Schedule trains missing from master:", len(missing_trains))

if missing_trains:
    print(sorted(missing_trains)[:20])

Unique trains in schedules: 5208
Trains in cleaned master: 5208
Schedule trains missing from master: 0


In [72]:
# Step 33 — Schedule station coverage against cleaned station master

schedule_station_codes = set(schedules_df["station_code"].dropna())
master_station_codes = set(final_stations["station_code"].dropna())

missing_stations = schedule_station_codes - master_station_codes

print("Unique stations in schedules:", len(schedule_station_codes))
print("Stations in cleaned master:", len(master_station_codes))
print("Schedule stations missing from master:", len(missing_stations))

if missing_stations:
    print(sorted(missing_stations)[:50])

Unique stations in schedules: 8539
Stations in cleaned master: 8990
Schedule stations missing from master: 0


In [73]:
# Step 34 — Validate source ordering

id_order_issues = []

for train_no, group in schedules_df.groupby("train_number", sort=False):
    ids = group["id"].to_numpy()

    if len(ids) > 1 and not (ids[1:] > ids[:-1]).all():
        id_order_issues.append(train_no)

print("Trains with non-increasing source IDs:", len(id_order_issues))

if id_order_issues:
    print(id_order_issues[:20])

Trains with non-increasing source IDs: 9
['04857', '04853', '04854', '09765', '09766', '08786', '04858', '08788', '08785']


In [74]:
# Step 35 — Inspect source-order anomalies

for train_no in id_order_issues:
    print(f"\n{'='*70}")
    print(f"TRAIN: {train_no}")
    
    print(
        schedules_df[
            schedules_df["train_number"] == train_no
        ][
            [
                "train_number",
                "station_code",
                "station_name",
                "arrival",
                "departure",
                "day",
                "id"
            ]
        ].to_string(index=False)
    )


TRAIN: 04857
train_number station_code  station_name  arrival departure  day    id
       04857          PPR PIPAR ROAD JN 06:49:00  06:50:00  1.0 11813
       04857          PPR PIPAR ROAD JN 06:49:00  06:50:00  1.0 11615
       04857          KSW   KHERI SALWA 07:00:00  07:01:00  1.0 11616
       04857          KSW   KHERI SALWA 07:00:00  07:01:00  1.0 11814
       04857          AAS      ASRANADA 07:12:00  07:14:00  1.0 11815
       04857          AAS      ASRANADA 07:12:00  07:14:00  1.0 11617
       04857          JWL       JAJIWAL 07:23:00  07:24:00  1.0 11816
       04857          JWL       JAJIWAL 07:23:00  07:24:00  1.0 11618
       04857          BNO         BANAR 07:46:00  07:48:00  1.0 11619
       04857          BNO         BANAR 07:46:00  07:48:00  1.0 11817
       04857          RKB    RAIKA BAGH 08:10:00  08:12:00  1.0 11620
       04857          RKB    RAIKA BAGH 08:10:00  08:12:00  1.0 11818
       04857           JU    JODHPUR JN 08:30:00      None  1.0 11621
      

In [75]:
# Step 36 — Detect consecutive duplicate station records

consecutive_duplicates = []

for train_no, group in schedules_df.groupby("train_number", sort=False):
    stations = group["station_code"].tolist()
    
    for i in range(1, len(stations)):
        if stations[i] == stations[i - 1]:
            consecutive_duplicates.append({
                "train_number": train_no,
                "row_position": i,
                "station_code": stations[i],
                "previous_id": group.iloc[i - 1]["id"],
                "current_id": group.iloc[i]["id"]
            })

consecutive_duplicates_df = pd.DataFrame(consecutive_duplicates)

print("Consecutive duplicate station records:",
      len(consecutive_duplicates_df))

print(
    "Affected trains:",
    consecutive_duplicates_df["train_number"].nunique()
)

consecutive_duplicates_df

Consecutive duplicate station records: 110
Affected trains: 10


,train_number,row_position,station_code,previous_id,current_id
0,04857,1,PPR,11813,11615
1,04857,3,KSW,11616,11814
2,04857,5,AAS,11815,11617
3,04857,7,JWL,11816,11618
4,04857,9,BNO,11619,11817
...,...,...,...,...,...
105,08785,9,SKSO,12207,6202
106,08785,11,LBO,12208,6203
107,08785,13,BXA,6204,12209
108,08785,15,KYS,12210,6205


In [76]:
# Step 37 — Remove consecutive duplicate station records
# Source order is preserved.

schedules_clean = schedules_df.copy()

schedules_clean["consecutive_duplicate"] = (
    schedules_clean.groupby("train_number")["station_code"]
    .transform(lambda s: s.eq(s.shift()))
)

print(
    "Rows flagged as consecutive duplicates:",
    schedules_clean["consecutive_duplicate"].sum()
)

print(
    "Affected trains:",
    schedules_clean.loc[
        schedules_clean["consecutive_duplicate"],
        "train_number"
    ].nunique()
)

schedules_clean = schedules_clean[
    ~schedules_clean["consecutive_duplicate"]
].copy()

print("\nOriginal rows:", len(schedules_df))
print("Cleaned rows:", len(schedules_clean))
print("Rows removed:", len(schedules_df) - len(schedules_clean))

Rows flagged as consecutive duplicates: 110
Affected trains: 10

Original rows: 417080
Cleaned rows: 416970
Rows removed: 110


In [77]:
# Step 38 — Assign stop sequence within each train

schedules_clean["stop_sequence"] = (
    schedules_clean.groupby("train_number", sort=False)
    .cumcount() + 1
)

print(
    schedules_clean[
        [
            "train_number",
            "station_code",
            "stop_sequence"
        ]
    ].head(30).to_string(index=False)
)

train_number station_code  stop_sequence
       47154           FM              1
       56044          TCR              1
       19269          PBR              1
       58205            R              1
       63542          GMO              1
       21068          RBL              1
       56646         SBHR              1
       56379          ERS              1
       55145          SRU              1
       51688          BIR              1
       63596          ASN              1
       52228          TPU              1
       59227          BTD              1
       14523          MFP              1
       58151         BRMP              1
       59478          PTN              1
       31929         SDAH              1
       04857          PPR              1
       53215          BXR              1
       38453          HWH              1
       53067          BWN              1
       12260         NDLS              1
       47128          LPI              1
       12720    

In [78]:
# Step 39 — Verify stop sequence integrity

sequence_issues = []

for train_no, group in schedules_clean.groupby("train_number", sort=False):
    expected = list(range(1, len(group) + 1))
    
    if group["stop_sequence"].tolist() != expected:
        sequence_issues.append(train_no)

print("Trains with sequence-number issues:", len(sequence_issues))

Trains with sequence-number issues: 0


In [79]:
# Step 40 — Verify duplicate removal

remaining_duplicates = (
    schedules_clean.groupby("train_number")["station_code"]
    .apply(lambda s: s.eq(s.shift()).sum())
)

print(
    "Remaining consecutive duplicate station records:",
    remaining_duplicates.sum()
)

Remaining consecutive duplicate station records: 0


In [80]:
# Step 41 — Schedule endpoint validation

schedule_endpoints = (
    schedules_clean
    .sort_values(["train_number", "stop_sequence"])
    .groupby("train_number")
    .agg(
        schedule_origin=("station_code", "first"),
        schedule_destination=("station_code", "last"),
        stop_count=("station_code", "size")
    )
    .reset_index()
)

endpoint_check = schedule_endpoints.merge(
    final_trains[
        [
            "train_number",
            "from_station_code",
            "to_station_code"
        ]
    ],
    on="train_number",
    how="left"
)

endpoint_check["origin_match"] = (
    endpoint_check["schedule_origin"]
    == endpoint_check["from_station_code"]
)

endpoint_check["destination_match"] = (
    endpoint_check["schedule_destination"]
    == endpoint_check["to_station_code"]
)

print("Total trains checked:", len(endpoint_check))
print(
    "Origin matches:",
    endpoint_check["origin_match"].sum()
)
print(
    "Destination matches:",
    endpoint_check["destination_match"].sum()
)

print(
    "Trains with any endpoint mismatch:",
    (
        ~(
            endpoint_check["origin_match"]
            & endpoint_check["destination_match"]
        )
    ).sum()
)

Total trains checked: 5208
Origin matches: 5203
Destination matches: 5192
Trains with any endpoint mismatch: 19


In [81]:
# Inspect endpoint mismatches

endpoint_mismatches = endpoint_check[
    ~(
        endpoint_check["origin_match"]
        & endpoint_check["destination_match"]
    )
]

endpoint_mismatches.to_string(index=False)

'train_number schedule_origin schedule_destination  stop_count from_station_code to_station_code  origin_match  destination_match\n       12495             BKN                  BWN         270               BKN            KOAA          True              False\n       13108            KOAA                 DSNB          37              KOAA            DHCA          True              False\n       13109            KOAA                 DSNB          37              KOAA            DHCA          True              False\n       15009             LJN                 ADTL          69               LJN             JBP          True              False\n       15901             SBC                 DBRG         627               SBC            NTSK          True              False\n       15904             CDG                 DBRG         414               CDG            DBRT          True              False\n       15934             ASR                 DBRG         452               ASR          

In [82]:
# Step 42 — Endpoint consistency flags

endpoint_check["endpoint_consistent"] = (
    endpoint_check["origin_match"]
    & endpoint_check["destination_match"]
)

endpoint_check["endpoint_mismatch_type"] = "none"

endpoint_check.loc[
    (~endpoint_check["origin_match"])
    & (endpoint_check["destination_match"]),
    "endpoint_mismatch_type"
] = "origin_only"

endpoint_check.loc[
    (endpoint_check["origin_match"])
    & (~endpoint_check["destination_match"]),
    "endpoint_mismatch_type"
] = "destination_only"

endpoint_check.loc[
    (~endpoint_check["origin_match"])
    & (~endpoint_check["destination_match"]),
    "endpoint_mismatch_type"
] = "both"

print(
    endpoint_check["endpoint_mismatch_type"].value_counts()
)

endpoint_mismatch_type
none                5189
destination_only      14
origin_only            3
both                   2
Name: count, dtype: int64


In [83]:
# Step 43 — Attach train-level endpoint QA to schedule records

endpoint_flags = endpoint_check[
    [
        "train_number",
        "schedule_origin",
        "schedule_destination",
        "stop_count",
        "origin_match",
        "destination_match",
        "endpoint_consistent",
        "endpoint_mismatch_type"
    ]
].copy()

schedules_clean = schedules_clean.merge(
    endpoint_flags,
    on="train_number",
    how="left",
    validate="many_to_one"
)

print("Schedule shape:", schedules_clean.shape)

print(
    "Rows without endpoint QA:",
    schedules_clean["endpoint_consistent"].isna().sum()
)

Schedule shape: (416970, 17)
Rows without endpoint QA: 0


In [84]:
# Step 44 — Validate schedule station names against station master

station_name_map = (
    final_stations
    .set_index("station_code")["station_name"]
    .to_dict()
)

schedules_clean["station_master_name"] = (
    schedules_clean["station_code"]
    .map(station_name_map)
)

def normalize_name(x):
    if pd.isna(x):
        return ""
    return (
        str(x)
        .strip()
        .upper()
        .replace(".", "")
        .replace(",", "")
    )

schedules_clean["schedule_name_normalized"] = (
    schedules_clean["station_name"].apply(normalize_name)
)

schedules_clean["master_name_normalized"] = (
    schedules_clean["station_master_name"].apply(normalize_name)
)

schedules_clean["station_name_match"] = (
    schedules_clean["schedule_name_normalized"]
    == schedules_clean["master_name_normalized"]
)

print(
    "Schedule station rows:",
    len(schedules_clean)
)

print(
    "Station names matching master:",
    schedules_clean["station_name_match"].sum()
)

print(
    "Station name mismatches:",
    (~schedules_clean["station_name_match"]).sum()
)

Schedule station rows: 416970
Station names matching master: 416970
Station name mismatches: 0


In [85]:
# Inspect station-name mismatches

schedules_clean[
    ~schedules_clean["station_name_match"]
][
    [
        "train_number",
        "station_code",
        "station_name",
        "station_master_name"
    ]
].drop_duplicates().to_string(index=False)

'Empty DataFrame\nColumns: [train_number, station_code, station_name, station_master_name]\nIndex: []'

In [86]:
# Step 45 — Convert source-encoded missing values to NaN

import numpy as np

for col in ["arrival", "departure"]:
    schedules_clean[col] = schedules_clean[col].replace("None", np.nan)

print("Arrival missing:", schedules_clean["arrival"].isna().sum())
print("Departure missing:", schedules_clean["departure"].isna().sum())

Arrival missing: 27828
Departure missing: 27819


In [87]:
# Step 46 — Validate arrival/departure time format

from datetime import datetime

def valid_time(value):
    if pd.isna(value):
        return True
    
    try:
        datetime.strptime(str(value), "%H:%M:%S")
        return True
    except ValueError:
        return False

for col in ["arrival", "departure"]:
    invalid = ~schedules_clean[col].apply(valid_time)
    
    print(f"{col} invalid values:", invalid.sum())
    
    if invalid.sum() > 0:
        print(
            schedules_clean.loc[invalid, [col]]
            .drop_duplicates()
            .head(20)
        )

arrival invalid values: 0
departure invalid values: 0


In [88]:
# Step 47 — Day-field quality

print("Missing day:", schedules_clean["day"].isna().sum())

print("\nUnique day values:")
print(
    sorted(
        schedules_clean["day"]
        .dropna()
        .unique()
    )
)

print("\nInvalid day values:")
print(
    schedules_clean[
        schedules_clean["day"].notna()
        & ~schedules_clean["day"].isin(range(1, 8))
    ]["day"].value_counts()
)

Missing day: 22561

Unique day values:
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0)]

Invalid day values:
day
10.0    14
8.0      5
9.0      5
11.0     5
12.0     5
13.0     1
Name: count, dtype: int64


In [89]:
# Investigate remaining non-time / missing-like values

for col in ["arrival", "departure"]:
    print(f"\n=== {col.upper()} ===")
    
    print("NaN:", schedules_clean[col].isna().sum())
    
    print("\nNon-standard values:")
    print(
        schedules_clean[
            schedules_clean[col].notna()
            & ~schedules_clean[col].astype(str).str.match(
                r"^\d{2}:\d{2}:\d{2}$"
            )
        ][col]
        .value_counts(dropna=False)
    )


=== ARRIVAL ===
NaN: 27828

Non-standard values:
Series([], Name: count, dtype: int64)

=== DEPARTURE ===
NaN: 27819

Non-standard values:
Series([], Name: count, dtype: int64)


In [90]:
# Step 48 — Inspect records with day > 7

extended_day = schedules_clean[
    schedules_clean["day"] > 7
].copy()

print("Records with day > 7:", len(extended_day))

print(
    extended_day[
        [
            "train_number",
            "train_name",
            "station_code",
            "station_name",
            "arrival",
            "departure",
            "day",
            "id"
        ]
    ].to_string(index=False)
)

Records with day > 7: 35
train_number                        train_name station_code       station_name  arrival departure  day     id
       53017          Katwa Azimganj Passenger         CWLE        CHOWRIGACHA 20:36:00  20:37:00  8.0 108659
       53017          Katwa Azimganj Passenger         KTLR    KANTHALIYA ROAD      NaN       NaN  9.0 108660
       53017          Katwa Azimganj Passenger         KNSN      KARNA SUBARNA 20:45:00  20:46:00  9.0 108661
       41112 Velachery Chennai Beach Passenger         MTCN         Triplicane 19:23:00  19:24:00  8.0  12868
       53017          Katwa Azimganj Passenger          JVT       JIVANTI HALT 20:51:00  20:52:00 10.0 108662
       41112 Velachery Chennai Beach Passenger         MCPK            Chepauk 19:25:00  19:26:00  8.0  12869
       53017          Katwa Azimganj Passenger         KGLE    KHAGRAGHAT ROAD 20:58:00  20:59:00 11.0 108663
       41112 Velachery Chennai Beach Passenger         MCPT       Chintadripet 19:27:00  19:28:

In [91]:
print(
    "Affected trains:",
    extended_day["train_number"].nunique()
)

print(
    extended_day.groupby("train_number")["day"]
    .agg(["min", "max", "count"])
    .to_string()
)

Affected trains: 4
              min   max  count
train_number                  
41112         8.0  12.0      6
53017         8.0  12.0     10
55715         8.0  10.0     13
55812         8.0  13.0      6


In [92]:
# Step 49 — Final schedule quality flags

schedules_clean["missing_arrival"] = (
    schedules_clean["arrival"].isna()
)

schedules_clean["missing_departure"] = (
    schedules_clean["departure"].isna()
)

schedules_clean["missing_day"] = (
    schedules_clean["day"].isna()
)

schedules_clean["schedule_data_quality"] = "complete"

schedules_clean.loc[
    schedules_clean["missing_arrival"]
    | schedules_clean["missing_departure"]
    | schedules_clean["missing_day"],
    "schedule_data_quality"
] = "partial"

In [93]:
print(
    schedules_clean["schedule_data_quality"]
    .value_counts()
)

print("\nMissing arrival:",
      schedules_clean["missing_arrival"].sum())

print("Missing departure:",
      schedules_clean["missing_departure"].sum())

print("Missing day:",
      schedules_clean["missing_day"].sum())

print("Endpoint mismatches:",
      (~schedules_clean["endpoint_consistent"]).sum())

schedule_data_quality
complete    384012
partial      32958
Name: count, dtype: int64

Missing arrival: 27828
Missing departure: 27819
Missing day: 22561
Endpoint mismatches: 2683


In [94]:
# Step 50 — Final duplicate audit

schedule_key_columns = [
    "train_number",
    "station_code",
    "stop_sequence"
]

print(
    "Duplicate train/station/sequence keys:",
    schedules_clean.duplicated(
        subset=schedule_key_columns
    ).sum()
)

Duplicate train/station/sequence keys: 0


In [95]:
# Step 51 — Select final schedule columns

final_schedule_columns = [
    "train_number",
    "train_name",
    "station_code",
    "station_name",
    "arrival",
    "departure",
    "day",
    "id",
    "stop_sequence",

    # Train-level endpoint QA
    "schedule_origin",
    "schedule_destination",
    "stop_count",
    "origin_match",
    "destination_match",
    "endpoint_consistent",
    "endpoint_mismatch_type",

    # Station validation
    "station_master_name",
    "station_name_match",

    # Schedule quality
    "missing_arrival",
    "missing_departure",
    "missing_day",
    "schedule_data_quality"
]

final_schedules = schedules_clean[
    final_schedule_columns
].copy()

print("Final schedule shape:", final_schedules.shape)
print("Unique trains:", final_schedules["train_number"].nunique())
print("Unique stations:", final_schedules["station_code"].nunique())

Final schedule shape: (416970, 22)
Unique trains: 5208
Unique stations: 8539


In [96]:
# Step 52 — Final schedule integrity check

print("=== FINAL SCHEDULE QA ===")

print("Shape:", final_schedules.shape)
print("Unique trains:", final_schedules["train_number"].nunique())
print("Unique stations:", final_schedules["station_code"].nunique())

print(
    "Duplicate train/station/sequence keys:",
    final_schedules.duplicated(
        subset=["train_number", "station_code", "stop_sequence"]
    ).sum()
)

print(
    "Missing station master validation:",
    final_schedules["station_master_name"].isna().sum()
)

print(
    "Station name mismatches:",
    (~final_schedules["station_name_match"]).sum()
)

print(
    "Endpoint-mismatch trains:",
    final_schedules[
        ~final_schedules["endpoint_consistent"]
    ]["train_number"].nunique()
)

print("\nSchedule quality:")
print(final_schedules["schedule_data_quality"].value_counts())

print("\nMissing arrival:", final_schedules["missing_arrival"].sum())
print("Missing departure:", final_schedules["missing_departure"].sum())
print("Missing day:", final_schedules["missing_day"].sum())

=== FINAL SCHEDULE QA ===
Shape: (416970, 22)
Unique trains: 5208
Unique stations: 8539
Duplicate train/station/sequence keys: 0
Missing station master validation: 0
Station name mismatches: 0
Endpoint-mismatch trains: 19

Schedule quality:
schedule_data_quality
complete    384012
partial      32958
Name: count, dtype: int64

Missing arrival: 27828
Missing departure: 27819
Missing day: 22561


In [97]:
# Step 53 — Export cleaned schedules

from pathlib import Path
schedule_output_path = Path.cwd() / "processed_data" / "cleaned_schedules.csv"

final_schedules.to_csv(
    schedule_output_path,
    index=False
)

print("Saved:", schedule_output_path.resolve())

Saved: C:\Users\vinja\Desktop\Rail-Intell\processed_data\cleaned_schedules.csv


In [98]:
# Step 54 — Reload validation

schedules_check = pd.read_csv(schedule_output_path)

print("Reloaded shape:", schedules_check.shape)
print("Unique trains:", schedules_check["train_number"].nunique())
print("Unique stations:", schedules_check["station_code"].nunique())

print(
    "Duplicate keys:",
    schedules_check.duplicated(
        subset=["train_number", "station_code", "stop_sequence"]
    ).sum()
)

Reloaded shape: (416970, 22)
Unique trains: 5208
Unique stations: 8539
Duplicate keys: 0


In [99]:
# Restore Railway Network dataframe

from pathlib import Path
network_path = Path.cwd() / "Railway_Network_.csv"

network_df = pd.read_csv(network_path)

print("Railway Network shape:", network_df.shape)
print("Columns:")
print(network_df.columns.tolist())

Railway Network shape: (119446, 15)
Columns:
['objectid', 'id', 'fromjunction', 'tojunction', 'speed', 'type', 'noofstations', 'tracklength', 'railwayzone', 'nooflanes', 'bridge_yn', 'tunnel_yn', 'iso_countrycode', 'remark', 'st_lengthshape']


C:\Users\vinja\AppData\Local\Temp\ipykernel_27052\2401644217.py:6: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  network_df = pd.read_csv(network_path)


In [100]:
# Task 55A — Railway Network attribute readiness

network_attributes = [
    "railwayzone",
    "fromjunction",
    "tojunction",
    "type",
    "noofstations",
    "tracklength",
    "speed",
    "nooflanes",
    "bridge_yn",
    "tunnel_yn"
]

print("=== RAILWAY NETWORK ATTRIBUTE READINESS ===")

for col in network_attributes:
    print(f"\n--- {col} ---")
    print("Missing:", network_df[col].isna().sum())
    print("Unique:", network_df[col].nunique())

    if network_df[col].dtype == "object":
        print(network_df[col].value_counts(dropna=False).head(10))

=== RAILWAY NETWORK ATTRIBUTE READINESS ===

--- railwayzone ---
Missing: 4248
Unique: 25
railwayzone
South Central Railway    14242
Western Railway          13761
South Western Railway     8475
Southern Railway          8011
North Western Railway     7768
Northern Railway          6862
West Central Railway      6829
East Coast Railway        6632
East Central Railway      6385
North Central Railway     6035
Name: count, dtype: int64

--- fromjunction ---
Missing: 1004
Unique: 1570
fromjunction
Mavli Junction            1117
NaN                       1004
Yerraguntla Junction       979
Dharmavaram Junction       790
Khurda Road Junction       785
Kurduwadi Junction         755
Marwar Junction            747
Vriddhachalam Junction     737
Madgaon Junction           730
Guna Junction              712
Name: count, dtype: int64

--- tojunction ---
Missing: 1004
Unique: 1679
tojunction
Naupada Junction      1042
NaN                   1004
Nainpur Junction       895
Pakala Junction        84

In [101]:
# Task 55B — Numeric attribute sanity

numeric_cols = [
    "noofstations",
    "tracklength",
    "speed",
    "nooflanes"
]

print("=== NUMERIC ATTRIBUTE SUMMARY ===")
print(network_df[numeric_cols].describe().T)

=== NUMERIC ATTRIBUTE SUMMARY ===
                 count       mean        std  min   25%    50%    75%    max
noofstations  118886.0  15.298975   9.576742  0.0   8.0   14.0   19.0   58.0
tracklength   118886.0  84.760392  64.921406  0.0  38.0   68.0  116.0  487.0
speed          78072.0  86.883441  43.691422  0.0  75.0  110.0  110.0  160.0
nooflanes     117978.0   1.491854   0.631247 -4.0   1.0    1.0    2.0    4.0


In [102]:
# Task 55C — Railway zone coverage

print("=== RAILWAY ZONE COVERAGE ===")
print(network_df["railwayzone"].value_counts(dropna=False))

=== RAILWAY ZONE COVERAGE ===
railwayzone
South Central Railway          14242
Western Railway                13761
South Western Railway           8475
Southern Railway                8011
North Western Railway           7768
Northern Railway                6862
West Central Railway            6829
East Coast Railway              6632
East Central Railway            6385
North Central Railway           6035
Central Railway                 6018
North Frontier Railway          5906
South Eastern Railway           4794
South East Central Railway      4464
NaN                             4248
North Eastern Railway           3715
Eastern Railway                 3503
Konkan Railway                  1426
 Northeast Frontier Railway      184
Northeast Frontier Railway       142
 Eastern Railway                  18
                                  17
Southen Railway                    5
 East Coast Railwa                 4
Westem Railway                     1
 East Central Railway            

In [103]:
# Task 55D — Remark column audit

print("=== REMARK COLUMN AUDIT ===")
print("dtype:", network_df["remark"].dtype)
print("Missing:", network_df["remark"].isna().sum())
print("Unique:", network_df["remark"].nunique())
print(network_df["remark"].value_counts(dropna=False).head(20))

=== REMARK COLUMN AUDIT ===
dtype: object
Missing: 119016
Unique: 2
remark
NaN                         119016
Under Construction Route       429
Connect with Bangladesh          1
Name: count, dtype: int64


In [104]:
# Task 56 — Identify invalid numeric values

print("=== INVALID NUMERIC VALUES ===")

for col in ["noofstations", "tracklength", "speed", "nooflanes"]:
    print(f"\n--- {col} ---")
    print("Zero:", (network_df[col] == 0).sum())
    print("Negative:", (network_df[col] < 0).sum())

print("\n=== INVALID NO-OF-LANES RECORDS ===")

invalid_lanes = network_df[
    network_df["nooflanes"] < 0
].copy()

print("Count:", len(invalid_lanes))

print(
    invalid_lanes[
        [
            "objectid",
            "id",
            "fromjunction",
            "tojunction",
            "nooflanes",
            "railwayzone",
            "type",
            "tracklength"
        ]
    ].to_string(index=False)
)

=== INVALID NUMERIC VALUES ===

--- noofstations ---
Zero: 471
Negative: 0

--- tracklength ---
Zero: 982
Negative: 0

--- speed ---
Zero: 13461
Negative: 0

--- nooflanes ---
Zero: 477
Negative: 20

=== INVALID NO-OF-LANES RECORDS ===
Count: 20
 objectid  id fromjunction tojunction  nooflanes                railwayzone        type  tracklength
   118555 NaN     Dhansiri     Molvom       -4.0 Northeast Frontier Railway Broad Gauge         29.0
   118558 NaN     Dhansiri     Molvom       -4.0 Northeast Frontier Railway Broad Gauge         29.0
   118559 NaN     Dhansiri     Molvom       -4.0 Northeast Frontier Railway Broad Gauge         29.0
   118560 NaN     Dhansiri     Molvom       -4.0 Northeast Frontier Railway Broad Gauge         29.0
   118561 NaN     Dhansiri     Molvom       -4.0 Northeast Frontier Railway Broad Gauge         29.0
   118562 NaN     Dhansiri     Molvom       -4.0 Northeast Frontier Railway Broad Gauge         29.0
   118563 NaN     Dhansiri     Molvom       -4.

In [105]:
# Task 57 — Exact categorical values

for col in ["type", "railwayzone", "bridge_yn", "tunnel_yn"]:
    print(f"\n=== {col.upper()} RAW VALUES ===")
    print(
        network_df[col]
        .astype("string")
        .str.strip()
        .value_counts(dropna=False)
    )


=== TYPE RAW VALUES ===
type
Broad Gauge       114968
Metre Gauge         2123
Narrow Gauge        1864
<NA>                 383
Standard Gauge        81
                      27
Name: count, dtype: Int64

=== RAILWAYZONE RAW VALUES ===
railwayzone
South Central Railway         14242
Western Railway               13761
South Western Railway          8475
Southern Railway               8011
North Western Railway          7768
Northern Railway               6862
West Central Railway           6829
East Coast Railway             6632
East Central Railway           6386
North Central Railway          6035
Central Railway                6018
North Frontier Railway         5906
South Eastern Railway          4794
South East Central Railway     4464
<NA>                           4248
North Eastern Railway          3715
Eastern Railway                3521
Konkan Railway                 1426
Northeast Frontier Railway      326
                                 17
Southen Railway               

In [106]:
# Task 58 — Railway Network cleaning and QA fields

network_clean = network_df.copy()

# --------------------------------------------------
# 1. Preserve raw categorical values
# --------------------------------------------------

network_clean["railwayzone_raw"] = network_clean["railwayzone"]
network_clean["type_raw"] = network_clean["type"]
network_clean["bridge_yn_raw"] = network_clean["bridge_yn"]
network_clean["tunnel_yn_raw"] = network_clean["tunnel_yn"]
network_clean["remark_raw"] = network_clean["remark"]


# --------------------------------------------------
# 2. Basic whitespace cleaning
# --------------------------------------------------

for col in [
    "railwayzone",
    "type",
    "bridge_yn",
    "tunnel_yn"
]:
    network_clean[col] = (
        network_clean[col]
        .astype("string")
        .str.strip()
    )

# Convert empty strings to missing
for col in [
    "railwayzone",
    "type",
    "bridge_yn",
    "tunnel_yn"
]:
    network_clean[col] = network_clean[col].replace("", pd.NA)


# --------------------------------------------------
# 3. Standardize only obvious source typos
# --------------------------------------------------

zone_corrections = {
    "Southen Railway": "Southern Railway",
    "East Coast Railwa": "East Coast Railway",
    "Westem Railway": "Western Railway"
}

network_clean["railwayzone_clean"] = (
    network_clean["railwayzone"]
    .replace(zone_corrections)
)


# --------------------------------------------------
# 4. Numeric QA flags
# --------------------------------------------------

for col in [
    "noofstations",
    "tracklength",
    "speed",
    "nooflanes"
]:
    network_clean[f"{col}_missing"] = (
        network_clean[col].isna()
    )

    network_clean[f"{col}_zero"] = (
        network_clean[col] == 0
    )

    network_clean[f"{col}_negative"] = (
        network_clean[col] < 0
    )


# --------------------------------------------------
# 5. Explicit invalid-lane flag
# --------------------------------------------------

network_clean["invalid_nooflanes"] = (
    network_clean["nooflanes"] < 0
)


# --------------------------------------------------
# 6. Geometry-network endpoint QA
# --------------------------------------------------

network_clean["missing_fromjunction"] = (
    network_clean["fromjunction"].isna()
)

network_clean["missing_tojunction"] = (
    network_clean["tojunction"].isna()
)


# --------------------------------------------------
# 7. Remark classification
# --------------------------------------------------

network_clean["is_under_construction"] = (
    network_clean["remark"]
    .astype("string")
    .str.strip()
    .eq("Under Construction Route")
)

network_clean["connects_bangladesh"] = (
    network_clean["remark"]
    .astype("string")
    .str.strip()
    .eq("Connect with Bangladesh")
)


print("Railway Network cleaned dataframe shape:",
      network_clean.shape)

print("\nInvalid lane records:",
      network_clean["invalid_nooflanes"].sum())

print("\nZone values after obvious typo correction:")
print(
    network_clean["railwayzone_clean"]
    .value_counts(dropna=False)
)

Railway Network cleaned dataframe shape: (119446, 38)

Invalid lane records: 20

Zone values after obvious typo correction:
railwayzone_clean
South Central Railway         14242
Western Railway               13762
South Western Railway          8475
Southern Railway               8016
North Western Railway          7768
Northern Railway               6862
West Central Railway           6829
East Coast Railway             6636
East Central Railway           6386
North Central Railway          6035
Central Railway                6018
North Frontier Railway         5906
South Eastern Railway          4794
South East Central Railway     4464
<NA>                           4265
North Eastern Railway          3715
Eastern Railway                3521
Konkan Railway                 1426
Northeast Frontier Railway      326
Name: count, dtype: Int64


In [107]:
# Task 59 — Cross-dataset railway zone comparison

print("=== TRAIN DATA ZONES ===")

train_zones = (
    final_trains["zone"]
    .astype("string")
    .str.strip()
    .value_counts(dropna=False)
)

print(train_zones)

print("\n=== STATION DATA ZONES ===")

station_zones = (
    final_stations["zone"]
    .astype("string")
    .str.strip()
    .value_counts(dropna=False)
)

print(station_zones)

print("\n=== RAILWAY NETWORK ZONES ===")

network_zones = (
    network_clean["railwayzone_clean"]
    .value_counts(dropna=False)
)

print(network_zones)

=== TRAIN DATA ZONES ===
zone
NR      628
SR      606
WR      470
SCR     437
NER     394
ER      369
CR      359
?       275
ECR     258
SER     251
NWR     232
SWR     227
NFR     219
NCR     135
ECoR    120
SECR    118
WCR      74
KR       21
         15
Name: count, dtype: Int64

=== STATION DATA ZONES ===
zone
<NA>    4532
NR       590
WR       504
NWR      426
SR       335
SCR      294
SWR      290
NER      264
ER       263
NFR      262
ECR      212
CR       191
WCR      172
NCR      158
SECR     144
SER      109
?        106
ECoR      75
KR        63
Name: count, dtype: Int64

=== RAILWAY NETWORK ZONES ===
railwayzone_clean
South Central Railway         14242
Western Railway               13762
South Western Railway          8475
Southern Railway               8016
North Western Railway          7768
Northern Railway               6862
West Central Railway           6829
East Coast Railway             6636
East Central Railway           6386
North Central Railway          6035
C

In [108]:
# Task 59B — Investigate Frontier Railway naming

print("=== FRONTIER ZONE COMPARISON ===")

for df_name, series in [
    ("Trains", final_trains["zone"]),
    ("Stations", final_stations["zone"]),
    ("Network", network_clean["railwayzone_clean"])
]:
    print(f"\n--- {df_name} ---")

    values = (
        series
        .astype("string")
        .str.strip()
    )

    print(
        values[
            values.str.contains(
                "Frontier",
                case=False,
                na=False
            )
        ].value_counts()
    )

=== FRONTIER ZONE COMPARISON ===

--- Trains ---
Series([], Name: count, dtype: Int64)

--- Stations ---
Series([], Name: count, dtype: Int64)

--- Network ---
railwayzone_clean
North Frontier Railway        5906
Northeast Frontier Railway     326
Name: count, dtype: Int64


In [109]:
# Task 59C — Remaining suspicious zone names

print("=== REMAINING NETWORK ZONE VALUES ===")

print(
    network_clean["railwayzone_clean"]
    .dropna()
    .sort_values()
    .unique()
)

=== REMAINING NETWORK ZONE VALUES ===
<StringArray>
[           'Central Railway',       'East Central Railway',
         'East Coast Railway',            'Eastern Railway',
             'Konkan Railway',      'North Central Railway',
      'North Eastern Railway',     'North Frontier Railway',
      'North Western Railway', 'Northeast Frontier Railway',
           'Northern Railway',      'South Central Railway',
 'South East Central Railway',      'South Eastern Railway',
      'South Western Railway',           'Southern Railway',
       'West Central Railway',            'Western Railway']
Length: 18, dtype: string


In [110]:
# Task 60 — Canonical railway zone normalization

network_clean["railwayzone_clean"] = (
    network_clean["railwayzone_clean"]
    .replace({
        "North Frontier Railway": "Northeast Frontier Railway"
    })
)

print("=== CANONICAL RAILWAY ZONES ===")

print(
    network_clean["railwayzone_clean"]
    .value_counts(dropna=False)
)

=== CANONICAL RAILWAY ZONES ===
railwayzone_clean
South Central Railway         14242
Western Railway               13762
South Western Railway          8475
Southern Railway               8016
North Western Railway          7768
Northern Railway               6862
West Central Railway           6829
East Coast Railway             6636
East Central Railway           6386
Northeast Frontier Railway     6232
North Central Railway          6035
Central Railway                6018
South Eastern Railway          4794
South East Central Railway     4464
<NA>                           4265
North Eastern Railway          3715
Eastern Railway                3521
Konkan Railway                 1426
Name: count, dtype: Int64


In [111]:
# Task 60B — Final zone validation

canonical_zones = sorted(
    network_clean["railwayzone_clean"]
    .dropna()
    .unique()
    .tolist()
)

print("Number of canonical zones:", len(canonical_zones))
print("\nCanonical zones:")

for zone in canonical_zones:
    print(zone)

Number of canonical zones: 17

Canonical zones:
Central Railway
East Central Railway
East Coast Railway
Eastern Railway
Konkan Railway
North Central Railway
North Eastern Railway
North Western Railway
Northeast Frontier Railway
Northern Railway
South Central Railway
South East Central Railway
South Eastern Railway
South Western Railway
Southern Railway
West Central Railway
Western Railway


In [112]:
# Task 60C — Verify Frontier consolidation

print(
    network_clean[
        network_clean["railwayzone_clean"]
        == "Northeast Frontier Railway"
    ].shape[0]
)

print(
    network_clean["railwayzone_clean"]
    .value_counts()
    .loc[["Northeast Frontier Railway"]]
)

6232
railwayzone_clean
Northeast Frontier Railway    6232
Name: count, dtype: Int64


In [113]:
# Task 61 — Network segment structure

print("=== NETWORK SEGMENT STRUCTURE ===")

print("Total rows:", len(network_clean))

print(
    "Unique objectid:",
    network_clean["objectid"].nunique()
)

print(
    "Unique id:",
    network_clean["id"].nunique()
)

print(
    "Unique fromjunction:",
    network_clean["fromjunction"].nunique()
)

print(
    "Unique tojunction:",
    network_clean["tojunction"].nunique()
)

print(
    "Unique junction pairs:",
    network_clean[
        ["fromjunction", "tojunction"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate objectid rows:",
    network_clean.duplicated(
        subset=["objectid"]
    ).sum()
)

print(
    "Duplicate junction-pair rows:",
    network_clean.duplicated(
        subset=["fromjunction", "tojunction"]
    ).sum()
)

=== NETWORK SEGMENT STRUCTURE ===
Total rows: 119446
Unique objectid: 119446
Unique id: 111727
Unique fromjunction: 1570
Unique tojunction: 1679
Unique junction pairs: 2306
Duplicate objectid rows: 0
Duplicate junction-pair rows: 117140


In [114]:
# Task 61B — Most repeated junction pairs

pair_counts = (
    network_clean
    .groupby(
        ["fromjunction", "tojunction"],
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
)

print("=== TOP REPEATED JUNCTION PAIRS ===")
print(pair_counts.head(20))

=== TOP REPEATED JUNCTION PAIRS ===
fromjunction          tojunction         
NaN                   NaN                    1004
Madgaon Junction      Thokur                  711
Khurda Road Junction  Naupada Junction        694
Dharmavaram Junction  Pakala Junction         681
Himmatnagar Junction  Jai Samand Road         653
Jammu Tawi            Baramula                643
Udhna Chord Cabin     Jalgaon Junction        611
Mudkhed Junction      Majri Bridge Cabin      610
Peddapalli Junction   Nizamabad Junction      584
Botad Junction        Sabarmati D Cabin       575
Yerraguntla Junction  Nandyal Junction        571
Guna Junction         Gwalior Junction        558
Chanderiya            Gurla West Cabin        545
Akola Junction        Purna Junction          534
Hosur                 Omalur Junction         521
Miraj Junction        Kurduwadi Junction      513
Veer                  Majorda Junction        508
Agartala              Sabroom                 505
Itwari Junction       

In [115]:
# Task 61C — Inspect one highly repeated pair

top_pair = pair_counts.index[0]

pair_sample = network_clean[
    (network_clean["fromjunction"] == top_pair[0]) &
    (network_clean["tojunction"] == top_pair[1])
]

print("Selected pair:", top_pair)
print("Rows:", len(pair_sample))

print(
    pair_sample[
        [
            "objectid",
            "id",
            "fromjunction",
            "tojunction",
            "type",
            "tracklength",
            "speed",
            "noofstations",
            "nooflanes",
            "railwayzone_clean",
            "bridge_yn",
            "tunnel_yn",
            "remark"
        ]
    ].to_string(index=False)
)

Selected pair: (nan, nan)
Rows: 0
Empty DataFrame
Columns: [objectid, id, fromjunction, tojunction, type, tracklength, speed, noofstations, nooflanes, railwayzone_clean, bridge_yn, tunnel_yn, remark]
Index: []


In [116]:
# Task 62 — Inspect most repeated valid junction pair

pair_counts_valid = (
    network_clean
    .dropna(subset=["fromjunction", "tojunction"])
    .groupby(["fromjunction", "tojunction"])
    .size()
    .sort_values(ascending=False)
)

print("=== TOP VALID JUNCTION PAIRS ===")
print(pair_counts_valid.head(20))

=== TOP VALID JUNCTION PAIRS ===
fromjunction          tojunction         
Madgaon Junction      Thokur                 711
Khurda Road Junction  Naupada Junction       694
Dharmavaram Junction  Pakala Junction        681
Himmatnagar Junction  Jai Samand Road        653
Jammu Tawi            Baramula               643
Udhna Chord Cabin     Jalgaon Junction       611
Mudkhed Junction      Majri Bridge Cabin     610
Peddapalli Junction   Nizamabad Junction     584
Botad Junction        Sabarmati D Cabin      575
Yerraguntla Junction  Nandyal Junction       571
Guna Junction         Gwalior Junction       558
Chanderiya            Gurla West Cabin       545
Akola Junction        Purna Junction         534
Hosur                 Omalur Junction        521
Miraj Junction        Kurduwadi Junction     513
Veer                  Majorda Junction       508
Agartala              Sabroom                505
Itwari Junction       Chhindwara Junction    499
Mavli Junction        Marwar Junction      

In [117]:
# Task 62B — Inspect repeated spatial features

top_pair = pair_counts_valid.index[0]

pair_sample = network_clean[
    (network_clean["fromjunction"] == top_pair[0]) &
    (network_clean["tojunction"] == top_pair[1])
].copy()

print("Selected pair:", top_pair)
print("Rows:", len(pair_sample))

print(
    pair_sample[
        [
            "objectid",
            "id",
            "fromjunction",
            "tojunction",
            "type",
            "tracklength",
            "speed",
            "noofstations",
            "nooflanes",
            "railwayzone_clean",
            "bridge_yn",
            "tunnel_yn",
            "remark"
        ]
    ].to_string(index=False)
)

Selected pair: ('Madgaon Junction', 'Thokur')
Rows: 711
 objectid       id     fromjunction tojunction        type  tracklength  speed  noofstations  nooflanes railwayzone_clean bridge_yn tunnel_yn remark
    60940 101999.0 Madgaon Junction     Thokur Broad Gauge        297.0  110.0          30.0        1.0    Konkan Railway         Y         N    NaN
    61111 100875.0 Madgaon Junction     Thokur Broad Gauge        297.0  110.0          30.0        1.0    Konkan Railway         N         N    NaN
    61112 101997.0 Madgaon Junction     Thokur Broad Gauge        297.0  110.0          30.0        1.0    Konkan Railway         Y         N    NaN
    61113 101998.0 Madgaon Junction     Thokur Broad Gauge        297.0  110.0          30.0        1.0    Konkan Railway         N         N    NaN
    61114 102000.0 Madgaon Junction     Thokur Broad Gauge        297.0  110.0          30.0        1.0    Konkan Railway         N         N    NaN
    61115 102001.0 Madgaon Junction     Thokur Bro

In [118]:
# Task 62C — Attribute variation within repeated pair

variation_columns = [
    "type",
    "tracklength",
    "speed",
    "noofstations",
    "nooflanes",
    "railwayzone_clean",
    "bridge_yn",
    "tunnel_yn",
    "remark"
]

print("=== ATTRIBUTE VARIATION ===")

for col in variation_columns:
    print(
        f"{col}:",
        pair_sample[col].nunique(dropna=False),
        "unique values"
    )

=== ATTRIBUTE VARIATION ===
type: 1 unique values
tracklength: 1 unique values
speed: 1 unique values
noofstations: 1 unique values
nooflanes: 1 unique values
railwayzone_clean: 1 unique values
bridge_yn: 2 unique values
tunnel_yn: 2 unique values
remark: 1 unique values


In [119]:
print("=== NETWORK CLEAN COLUMNS ===")
print(network_clean.columns.tolist())

print("\n=== GEOMETRY COLUMN CHECK ===")
print("geometry" in network_clean.columns)

if "geometry" in network_clean.columns:
    print("\nGeometry dtype:")
    print(network_clean["geometry"].dtype)

    print("\nFirst geometry value:")
    print(network_clean["geometry"].iloc[0])

=== NETWORK CLEAN COLUMNS ===
['objectid', 'id', 'fromjunction', 'tojunction', 'speed', 'type', 'noofstations', 'tracklength', 'railwayzone', 'nooflanes', 'bridge_yn', 'tunnel_yn', 'iso_countrycode', 'remark', 'st_lengthshape', 'railwayzone_raw', 'type_raw', 'bridge_yn_raw', 'tunnel_yn_raw', 'remark_raw', 'railwayzone_clean', 'noofstations_missing', 'noofstations_zero', 'noofstations_negative', 'tracklength_missing', 'tracklength_zero', 'tracklength_negative', 'speed_missing', 'speed_zero', 'speed_negative', 'nooflanes_missing', 'nooflanes_zero', 'nooflanes_negative', 'invalid_nooflanes', 'missing_fromjunction', 'missing_tojunction', 'is_under_construction', 'connects_bangladesh']

=== GEOMETRY COLUMN CHECK ===
False


In [120]:
import geopandas as gpd
from pathlib import Path

network_geo = gpd.read_file(
    Path.cwd() / "Railway_Network_.geojson"
)

print("Shape:", network_geo.shape)
print("CRS:", network_geo.crs)
print("Geometry types:")
print(network_geo.geometry.geom_type.value_counts())
print("Columns:")
print(network_geo.columns.tolist())

C:\Users\vinja\Projects\llm_engineering\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Several features with id = 80732 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


Shape: (119446, 16)
CRS: EPSG:4326
Geometry types:
LineString         119443
MultiLineString         3
Name: count, dtype: int64
Columns:
['objectid', 'id', 'fromjunction', 'tojunction', 'speed', 'type', 'noofstations', 'tracklength', 'railwayzone', 'nooflanes', 'bridge_yn', 'tunnel_yn', 'iso_countrycode', 'remark', 'st_lengthshape', 'geometry']


In [121]:
top_pair_geo = ("Madgaon Junction", "Thokur")

pair_geo = network_geo[
    (network_geo["fromjunction"] == top_pair_geo[0]) &
    (network_geo["tojunction"] == top_pair_geo[1])
].copy()

print("=== GEOJSON PAIR CHECK ===")
print("Selected pair:", top_pair_geo)
print("Rows:", len(pair_geo))
print("Unique objectids:", pair_geo["objectid"].nunique())
print("Unique ids:", pair_geo["id"].nunique())
print("Unique geometries:", pair_geo.geometry.apply(lambda g: g.wkb).nunique())

=== GEOJSON PAIR CHECK ===
Selected pair: ('Madgaon Junction', 'Thokur')
Rows: 711
Unique objectids: 711
Unique ids: 691
Unique geometries: 711


In [122]:
print("=== GEOMETRY DUPLICATION ===")

geom_wkb = pair_geo.geometry.apply(lambda g: g.wkb)

print("Total rows:", len(pair_geo))
print("Unique geometries:", geom_wkb.nunique())
print("Duplicate geometry rows:", geom_wkb.duplicated().sum())

=== GEOMETRY DUPLICATION ===
Total rows: 711
Unique geometries: 711
Duplicate geometry rows: 0


In [123]:
from shapely.ops import unary_union

print("=== SPATIAL STRUCTURE OF TOP CORRIDOR ===")

geom_union = unary_union(pair_geo.geometry)

print("Union geometry type:", geom_union.geom_type)

if geom_union.geom_type == "MultiLineString":
    print("Number of spatial components:", len(geom_union.geoms))
elif geom_union.geom_type == "LineString":
    print("Number of spatial components: 1")

=== SPATIAL STRUCTURE OF TOP CORRIDOR ===
Union geometry type: MultiLineString
Number of spatial components: 711


In [124]:
print("=== SEGMENT LENGTH DISTRIBUTION ===")

lengths = pair_geo.geometry.to_crs(epsg=32643).length / 1000

print(lengths.describe())

=== SEGMENT LENGTH DISTRIBUTION ===
count    711.000000
mean       0.415885
std        0.638754
min        0.006050
25%        0.019264
50%        0.157962
75%        0.503095
max        4.167760
dtype: float64


In [125]:
print("=== TRACKLENGTH VS GEOMETRY LENGTH ===")

pair_check = pair_geo.copy()

pair_check["geometry_length_km"] = (
    pair_check.geometry.to_crs(epsg=32643).length / 1000
)

print(
    pair_check[
        ["objectid", "tracklength", "geometry_length_km"]
    ].describe()
)

=== TRACKLENGTH VS GEOMETRY LENGTH ===
           objectid  tracklength  geometry_length_km
count    711.000000        711.0          711.000000
mean   63353.033755        297.0            0.415885
std     1670.127057          0.0            0.638754
min    60940.000000        297.0            0.006050
25%    61318.500000        297.0            0.019264
50%    64184.000000        297.0            0.157962
75%    64505.500000        297.0            0.503095
max    67849.000000        297.0            4.167760


In [126]:
print("=== TRACKLENGTH UNIQUE VALUES ===")

print(pair_check["tracklength"].value_counts(dropna=False).head(20))

=== TRACKLENGTH UNIQUE VALUES ===
tracklength
297.0    711
Name: count, dtype: int64


In [127]:
print("=== TOTAL GEOMETRY LENGTH ===")

print(
    "Sum of geometry lengths:",
    pair_check["geometry_length_km"].sum(),
    "km"
)

print(
    "Tracklength attribute:",
    pair_check["tracklength"].iloc[0],
    "km"
)

=== TOTAL GEOMETRY LENGTH ===
Sum of geometry lengths: 295.69400139790616 km
Tracklength attribute: 297.0 km


In [128]:
network_clean.groupby(
    ["fromjunction", "tojunction"]
).size()

fromjunction          tojunction        
Abhanpur Junction     Dhamtari              75
                      Kendri                 3
                      Rajim                 13
Abhayapuri            Dudhnai Junction      32
Abohar Junction       Bathinda Junction     13
                                            ..
Yog Nagari Rishikesh  Virbhadra              1
Zafarabad Junction    Janghai Junction      87
                      Jaunpur Junction      20
                      Sultanpur Junction    67
Zero Point Station    Munabao                1
Length: 2305, dtype: int64

In [129]:
corridor_segment_counts = (
    network_clean
    .dropna(subset=["fromjunction", "tojunction"])
    .groupby(["fromjunction", "tojunction"])
    .size()
)

print("=== CORRIDOR SEGMENTATION ===")
print("Valid corridors:", len(corridor_segment_counts))
print(corridor_segment_counts.describe())

=== CORRIDOR SEGMENTATION ===
Valid corridors: 2305
count    2305.000000
mean       51.384816
std        85.540544
min         1.000000
25%         4.000000
50%        19.000000
75%        60.000000
max       711.000000
dtype: float64


In [130]:
print("=== SEGMENTS PER CORRIDOR ===")

print("1 segment:", (corridor_segment_counts == 1).sum())
print("2–5 segments:", ((corridor_segment_counts >= 2) & (corridor_segment_counts <= 5)).sum())
print("6–10 segments:", ((corridor_segment_counts >= 6) & (corridor_segment_counts <= 10)).sum())
print("11–50 segments:", ((corridor_segment_counts >= 11) & (corridor_segment_counts <= 50)).sum())
print(">50 segments:", (corridor_segment_counts > 50).sum())

=== SEGMENTS PER CORRIDOR ===
1 segment: 289
2–5 segments: 407
6–10 segments: 208
11–50 segments: 742
>50 segments: 659


In [131]:
print("=== TOP 20 CORRIDORS BY SEGMENT COUNT ===")
print(corridor_segment_counts.sort_values(ascending=False).head(20))

=== TOP 20 CORRIDORS BY SEGMENT COUNT ===
fromjunction          tojunction         
Madgaon Junction      Thokur                 711
Khurda Road Junction  Naupada Junction       694
Dharmavaram Junction  Pakala Junction        681
Himmatnagar Junction  Jai Samand Road        653
Jammu Tawi            Baramula               643
Udhna Chord Cabin     Jalgaon Junction       611
Mudkhed Junction      Majri Bridge Cabin     610
Peddapalli Junction   Nizamabad Junction     584
Botad Junction        Sabarmati D Cabin      575
Yerraguntla Junction  Nandyal Junction       571
Guna Junction         Gwalior Junction       558
Chanderiya            Gurla West Cabin       545
Akola Junction        Purna Junction         534
Hosur                 Omalur Junction        521
Miraj Junction        Kurduwadi Junction     513
Veer                  Majorda Junction       508
Agartala              Sabroom                505
Itwari Junction       Chhindwara Junction    499
Mavli Junction        Marwar Junct

In [132]:
corridor_attribute_consistency = (
    network_clean
    .dropna(subset=["fromjunction", "tojunction"])
    .groupby(["fromjunction", "tojunction"])
    .agg(
        segment_count=("objectid", "size"),
        tracklength_unique=("tracklength", "nunique"),
        speed_unique=("speed", "nunique"),
        type_unique=("type", "nunique"),
        noofstations_unique=("noofstations", "nunique"),
        nooflanes_unique=("nooflanes", "nunique"),
        railwayzone_unique=("railwayzone_clean", "nunique")
    )
)

print("=== CORRIDOR ATTRIBUTE CONSISTENCY ===")

for col in [
    "tracklength_unique",
    "speed_unique",
    "type_unique",
    "noofstations_unique",
    "nooflanes_unique",
    "railwayzone_unique"
]:
    print(
        col,
        "— corridors with >1 value:",
        (corridor_attribute_consistency[col] > 1).sum()
    )

=== CORRIDOR ATTRIBUTE CONSISTENCY ===
tracklength_unique — corridors with >1 value: 57
speed_unique — corridors with >1 value: 22
type_unique — corridors with >1 value: 11
noofstations_unique — corridors with >1 value: 24
nooflanes_unique — corridors with >1 value: 60
railwayzone_unique — corridors with >1 value: 59


In [133]:
print("=== CORRIDORS WITH ATTRIBUTE CONFLICTS ===")

conflict_mask = (
    (corridor_attribute_consistency["tracklength_unique"] > 1) |
    (corridor_attribute_consistency["speed_unique"] > 1) |
    (corridor_attribute_consistency["type_unique"] > 1) |
    (corridor_attribute_consistency["noofstations_unique"] > 1) |
    (corridor_attribute_consistency["nooflanes_unique"] > 1) |
    (corridor_attribute_consistency["railwayzone_unique"] > 1)
)

print("Corridors with any conflict:", conflict_mask.sum())
print(
    corridor_attribute_consistency[conflict_mask]
    .sort_values("segment_count", ascending=False)
    .head(20)
)

=== CORRIDORS WITH ATTRIBUTE CONFLICTS ===
Corridors with any conflict: 201
                                             segment_count  \
fromjunction          tojunction                             
Chanderiya            Gurla West Cabin                 545   
Itwari Junction       Chhindwara Junction              499   
Golakganj Junction    Bilasipara                       443   
Marwar Junction       Palanpur Junction                419   
Mavli Junction        Bari Sadri                       398   
Kurduwadi Junction    Latur Road Junction              373   
Samdari Junction      Bhildi Junction                  365   
Duvvada               Samalkot Bypass Cabin            355   
Khairar Junction      Ohan Cabin                       349   
Vizianagaram Junction Naupada Junction                 348   
Yerraguntla Junction  Gooty Cabin                      294   
Sogaria (Kota)        Ruthiyai Junction                260   
Bharoli Junction      Jammu Tawi                       2

In [134]:
print("=== CONFLICT PATTERNS ===")

conflict_columns = [
    "tracklength_unique",
    "speed_unique",
    "type_unique",
    "noofstations_unique",
    "nooflanes_unique",
    "railwayzone_unique"
]

for col in conflict_columns:
    counts = corridor_attribute_consistency[col].value_counts().sort_index()
    print(f"\n{col}")
    print(counts)

=== CONFLICT PATTERNS ===

tracklength_unique
tracklength_unique
1    2248
2      55
3       2
Name: count, dtype: int64

speed_unique
speed_unique
0     818
1    1465
2      21
3       1
Name: count, dtype: int64

type_unique
type_unique
0       1
1    2293
2      11
Name: count, dtype: int64

noofstations_unique
noofstations_unique
1    2281
2      24
Name: count, dtype: int64

nooflanes_unique
nooflanes_unique
0      53
1    2192
2      58
3       2
Name: count, dtype: int64

railwayzone_unique
railwayzone_unique
0      48
1    2198
2      59
Name: count, dtype: int64


In [135]:
conflict_summary = (
    corridor_attribute_consistency[conflict_mask]
    .copy()
)

for col in conflict_columns:
    conflict_summary[col + "_conflict"] = (
        conflict_summary[col] > 1
    )

print("=== CONFLICT COMBINATIONS ===")

print(
    conflict_summary[
        [col + "_conflict" for col in conflict_columns]
    ]
    .value_counts()
    .head(20)
)

=== CONFLICT COMBINATIONS ===
tracklength_unique_conflict  speed_unique_conflict  type_unique_conflict  noofstations_unique_conflict  nooflanes_unique_conflict  railwayzone_unique_conflict
False                        False                  False                 False                         True                       False                          53
                                                                                                        False                      True                           52
True                         False                  False                 False                         False                      False                          37
False                        True                   False                 False                         False                      False                          15
True                         False                  False                 True                          False                      False               

In [136]:
conflict_attributes = [
    "tracklength",
    "speed",
    "type",
    "noofstations",
    "nooflanes",
    "railwayzone_clean"
]

for col in conflict_attributes:
    print(f"\n=== {col.upper()} CONFLICT VALUES ===")

    temp = (
        network_clean
        .dropna(subset=["fromjunction", "tojunction"])
        .groupby(["fromjunction", "tojunction"])[col]
        .agg(lambda x: sorted(x.dropna().unique().tolist()))
    )

    conflicts = temp[temp.apply(len) > 1]

    print("Conflicting corridors:", len(conflicts))
    print(conflicts.head(15))


=== TRACKLENGTH CONFLICT VALUES ===
Conflicting corridors: 57
fromjunction             tojunction                 
Achhnera Junction        Mathura Junction                     [29.0, 35.0]
Agartala                 Gangasagar Bangladesh                  [4.0, 6.0]
Arakkonam Junction       Melpakkam Cabin                       [3.0, 66.0]
Bakulha                  Phephna Junction                     [54.0, 66.0]
Bankura Junction         Mathnashipur                       [108.0, 118.0]
Bara Bani                Tapasi                               [16.0, 17.0]
Bathinda Junction        Bathinda C Cabin                       [4.0, 9.0]
Belwandi                 Kashti                               [24.0, 25.0]
Bengaluru City Junction  Bengaluru Cantt.                       [4.0, 9.0]
Bhadrak                  Ranital Link Junction Cabin           [5.0, 44.0]
Biratoli                 Shivpur                              [10.0, 35.0]
Chennai Egmore           Chennai Fort                      

In [137]:
print("=== EXAMPLE: NO. OF LANES CONFLICTS ===")

lanes_conflicts = (
    network_clean
    .dropna(subset=["fromjunction", "tojunction"])
    .groupby(["fromjunction", "tojunction"])["nooflanes"]
    .agg(lambda x: sorted(x.dropna().unique().tolist()))
)

lanes_conflicts = lanes_conflicts[lanes_conflicts.apply(len) > 1]

for pair in lanes_conflicts.head(10).index:
    sample = network_clean[
        (network_clean["fromjunction"] == pair[0]) &
        (network_clean["tojunction"] == pair[1])
    ]

    print("\nPAIR:", pair)
    print(sample["nooflanes"].value_counts(dropna=False))

=== EXAMPLE: NO. OF LANES CONFLICTS ===

PAIR: ('Ambale', 'Nira')
nooflanes
2.0    36
1.0     1
Name: count, dtype: int64

PAIR: ('Balamau Junction', 'Unnao Junction')
nooflanes
1.0    106
2.0      3
Name: count, dtype: int64

PAIR: ('Bangurgram Halt', 'Ajmer Junction')
nooflanes
2.0    113
1.0      7
Name: count, dtype: int64

PAIR: ('Belsonda', 'Titlagarh Junction')
nooflanes
2.0    187
1.0     32
Name: count, dtype: int64

PAIR: ('Bharoli Junction', 'Jammu Tawi')
nooflanes
2.0    259
1.0      1
Name: count, dtype: int64

PAIR: ('Bhimavaram A Cabin', 'Nidadavolu Junction')
nooflanes
2.0    73
1.0     2
Name: count, dtype: int64

PAIR: ('Bhimavaram Junction', 'Narasapur')
nooflanes
2.0    30
1.0    13
Name: count, dtype: int64

PAIR: ('Biratoli', 'Shivpur')
nooflanes
3.0    38
4.0    10
1.0     1
Name: count, dtype: int64

PAIR: ('Bortalao', 'Paniajob')
nooflanes
3.0    12
2.0     1
Name: count, dtype: int64

PAIR: ('Burhwal Junction', 'Gonda Junction')
nooflanes
3.0    61
2.0     1
N

In [138]:
corridor_missingness = (
    network_clean
    .dropna(subset=["fromjunction", "tojunction"])
    .groupby(["fromjunction", "tojunction"])
    .agg(
        segment_count=("objectid", "size"),

        tracklength_nonnull=("tracklength", lambda x: x.notna().sum()),
        speed_nonnull=("speed", lambda x: x.notna().sum()),
        type_nonnull=("type", lambda x: x.notna().sum()),
        noofstations_nonnull=("noofstations", lambda x: x.notna().sum()),
        nooflanes_nonnull=("nooflanes", lambda x: x.notna().sum()),
        railwayzone_nonnull=("railwayzone_clean", lambda x: x.notna().sum())
    )
)

print("=== CORRIDOR ATTRIBUTE MISSINGNESS ===")

for col in [
    "tracklength_nonnull",
    "speed_nonnull",
    "type_nonnull",
    "noofstations_nonnull",
    "nooflanes_nonnull",
    "railwayzone_nonnull"
]:
    print(
        col,
        "— completely missing:",
        (corridor_missingness[col] == 0).sum(),
        "— partially missing:",
        (
            (corridor_missingness[col] > 0) &
            (corridor_missingness[col] < corridor_missingness["segment_count"])
        ).sum()
    )

=== CORRIDOR ATTRIBUTE MISSINGNESS ===
tracklength_nonnull — completely missing: 0 — partially missing: 2
speed_nonnull — completely missing: 818 — partially missing: 58
type_nonnull — completely missing: 1 — partially missing: 10
noofstations_nonnull — completely missing: 0 — partially missing: 2
nooflanes_nonnull — completely missing: 53 — partially missing: 7
railwayzone_nonnull — completely missing: 48 — partially missing: 20


In [139]:
print("=== CORRIDORS WITH PARTIAL ATTRIBUTE MISSINGNESS ===")

for col in [
    "tracklength_nonnull",
    "speed_nonnull",
    "type_nonnull",
    "noofstations_nonnull",
    "nooflanes_nonnull",
    "railwayzone_nonnull"
]:
    partial = corridor_missingness[
        (corridor_missingness[col] > 0) &
        (corridor_missingness[col] < corridor_missingness["segment_count"])
    ]

    print(f"{col}: {len(partial)}")

=== CORRIDORS WITH PARTIAL ATTRIBUTE MISSINGNESS ===
tracklength_nonnull: 2
speed_nonnull: 58
type_nonnull: 10
noofstations_nonnull: 2
nooflanes_nonnull: 7
railwayzone_nonnull: 20


In [140]:
self_loop_corridors = (
    network_clean[
        network_clean["fromjunction"].notna() &
        network_clean["tojunction"].notna() &
        (network_clean["fromjunction"] == network_clean["tojunction"])
    ]
    .groupby(["fromjunction", "tojunction"])
    .size()
    .sort_values(ascending=False)
)

print("=== SELF-LOOP CORRIDORS ===")
print("Number of self-loop corridors:", len(self_loop_corridors))
print("Total self-loop segments:", self_loop_corridors.sum())
print(self_loop_corridors)

=== SELF-LOOP CORRIDORS ===
Number of self-loop corridors: 5
Total self-loop segments: 12
fromjunction            tojunction            
Magnesite Bypass Cabin  Magnesite Bypass Cabin    4
Ahiran Halt             Ahiran Halt               3
Biratnagar              Biratnagar                3
Khairar Junction        Khairar Junction          1
Pipar Road Junction     Pipar Road Junction       1
dtype: int64


In [141]:
print("=== SELF-LOOP DETAILS ===")

self_loop_rows = network_clean[
    network_clean["fromjunction"].notna() &
    network_clean["tojunction"].notna() &
    (network_clean["fromjunction"] == network_clean["tojunction"])
].copy()

print(
    self_loop_rows[
        [
            "objectid",
            "id",
            "fromjunction",
            "tojunction",
            "tracklength",
            "speed",
            "type",
            "noofstations",
            "nooflanes",
            "railwayzone_clean"
        ]
    ].to_string(index=False)
)

=== SELF-LOOP DETAILS ===
 objectid       id           fromjunction             tojunction  tracklength  speed        type  noofstations  nooflanes          railwayzone_clean
     3502 110218.0 Magnesite Bypass Cabin Magnesite Bypass Cabin          1.0    NaN Broad Gauge           2.0        1.0           Southern Railway
     3764  11226.0 Magnesite Bypass Cabin Magnesite Bypass Cabin          1.0    NaN Broad Gauge           2.0        1.0           Southern Railway
     7251 108977.0 Magnesite Bypass Cabin Magnesite Bypass Cabin          1.0    NaN Broad Gauge           2.0        1.0           Southern Railway
     7253 108977.0 Magnesite Bypass Cabin Magnesite Bypass Cabin          1.0    NaN Broad Gauge           2.0        1.0           Southern Railway
    32406  48875.0       Khairar Junction       Khairar Junction        104.0    NaN Broad Gauge          12.0        1.0      North Central Railway
    57849  39951.0            Ahiran Halt            Ahiran Halt          7.0   

In [142]:
print("=== SELF-LOOP GEOMETRY CHECK ===")

self_loop_geo = network_geo[
    network_geo["fromjunction"].notna() &
    network_geo["tojunction"].notna() &
    (network_geo["fromjunction"] == network_geo["tojunction"])
].copy()

self_loop_geo["geometry_length_km"] = (
    self_loop_geo.geometry.to_crs(epsg=32643).length / 1000
)

print(
    self_loop_geo[
        [
            "objectid",
            "id",
            "fromjunction",
            "tojunction",
            "tracklength",
            "geometry_length_km",
            "type",
            "noofstations",
            "nooflanes",
            "railwayzone"
        ]
    ].to_string(index=False)
)

=== SELF-LOOP GEOMETRY CHECK ===
 objectid       id           fromjunction             tojunction  tracklength  geometry_length_km        type  noofstations  nooflanes                railwayzone
     3502 110218.0 Magnesite Bypass Cabin Magnesite Bypass Cabin          1.0            0.025490 Broad Gauge           2.0        1.0           Southern Railway
     3764  11226.0 Magnesite Bypass Cabin Magnesite Bypass Cabin          1.0            0.313392 Broad Gauge           2.0        1.0           Southern Railway
     7251 108977.0 Magnesite Bypass Cabin Magnesite Bypass Cabin          1.0            0.428273 Broad Gauge           2.0        1.0           Southern Railway
     7253 108977.0 Magnesite Bypass Cabin Magnesite Bypass Cabin          1.0            0.251414 Broad Gauge           2.0        1.0           Southern Railway
    32406  48875.0       Khairar Junction       Khairar Junction        104.0            0.208181 Broad Gauge          12.0        1.0      North Central Rai

In [143]:
print("=== SELF-LOOP GEOMETRY TYPES ===")
print(self_loop_geo.geometry.geom_type.value_counts())

=== SELF-LOOP GEOMETRY TYPES ===
LineString    12
Name: count, dtype: int64


In [144]:
print("=== SELF-LOOP SPATIAL EXTENT ===")

for _, row in self_loop_geo.iterrows():
    print(
        row["fromjunction"],
        "|",
        "geometry length:",
        round(row["geometry_length_km"], 3),
        "km |",
        "tracklength:",
        row["tracklength"],
        "km"
    )

=== SELF-LOOP SPATIAL EXTENT ===
Magnesite Bypass Cabin | geometry length: 0.025 km | tracklength: 1.0 km
Magnesite Bypass Cabin | geometry length: 0.313 km | tracklength: 1.0 km
Magnesite Bypass Cabin | geometry length: 0.428 km | tracklength: 1.0 km
Magnesite Bypass Cabin | geometry length: 0.251 km | tracklength: 1.0 km
Khairar Junction | geometry length: 0.208 km | tracklength: 104.0 km
Ahiran Halt | geometry length: 0.654 km | tracklength: 7.0 km
Ahiran Halt | geometry length: 0.049 km | tracklength: 7.0 km
Ahiran Halt | geometry length: 2.324 km | tracklength: 7.0 km
Pipar Road Junction | geometry length: 0.427 km | tracklength: 56.0 km
Biratnagar | geometry length: 8.684 km | tracklength: 10.0 km
Biratnagar | geometry length: 0.096 km | tracklength: 10.0 km
Biratnagar | geometry length: 1.846 km | tracklength: 10.0 km


In [145]:
print("=== JUNCTION COMPLETENESS ===")

print("Total segments:", len(network_clean))

print(
    "Missing fromjunction:",
    network_clean["fromjunction"].isna().sum()
)

print(
    "Missing tojunction:",
    network_clean["tojunction"].isna().sum()
)

print(
    "Missing both:",
    (
        network_clean["fromjunction"].isna() &
        network_clean["tojunction"].isna()
    ).sum()
)

print(
    "Missing either:",
    (
        network_clean["fromjunction"].isna() |
        network_clean["tojunction"].isna()
    ).sum()
)

print(
    "Complete endpoints:",
    (
        network_clean["fromjunction"].notna() &
        network_clean["tojunction"].notna()
    ).sum()
)

=== JUNCTION COMPLETENESS ===
Total segments: 119446
Missing fromjunction: 1004
Missing tojunction: 1004
Missing both: 1004
Missing either: 1004
Complete endpoints: 118442


In [146]:
print("=== ENDPOINT COMPLETENESS PATTERN ===")

endpoint_pattern = pd.Series(
    np.select(
        [
            network_clean["fromjunction"].isna() &
            network_clean["tojunction"].isna(),
            network_clean["fromjunction"].isna(),
            network_clean["tojunction"].isna()
        ],
        [
            "both_missing",
            "from_missing",
            "to_missing"
        ],
        default="complete"
    ),
    name="endpoint_status"
)

print(endpoint_pattern.value_counts())

=== ENDPOINT COMPLETENESS PATTERN ===
endpoint_status
complete        118442
both_missing      1004
Name: count, dtype: int64


In [147]:
valid_network = network_clean.dropna(
    subset=["fromjunction", "tojunction"]
).copy()

corridor_qa = (
    valid_network
    .groupby(["fromjunction", "tojunction"], sort=False)
    .agg(
        segment_count=("objectid", "size"),

        tracklength_unique=("tracklength", "nunique"),
        speed_unique=("speed", "nunique"),
        type_unique=("type", "nunique"),
        noofstations_unique=("noofstations", "nunique"),
        nooflanes_unique=("nooflanes", "nunique"),
        railwayzone_unique=("railwayzone_clean", "nunique"),

        tracklength_missing=("tracklength", lambda x: x.isna().sum()),
        speed_missing=("speed", lambda x: x.isna().sum()),
        type_missing=("type", lambda x: x.isna().sum()),
        noofstations_missing=("noofstations", lambda x: x.isna().sum()),
        nooflanes_missing=("nooflanes", lambda x: x.isna().sum()),
        railwayzone_missing=("railwayzone_clean", lambda x: x.isna().sum())
    )
)

print("=== CORRIDOR QA TABLE ===")
print("Shape:", corridor_qa.shape)
print(corridor_qa.head())

=== CORRIDOR QA TABLE ===
Shape: (2305, 13)
                                                   segment_count  \
fromjunction               tojunction                              
Tirunelveli Junction       Tenkasi Junction                  177   
Thiruvananthapuram Central Nagercoil Bypass Cabin            107   
Virudunagar Junction       Tenkasi Junction                  316   
Edamann                    Bhagavathipuram                    56   
Bhagavathipuram            Tenkasi Junction                   18   

                                                   tracklength_unique  \
fromjunction               tojunction                                   
Tirunelveli Junction       Tenkasi Junction                         1   
Thiruvananthapuram Central Nagercoil Bypass Cabin                   1   
Virudunagar Junction       Tenkasi Junction                         1   
Edamann                    Bhagavathipuram                          3   
Bhagavathipuram            Tenkasi Juncti

In [148]:
for col in [
    "tracklength",
    "speed",
    "type",
    "noofstations",
    "nooflanes",
    "railwayzone"
]:
    unique_col = f"{col}_unique"
    corridor_qa[f"{col}_conflict"] = corridor_qa[unique_col] > 1

corridor_qa["any_attribute_conflict"] = (
    corridor_qa[
        [
            "tracklength_conflict",
            "speed_conflict",
            "type_conflict",
            "noofstations_conflict",
            "nooflanes_conflict",
            "railwayzone_conflict"
        ]
    ]
    .any(axis=1)
)

print("=== FINAL CORRIDOR QA STATUS ===")
print(
    corridor_qa["any_attribute_conflict"]
    .value_counts()
)

=== FINAL CORRIDOR QA STATUS ===
any_attribute_conflict
False    2104
True      201
Name: count, dtype: int64


In [149]:
def mode_or_nan(series):
    s = series.dropna()

    if len(s) == 0:
        return np.nan

    return s.mode().iloc[0]


corridor_attributes = (
    valid_network
    .groupby(["fromjunction", "tojunction"], sort=False)
    .agg(
        segment_count=("objectid", "size"),

        tracklength=("tracklength", mode_or_nan),
        speed=("speed", mode_or_nan),
        type=("type", mode_or_nan),
        noofstations=("noofstations", mode_or_nan),
        nooflanes=("nooflanes", mode_or_nan),
        railwayzone=("railwayzone_clean", mode_or_nan),

        tracklength_conflict=("tracklength", lambda x: x.nunique(dropna=True) > 1),
        speed_conflict=("speed", lambda x: x.nunique(dropna=True) > 1),
        type_conflict=("type", lambda x: x.nunique(dropna=True) > 1),
        noofstations_conflict=("noofstations", lambda x: x.nunique(dropna=True) > 1),
        nooflanes_conflict=("nooflanes", lambda x: x.nunique(dropna=True) > 1),
        railwayzone_conflict=("railwayzone_clean", lambda x: x.nunique(dropna=True) > 1),

        tracklength_missing=("tracklength", lambda x: x.isna().sum()),
        speed_missing=("speed", lambda x: x.isna().sum()),
        type_missing=("type", lambda x: x.isna().sum()),
        noofstations_missing=("noofstations", lambda x: x.isna().sum()),
        nooflanes_missing=("nooflanes", lambda x: x.isna().sum()),
        railwayzone_missing=("railwayzone_clean", lambda x: x.isna().sum())
    )
)

corridor_attributes["any_attribute_conflict"] = (
    corridor_attributes[
        [
            "tracklength_conflict",
            "speed_conflict",
            "type_conflict",
            "noofstations_conflict",
            "nooflanes_conflict",
            "railwayzone_conflict"
        ]
    ].any(axis=1)
)

print("=== CORRIDOR ATTRIBUTE TABLE ===")
print("Shape:", corridor_attributes.shape)
print(corridor_attributes.head())

=== CORRIDOR ATTRIBUTE TABLE ===
Shape: (2305, 20)
                                                   segment_count  tracklength  \
fromjunction               tojunction                                           
Tirunelveli Junction       Tenkasi Junction                  177         72.0   
Thiruvananthapuram Central Nagercoil Bypass Cabin            107         70.0   
Virudunagar Junction       Tenkasi Junction                  316        122.0   
Edamann                    Bhagavathipuram                    56         31.0   
Bhagavathipuram            Tenkasi Junction                   18         14.0   

                                                   speed         type  \
fromjunction               tojunction                                   
Tirunelveli Junction       Tenkasi Junction        110.0  Broad Gauge   
Thiruvananthapuram Central Nagercoil Bypass Cabin   80.0  Broad Gauge   
Virudunagar Junction       Tenkasi Junction        100.0  Broad Gauge   
Edamann         

In [150]:
print("=== REPRESENTATIVE VALUE COVERAGE ===")

for col in [
    "tracklength",
    "speed",
    "type",
    "noofstations",
    "nooflanes",
    "railwayzone"
]:
    print(
        f"{col}:",
        "missing representative values =",
        corridor_attributes[col].isna().sum()
    )

=== REPRESENTATIVE VALUE COVERAGE ===
tracklength: missing representative values = 0
speed: missing representative values = 818
type: missing representative values = 1
noofstations: missing representative values = 0
nooflanes: missing representative values = 53
railwayzone: missing representative values = 48


In [151]:
print("=== CONFLICTING CORRIDORS: REPRESENTATIVE VALUES ===")

print(
    corridor_attributes[
        corridor_attributes["any_attribute_conflict"]
    ][
        [
            "segment_count",
            "tracklength",
            "speed",
            "type",
            "noofstations",
            "nooflanes",
            "railwayzone",
            "tracklength_conflict",
            "speed_conflict",
            "type_conflict",
            "noofstations_conflict",
            "nooflanes_conflict",
            "railwayzone_conflict"
        ]
    ]
    .sort_values("segment_count", ascending=False)
    .head(20)
)

=== CONFLICTING CORRIDORS: REPRESENTATIVE VALUES ===
                                             segment_count  tracklength  \
fromjunction          tojunction                                          
Chanderiya            Gurla West Cabin                 545        156.0   
Itwari Junction       Chhindwara Junction              499        147.0   
Golakganj Junction    Bilasipara                       443         49.0   
Marwar Junction       Palanpur Junction                419        217.0   
Mavli Junction        Bari Sadri                       398         82.0   
Kurduwadi Junction    Latur Road Junction              373        185.0   
Samdari Junction      Bhildi Junction                  365        224.0   
Duvvada               Samalkot Bypass Cabin            355        131.0   
Khairar Junction      Ohan Cabin                       349        104.0   
Vizianagaram Junction Naupada Junction                 348        117.0   
Yerraguntla Junction  Gooty Cabin              

In [152]:
print("=== CORRIDORS WITH HIGH SEGMENT COUNTS AND CONFLICTS ===")

print(
    corridor_attributes[
        corridor_attributes["any_attribute_conflict"] &
        (corridor_attributes["segment_count"] >= 100)
    ][
        [
            "segment_count",
            "tracklength",
            "speed",
            "type",
            "noofstations",
            "nooflanes",
            "railwayzone",
            "tracklength_conflict",
            "speed_conflict",
            "type_conflict",
            "noofstations_conflict",
            "nooflanes_conflict",
            "railwayzone_conflict"
        ]
    ]
    .sort_values("segment_count", ascending=False)
    .head(30)
)

=== CORRIDORS WITH HIGH SEGMENT COUNTS AND CONFLICTS ===
                                                             segment_count  \
fromjunction           tojunction                                            
Chanderiya             Gurla West Cabin                                545   
Itwari Junction        Chhindwara Junction                             499   
Golakganj Junction     Bilasipara                                      443   
Marwar Junction        Palanpur Junction                               419   
Mavli Junction         Bari Sadri                                      398   
Kurduwadi Junction     Latur Road Junction                             373   
Samdari Junction       Bhildi Junction                                 365   
Duvvada                Samalkot Bypass Cabin                           355   
Khairar Junction       Ohan Cabin                                      349   
Vizianagaram Junction  Naupada Junction                                348   
Yerragu

In [153]:
print("=== ZERO-VALUE CORRIDOR AUDIT ===")

zero_checks = {
    "tracklength": "tracklength_zero",
    "speed": "speed_zero",
    "noofstations": "noofstations_zero",
    "nooflanes": "nooflanes_zero"
}

for attribute, flag in zero_checks.items():

    temp = (
        valid_network
        .groupby(["fromjunction", "tojunction"])[flag]
        .sum()
    )

    print(
        f"{attribute}:",
        "corridors containing zero values =",
        (temp > 0).sum(),
        "| zero-valued segments =",
        temp.sum()
    )

=== ZERO-VALUE CORRIDOR AUDIT ===
tracklength: corridors containing zero values = 7 | zero-valued segments = 632
speed: corridors containing zero values = 346 | zero-valued segments = 13086
noofstations: corridors containing zero values = 4 | zero-valued segments = 94
nooflanes: corridors containing zero values = 5 | zero-valued segments = 53


In [154]:
speed_zero_distribution = (
    valid_network
    .groupby(["fromjunction", "tojunction"])["speed"]
    .agg(
        total=lambda x: x.notna().sum(),
        zero=lambda x: (x == 0).sum(),
        positive=lambda x: (x > 0).sum()
    )
)

speed_zero_distribution["zero_share"] = (
    speed_zero_distribution["zero"] /
    speed_zero_distribution["total"]
)

print("=== SPEED ZERO DISTRIBUTION ===")
print(speed_zero_distribution.describe())

print("\nCorridors where zero is the only non-null speed:")
print(
    speed_zero_distribution[
        (speed_zero_distribution["zero"] > 0) &
        (speed_zero_distribution["positive"] == 0)
    ].head(20)
)

=== SPEED ZERO DISTRIBUTION ===
             total         zero     positive   zero_share
count  2305.000000  2305.000000  2305.000000  1487.000000
mean     33.708026     5.677223    28.030803     0.230144
std      71.714660    30.753824    67.187357     0.420240
min       0.000000     0.000000     0.000000     0.000000
25%       0.000000     0.000000     0.000000     0.000000
50%       4.000000     0.000000     0.000000     0.000000
75%      35.000000     0.000000    27.000000     0.000000
max     711.000000   513.000000   711.000000     1.000000

Corridors where zero is the only non-null speed:
                                             total  zero  positive  zero_share
fromjunction         tojunction                                               
Achalpur             Murtizapur Junction NG     37    37         0         1.0
Adarki               Palsi                      30    30         0         1.0
Adarshnagar          Daurai Chord Cabin          9     9         0         1.0
A

In [155]:
print("=== ZERO-VALUE ATTRIBUTE DETAILS ===")

for attribute in ["tracklength", "speed", "noofstations", "nooflanes"]:

    stats = (
        valid_network
        .groupby(["fromjunction", "tojunction"])[attribute]
        .agg(
            total=lambda x: x.notna().sum(),
            zero=lambda x: (x == 0).sum(),
            positive=lambda x: (x > 0).sum()
        )
    )

    stats["zero_share"] = (
        stats["zero"] / stats["total"]
    )

    print(f"\n--- {attribute.upper()} ---")
    print(stats.describe())
    
    print("\nCorridors where zero is the only non-null value:")
    print(
        stats[
            (stats["zero"] > 0) &
            (stats["positive"] == 0)
        ]
        .sort_values("zero", ascending=False)
        .head(15)
    )

=== ZERO-VALUE ATTRIBUTE DETAILS ===

--- TRACKLENGTH ---
             total         zero     positive   zero_share
count  2305.000000  2305.000000  2305.000000  2305.000000
mean     51.381345     0.274187    51.107158     0.001764
std      85.541849    11.282298    84.958932     0.041638
min       1.000000     0.000000     0.000000     0.000000
25%       4.000000     0.000000     4.000000     0.000000
50%      19.000000     0.000000    19.000000     0.000000
75%      60.000000     0.000000    59.000000     0.000000
max     711.000000   534.000000   711.000000     1.000000

Corridors where zero is the only non-null value:
                                                  total  zero  positive  \
fromjunction            tojunction                                        
Akola Junction          Purna Junction              534   534         0   
Pulgaon Junction        Arvi                         91    91         0   
Toranagallu Chord Cabin Toranagallu Bypass Cabin      3     3         

In [156]:
for attribute in ["tracklength", "noofstations", "nooflanes"]:

    zero_only = (
        valid_network
        .groupby(["fromjunction", "tojunction"])[attribute]
        .agg(
            total=lambda x: x.notna().sum(),
            zero=lambda x: (x == 0).sum(),
            positive=lambda x: (x > 0).sum()
        )
    )

    zero_only["zero_share"] = (
        zero_only["zero"] / zero_only["total"]
    )

    print(f"\n=== {attribute.upper()} ZERO-ONLY CORRIDORS ===")
    print(
        zero_only[
            (zero_only["zero"] > 0) &
            (zero_only["positive"] == 0)
        ]
        .sort_values("zero", ascending=False)
        .head(20)
    )


=== TRACKLENGTH ZERO-ONLY CORRIDORS ===
                                                  total  zero  positive  \
fromjunction            tojunction                                        
Akola Junction          Purna Junction              534   534         0   
Pulgaon Junction        Arvi                         91    91         0   
Toranagallu Chord Cabin Toranagallu Bypass Cabin      3     3         0   
Jalamb Outer C Cabin    Jalamb Outer A Cabin          1     1         0   

                                                  zero_share  
fromjunction            tojunction                            
Akola Junction          Purna Junction                   1.0  
Pulgaon Junction        Arvi                             1.0  
Toranagallu Chord Cabin Toranagallu Bypass Cabin         1.0  
Jalamb Outer C Cabin    Jalamb Outer A Cabin             1.0  

=== NOOFSTATIONS ZERO-ONLY CORRIDORS ===
                             total  zero  positive  zero_share
fromjunction     tojuncti

In [157]:
# ============================================================
# TASK 62T — NORMALIZE ZERO VALUES
# ============================================================

network_for_corridor = valid_network.copy()

zero_to_nan = [
    "tracklength",
    "speed",
    "noofstations",
    "nooflanes"
]

for col in zero_to_nan:
    network_for_corridor.loc[
        network_for_corridor[col] == 0, col
    ] = np.nan

print("=== ZERO VALUES AFTER NORMALIZATION ===")

for col in zero_to_nan:
    print(
        col,
        "remaining zeros =",
        (network_for_corridor[col] == 0).sum(),
        "| missing values =",
        network_for_corridor[col].isna().sum()
    )

=== ZERO VALUES AFTER NORMALIZATION ===
tracklength remaining zeros = 0 | missing values = 640
speed remaining zeros = 0 | missing values = 53831
noofstations remaining zeros = 0 | missing values = 102
nooflanes remaining zeros = 0 | missing values = 1273


In [158]:
# ============================================================
# REBUILD CORRIDOR ATTRIBUTES
# ============================================================

corridor_attributes = (
    network_for_corridor
    .groupby(
        ["fromjunction", "tojunction"],
        sort=False
    )
    .agg(
        segment_count=("objectid", "size"),

        tracklength=("tracklength", mode_or_nan),
        speed=("speed", mode_or_nan),
        type=("type", mode_or_nan),
        noofstations=("noofstations", mode_or_nan),
        nooflanes=("nooflanes", mode_or_nan),
        railwayzone=("railwayzone_clean", mode_or_nan),

        tracklength_conflict=(
            "tracklength",
            lambda x: x.nunique(dropna=True) > 1
        ),

        speed_conflict=(
            "speed",
            lambda x: x.nunique(dropna=True) > 1
        ),

        type_conflict=(
            "type",
            lambda x: x.nunique(dropna=True) > 1
        ),

        noofstations_conflict=(
            "noofstations",
            lambda x: x.nunique(dropna=True) > 1
        ),

        nooflanes_conflict=(
            "nooflanes",
            lambda x: x.nunique(dropna=True) > 1
        ),

        railwayzone_conflict=(
            "railwayzone_clean",
            lambda x: x.nunique(dropna=True) > 1
        ),

        tracklength_missing=(
            "tracklength",
            lambda x: x.isna().sum()
        ),

        speed_missing=(
            "speed",
            lambda x: x.isna().sum()
        ),

        type_missing=(
            "type",
            lambda x: x.isna().sum()
        ),

        noofstations_missing=(
            "noofstations",
            lambda x: x.isna().sum()
        ),

        nooflanes_missing=(
            "nooflanes",
            lambda x: x.isna().sum()
        ),

        railwayzone_missing=(
            "railwayzone_clean",
            lambda x: x.isna().sum()
        )
    )
)

corridor_attributes["any_attribute_conflict"] = (
    corridor_attributes[
        [
            "tracklength_conflict",
            "speed_conflict",
            "type_conflict",
            "noofstations_conflict",
            "nooflanes_conflict",
            "railwayzone_conflict"
        ]
    ].any(axis=1)
)

print("=== CORRIDOR ATTRIBUTES ===")
print("Shape:", corridor_attributes.shape)
print(
    "Unique corridors:",
    len(corridor_attributes)
)

=== CORRIDOR ATTRIBUTES ===
Shape: (2305, 20)
Unique corridors: 2305


In [159]:
# ============================================================
# TASK 62U — FINAL CORRIDOR QA
# ============================================================

print("=== FINAL CORRIDOR QA ===")

# ------------------------------------------------------------
# 1. REPRESENTATIVE VALUE MISSINGNESS
# ------------------------------------------------------------

representative_columns = [
    "tracklength",
    "speed",
    "type",
    "noofstations",
    "nooflanes",
    "railwayzone"
]

print("\n--- REPRESENTATIVE VALUE MISSINGNESS ---")

for col in representative_columns:
    print(
        f"{col}:",
        corridor_attributes[col].isna().sum()
    )


# ------------------------------------------------------------
# 2. CONFLICT SUMMARY
# ------------------------------------------------------------

conflict_columns = [
    "tracklength_conflict",
    "speed_conflict",
    "type_conflict",
    "noofstations_conflict",
    "nooflanes_conflict",
    "railwayzone_conflict"
]

print("\n--- CONFLICT SUMMARY ---")

for col in conflict_columns:
    print(
        f"{col}:",
        corridor_attributes[col].sum()
    )

print(
    "\nAny attribute conflict:",
    corridor_attributes["any_attribute_conflict"].sum()
)


# ------------------------------------------------------------
# 3. REPRESENTATIVE VALUE MUST EXIST IN SOURCE
# ------------------------------------------------------------

print("\n--- REPRESENTATIVE VALUE VALIDATION ---")

for col in representative_columns:

    source_values = (
        network_for_corridor
        .groupby(["fromjunction", "tojunction"])[col]
        .apply(lambda x: set(x.dropna()))
    )

    rep_values = corridor_attributes[col]

    invalid = []

    for corridor, representative in rep_values.items():

        if pd.isna(representative):
            continue

        if representative not in source_values.loc[corridor]:
            invalid.append(corridor)

    print(
        f"{col}: invalid representative values =",
        len(invalid)
    )


# ------------------------------------------------------------
# 4. SELF-LOOP CORRIDORS
# ------------------------------------------------------------

self_loop_mask = (
    corridor_attributes.index.get_level_values("fromjunction")
    ==
    corridor_attributes.index.get_level_values("tojunction")
)

print(
    "\nSelf-loop corridors:",
    self_loop_mask.sum()
)

print("\n--- SELF-LOOP CORRIDORS ---")

print(
    corridor_attributes.loc[self_loop_mask]
)

=== FINAL CORRIDOR QA ===

--- REPRESENTATIVE VALUE MISSINGNESS ---
tracklength: 4
speed: 1155
type: 1
noofstations: 1
nooflanes: 57
railwayzone: 48

--- CONFLICT SUMMARY ---
tracklength_conflict: 54
speed_conflict: 14
type_conflict: 11
noofstations_conflict: 21
nooflanes_conflict: 59
railwayzone_conflict: 59

Any attribute conflict: 192

--- REPRESENTATIVE VALUE VALIDATION ---
tracklength: invalid representative values = 0
speed: invalid representative values = 0
type: invalid representative values = 0
noofstations: invalid representative values = 0
nooflanes: invalid representative values = 0
railwayzone: invalid representative values = 129

Self-loop corridors: 5

--- SELF-LOOP CORRIDORS ---
                                               segment_count  tracklength  \
fromjunction           tojunction                                           
Magnesite Bypass Cabin Magnesite Bypass Cabin              4          1.0   
Khairar Junction       Khairar Junction                    1     

In [160]:
# ============================================================
# TASK 62V — RAILWAY ZONE REPRESENTATIVE VALIDATION
# ============================================================

zone_source_values = (
    network_for_corridor
    .groupby(["fromjunction", "tojunction"])["railwayzone_clean"]
    .apply(lambda x: set(x.dropna()))
)

invalid_zone_corridors = []

for corridor, representative in corridor_attributes["railwayzone"].items():

    if pd.isna(representative):
        continue

    source_values = zone_source_values.loc[corridor]

    if representative not in source_values:
        invalid_zone_corridors.append(
            (
                corridor[0],
                corridor[1],
                representative,
                source_values
            )
        )

print("Invalid railway-zone corridors:", len(invalid_zone_corridors))

for row in invalid_zone_corridors[:20]:
    print("\nFROM:", row[0])
    print("TO:", row[1])
    print("Representative:", repr(row[2]))
    print("Source values:", [repr(x) for x in row[3]])

Invalid railway-zone corridors: 0


In [161]:
corridor_attributes["is_self_loop"] = (
    corridor_attributes.index.get_level_values("fromjunction")
    ==
    corridor_attributes.index.get_level_values("tojunction")
)

In [162]:
print("=== CORRIDOR LAYER FINAL CHECK ===")

print("Shape:", corridor_attributes.shape)
print("Total corridors:", len(corridor_attributes))
print("Self-loop corridors:", corridor_attributes["is_self_loop"].sum())
print(
    "Corridors with attribute conflicts:",
    corridor_attributes["any_attribute_conflict"].sum()
)

=== CORRIDOR LAYER FINAL CHECK ===
Shape: (2305, 21)
Total corridors: 2305
Self-loop corridors: 5
Corridors with attribute conflicts: 192


In [163]:
# ============================================================
# TASK 63 — CORRIDOR ENDPOINT ↔ STATION MASTER MATCHING
# ============================================================

station_names = set(
    final_stations["station_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

corridor_endpoints = set(
    corridor_attributes.index
    .get_level_values("fromjunction")
    .astype(str)
    .str.strip()
    .str.upper()
).union(
    set(
        corridor_attributes.index
        .get_level_values("tojunction")
        .astype(str)
        .str.strip()
        .str.upper()
    )
)

matched_endpoints = corridor_endpoints.intersection(station_names)

print("=== CORRIDOR ENDPOINT MATCHING ===")
print("Unique corridor endpoints:", len(corridor_endpoints))
print("Matched to station master:", len(matched_endpoints))
print("Unmatched:", len(corridor_endpoints - matched_endpoints))

=== CORRIDOR ENDPOINT MATCHING ===
Unique corridor endpoints: 2026
Matched to station master: 716
Unmatched: 1310


In [164]:
unmatched_endpoints = sorted(
    corridor_endpoints - matched_endpoints
)

print("\n=== UNMATCHED CORRIDOR ENDPOINTS ===")
print("Count:", len(unmatched_endpoints))

for name in unmatched_endpoints[:100]:
    print(name)


=== UNMATCHED CORRIDOR ENDPOINTS ===
Count: 1310
ABHAYAPURI
ABOHAR JUNCTION
ACHHNERA JUNCTION
ADIPUR JUNCTION
ADRA JUNCTION
AFK ANGAMALI
AGASOD JUNCTION
AGASTIYAMPALL
AGASTIYAMPALLI
AGRA CANTT.
AHIRAN HALT
AHMADPUR JUNCTION
AHMEDABAD JUNCTION
AHMEDNAGAR JUNCTION
AIR FORCE SIDING, AMLA
AISHBAGH JUNCTION
AIT JUNCTION
AJAYGARH
AJMER JUNCTION
AKANAPET JUNCTION
AKBARPUR JUNCTION
AKOLA JUNCTION
ALAMBAGH WEST CABIN
ALAULI
ALIGARH JUNCTION
ALIPUR DUAR JUNCTION
ALIRAJPUR
ALNAVAR JUNCTION
ALUABARI ROAD JUNCTION
ALWAR BYPASS CABIN
ALWAR CHORD CABIN
ALWAR OUTER CABIN
AMARAVATHI COLONY JUNCTION
AMBALA CANTT.
AMBALA CANTT. JUNCTION
AMBALAPPUZHA
AMBATURI
AMBEWADI
AMBLIYASAN JUNCTION
AMGURI JUNCTION
AMLA JUNCTION
AMRAVATI (TERMINAL)
AMRAVATI MIDC CABIN
AMRAVATI MIDC FREIGHT YARD
AMRITSAR JUNCTION
AMTA
ANAND BYPASS CABIN
ANAND CHORD CABIN
ANAND JUNCTION
ANAND NAGAR JUNCTION
ANDAL JUNCTION
ANKLESHWAR JUNCTION
ANKORHA
ANNA NAGAR
ANTAGARH
ANUPPUR JUNCTION
ARA JUNCTION
ARAKKONAM JUNCTION
ARAKKONAM NORTH C

In [165]:
# ============================================================
# TASK 64 — ENDPOINT NAME NORMALIZATION AUDIT
# ============================================================

def normalize_station_name(name):
    if pd.isna(name):
        return ""
    
    name = str(name).strip().upper()

    # Normalize punctuation
    name = name.replace(".", "")
    name = name.replace(",", "")
    
    # Normalize whitespace
    name = " ".join(name.split())

    return name


network_endpoint_df = pd.DataFrame({
    "endpoint_name": list(corridor_endpoints)
})

station_name_df = pd.DataFrame({
    "station_name": list(station_names)
})

network_endpoint_df["normalized"] = (
    network_endpoint_df["endpoint_name"]
    .apply(normalize_station_name)
)

station_name_df["normalized"] = (
    station_name_df["station_name"]
    .apply(normalize_station_name)
)

normalized_station_names = set(
    station_name_df["normalized"]
)

network_endpoint_df["normalized_exact_match"] = (
    network_endpoint_df["normalized"]
    .isin(normalized_station_names)
)

print("=== NORMALIZATION AUDIT ===")

print(
    "Previously unmatched:",
    len(unmatched_endpoints)
)

print(
    "Matched after basic normalization:",
    network_endpoint_df["normalized_exact_match"].sum()
)

print(
    "Still unmatched:",
    (~network_endpoint_df["normalized_exact_match"]).sum()
)

=== NORMALIZATION AUDIT ===
Previously unmatched: 1310
Matched after basic normalization: 719
Still unmatched: 1307


In [166]:
still_unmatched = (
    network_endpoint_df[
        ~network_endpoint_df["normalized_exact_match"]
    ]
    ["endpoint_name"]
    .sort_values()
    .tolist()
)

print("\n=== STILL UNMATCHED ===")
print("Count:", len(still_unmatched))

for name in still_unmatched[:150]:
    print(name)


=== STILL UNMATCHED ===
Count: 1307
ABHAYAPURI
ABOHAR JUNCTION
ACHHNERA JUNCTION
ADIPUR JUNCTION
ADRA JUNCTION
AFK ANGAMALI
AGASOD JUNCTION
AGASTIYAMPALL
AGASTIYAMPALLI
AHIRAN HALT
AHMADPUR JUNCTION
AHMEDABAD JUNCTION
AHMEDNAGAR JUNCTION
AIR FORCE SIDING, AMLA
AISHBAGH JUNCTION
AIT JUNCTION
AJAYGARH
AJMER JUNCTION
AKANAPET JUNCTION
AKBARPUR JUNCTION
AKOLA JUNCTION
ALAMBAGH WEST CABIN
ALAULI
ALIGARH JUNCTION
ALIPUR DUAR JUNCTION
ALIRAJPUR
ALNAVAR JUNCTION
ALUABARI ROAD JUNCTION
ALWAR BYPASS CABIN
ALWAR CHORD CABIN
ALWAR OUTER CABIN
AMARAVATHI COLONY JUNCTION
AMBALA CANTT.
AMBALA CANTT. JUNCTION
AMBALAPPUZHA
AMBATURI
AMBEWADI
AMBLIYASAN JUNCTION
AMGURI JUNCTION
AMLA JUNCTION
AMRAVATI (TERMINAL)
AMRAVATI MIDC CABIN
AMRAVATI MIDC FREIGHT YARD
AMRITSAR JUNCTION
AMTA
ANAND BYPASS CABIN
ANAND CHORD CABIN
ANAND JUNCTION
ANAND NAGAR JUNCTION
ANDAL JUNCTION
ANKLESHWAR JUNCTION
ANKORHA
ANNA NAGAR
ANTAGARH
ANUPPUR JUNCTION
ARA JUNCTION
ARAKKONAM JUNCTION
ARAKKONAM NORTH CABIN
ARAMBAGH (PRAFULLA C

In [167]:
from pathlib import Path

# Search common project locations
search_roots = [
    Path.cwd(),
    Path.home() / "Desktop",
]

matches = []

for root in search_roots:
    if root.exists():
        matches.extend(root.rglob("Railway_Network_.geojson"))

# Remove duplicates
matches = list(dict.fromkeys(matches))

print("=== GEOJSON FILE SEARCH ===")
print("Files found:", len(matches))

for i, path in enumerate(matches):
    print(i, path)

=== GEOJSON FILE SEARCH ===
Files found: 1
0 C:\Users\vinja\Desktop\Rail-Intell\Railway_Network_.geojson


In [168]:
network_geo_path = matches[0]

print("Loading:", network_geo_path)

network_geo = gpd.read_file(network_geo_path)

print("Loaded successfully")
print("Shape:", network_geo.shape)
print("CRS:", network_geo.crs)

Loading: C:\Users\vinja\Desktop\Rail-Intell\Railway_Network_.geojson


C:\Users\vinja\Projects\llm_engineering\.venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Several features with id = 80732 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


Loaded successfully
Shape: (119446, 16)
CRS: EPSG:4326


In [169]:
# ============================================================
# TASK 65A — DERIVE NETWORK CORRIDOR ENDPOINT COORDINATES
# ============================================================

print("Network CRS:", network_geo.crs)

print("\nGeometry types:")
print(network_geo.geometry.geom_type.value_counts())

network_geo_valid = network_geo.dropna(
    subset=["fromjunction", "tojunction"]
).copy()

network_geo_valid["start_point"] = (
    network_geo_valid.geometry.apply(
        lambda g: g.coords[0]
        if g.geom_type == "LineString"
        else None
    )
)

network_geo_valid["end_point"] = (
    network_geo_valid.geometry.apply(
        lambda g: g.coords[-1]
        if g.geom_type == "LineString"
        else None
    )
)

print(
    "\nValid endpoint-labelled GIS segments:",
    len(network_geo_valid)
)

print(
    "Segments with usable start/end points:",
    network_geo_valid[
        network_geo_valid["start_point"].notna()
        & network_geo_valid["end_point"].notna()
    ].shape[0]
)

Network CRS: EPSG:4326

Geometry types:
LineString         119443
MultiLineString         3
Name: count, dtype: int64

Valid endpoint-labelled GIS segments: 118442
Segments with usable start/end points: 118439


In [170]:
# ============================================================
# TASK 65A — DERIVE NETWORK CORRIDOR ENDPOINT COORDINATES
# ============================================================

print("Network CRS:", network_geo.crs)

print("\nGeometry types:")
print(network_geo.geometry.geom_type.value_counts())

network_geo_valid = network_geo.dropna(
    subset=["fromjunction", "tojunction"]
).copy()

# Extract start/end coordinates
network_geo_valid["start_point"] = (
    network_geo_valid.geometry.apply(
        lambda g: g.coords[0]
        if g.geom_type == "LineString"
        else None
    )
)

network_geo_valid["end_point"] = (
    network_geo_valid.geometry.apply(
        lambda g: g.coords[-1]
        if g.geom_type == "LineString"
        else None
    )
)

print(
    "\nValid endpoint-labelled GIS segments:",
    len(network_geo_valid)
)

print(
    "Segments with usable start/end points:",
    network_geo_valid[
        network_geo_valid["start_point"].notna()
        & network_geo_valid["end_point"].notna()
    ].shape[0]
)

Network CRS: EPSG:4326

Geometry types:
LineString         119443
MultiLineString         3
Name: count, dtype: int64

Valid endpoint-labelled GIS segments: 118442
Segments with usable start/end points: 118439


In [171]:
from pathlib import Path

matches = list(
    Path.home().rglob("Railway_Network_.geojson")
)

print("Found files:")
for path in matches:
    print(path)

Found files:
C:\Users\vinja\Desktop\Rail-Intell\Railway_Network_.geojson


In [172]:
# ============================================================
# TASK 65B — CORRIDOR-LEVEL GEOMETRY ENDPOINTS
# ============================================================

network_lines = network_geo[
    (network_geo["fromjunction"].notna()) &
    (network_geo["tojunction"].notna()) &
    (network_geo.geometry.geom_type == "LineString")
].copy()

corridor_groups = network_lines.groupby(
    ["fromjunction", "tojunction"],
    sort=False
)

print("Valid GIS LineString segments:", len(network_lines))
print("Unique GIS corridors:", len(corridor_groups))

Valid GIS LineString segments: 118439
Unique GIS corridors: 2305


In [173]:
# ============================================================
# TASK 65C — CORRIDOR GEOMETRY CONNECTIVITY SAMPLE
# ============================================================

sample_corridors = list(corridor_groups.groups.keys())[:20]

for from_junction, to_junction in sample_corridors:

    group = corridor_groups.get_group(
        (from_junction, to_junction)
    )

    print("\n--------------------------------")
    print("FROM:", from_junction)
    print("TO:", to_junction)
    print("SEGMENTS:", len(group))

    # First and last GIS segments in source order
    first_geom = group.geometry.iloc[0]
    last_geom = group.geometry.iloc[-1]

    print("First segment start:", first_geom.coords[0])
    print("First segment end:", first_geom.coords[-1])

    print("Last segment start:", last_geom.coords[0])
    print("Last segment end:", last_geom.coords[-1])


--------------------------------
FROM: Abhanpur Junction
TO: Dhamtari
SEGMENTS: 75
First segment start: (81.74318678117633, 21.041925084676397)
First segment end: (81.74574119997551, 21.053455399937917)
Last segment start: (81.7058387626903, 20.873455623581343)
Last segment end: (81.70581069033767, 20.873311295812094)

--------------------------------
FROM: Abhanpur Junction
TO: Kendri
SEGMENTS: 3
First segment start: (81.7504406061287, 21.07523158438136)
First segment end: (81.7393809660672, 21.106432799568534)
Last segment start: (81.75043426671772, 21.07504228513697)
Last segment end: (81.7504406061287, 21.07523158438136)

--------------------------------
FROM: Abhanpur Junction
TO: Rajim
SEGMENTS: 13
First segment start: (81.8333500691008, 20.97976333014656)
First segment end: (81.77748476235458, 21.01180902655741)
Last segment start: (81.74729863343813, 21.045679358670938)
Last segment end: (81.74721062548971, 21.045778909379905)

--------------------------------
FROM: Abhayapuri

In [174]:
import shapely

network_lines = network_geo[
    (network_geo["fromjunction"].notna()) &
    (network_geo["tojunction"].notna()) &
    (network_geo.geometry.geom_type == "LineString")
].copy()

print("Valid LineString segments:", len(network_lines))

# Extract first and last coordinate of every LineString
network_lines["start_geom"] = shapely.get_point(
    network_lines.geometry.array, 0
)

network_lines["end_geom"] = shapely.get_point(
    network_lines.geometry.array, -1
)

# Convert points to coordinates
start_coords = shapely.get_coordinates(
    network_lines["start_geom"].array
)

end_coords = shapely.get_coordinates(
    network_lines["end_geom"].array
)

network_lines["start_lon"] = start_coords[:, 0]
network_lines["start_lat"] = start_coords[:, 1]

network_lines["end_lon"] = end_coords[:, 0]
network_lines["end_lat"] = end_coords[:, 1]

print("Endpoint extraction completed.")

print(
    network_lines[
        [
            "fromjunction",
            "tojunction",
            "start_lon",
            "start_lat",
            "end_lon",
            "end_lat"
        ]
    ].head()
)

Valid LineString segments: 118439
Endpoint extraction completed.
           fromjunction        tojunction  start_lon  start_lat    end_lon  \
0  Tirunelveli Junction  Tenkasi Junction  77.323439   8.956365  77.376855   
1  Tirunelveli Junction  Tenkasi Junction  77.389645   8.766058  77.390178   
2  Tirunelveli Junction  Tenkasi Junction  77.376968   8.851557  77.382268   
3  Tirunelveli Junction  Tenkasi Junction  77.376855   8.851678  77.376968   
4  Tirunelveli Junction  Tenkasi Junction  77.382359   8.775492  77.385934   

    end_lat  
0  8.851678  
1  8.765121  
2  8.775694  
3  8.851557  
4  8.770386  


In [175]:
# Round coordinates to avoid tiny floating-point differences
network_lines["start_key"] = list(
    zip(
        network_lines["start_lon"].round(6),
        network_lines["start_lat"].round(6)
    )
)

network_lines["end_key"] = list(
    zip(
        network_lines["end_lon"].round(6),
        network_lines["end_lat"].round(6)
    )
)

# Create endpoint records
start_points = network_lines[
    ["fromjunction", "tojunction", "start_key"]
].copy()

start_points["endpoint_type"] = "start"

start_points = start_points.rename(
    columns={"start_key": "endpoint_key"}
)

end_points = network_lines[
    ["fromjunction", "tojunction", "end_key"]
].copy()

end_points["endpoint_type"] = "end"

end_points = end_points.rename(
    columns={"end_key": "endpoint_key"}
)

endpoint_records = pd.concat(
    [start_points, end_points],
    ignore_index=True
)

# Count how many segment endpoints meet at each coordinate
endpoint_degree = (
    endpoint_records
    .groupby("endpoint_key")
    .size()
    .reset_index(name="endpoint_degree")
)

print("Unique spatial endpoints:", len(endpoint_degree))

print("\nEndpoint degree distribution:")
print(
    endpoint_degree["endpoint_degree"]
    .value_counts()
    .sort_index()
)

Unique spatial endpoints: 117943

Endpoint degree distribution:
endpoint_degree
1       410
2    116192
3      1282
4        57
5         2
Name: count, dtype: int64


In [176]:
print("\nDegree 1 endpoints:")
print(
    (endpoint_degree["endpoint_degree"] == 1).sum()
)

print("\nDegree 2 endpoints:")
print(
    (endpoint_degree["endpoint_degree"] == 2).sum()
)

print("\nDegree >2 endpoints:")
print(
    (endpoint_degree["endpoint_degree"] > 2).sum()
)

print("\nMaximum endpoint degree:")
print(
    endpoint_degree["endpoint_degree"].max()
)


Degree 1 endpoints:
410

Degree 2 endpoints:
116192

Degree >2 endpoints:
1341

Maximum endpoint degree:
5


In [177]:
# ---------------------------------------------------------
# TASK 65F — Corridor spatial boundary analysis
# ---------------------------------------------------------

# Count endpoint degree for each actual segment endpoint
endpoint_degree_map = dict(
    zip(
        endpoint_degree["endpoint_key"],
        endpoint_degree["endpoint_degree"]
    )
)

# Attach degree to every segment endpoint
network_lines["start_degree"] = (
    network_lines["start_key"].map(endpoint_degree_map)
)

network_lines["end_degree"] = (
    network_lines["end_key"].map(endpoint_degree_map)
)

# A segment endpoint is a spatial boundary if its degree == 1
network_lines["start_is_boundary"] = (
    network_lines["start_degree"] == 1
)

network_lines["end_is_boundary"] = (
    network_lines["end_degree"] == 1
)

# ---------------------------------------------------------
# Summarize each corridor
# ---------------------------------------------------------

corridor_topology = (
    network_lines
    .groupby(["fromjunction", "tojunction"])
    .agg(
        segments=("geometry", "size"),

        boundary_start_count=(
            "start_is_boundary",
            "sum"
        ),

        boundary_end_count=(
            "end_is_boundary",
            "sum"
        ),

        min_start_degree=(
            "start_degree",
            "min"
        ),

        min_end_degree=(
            "end_degree",
            "min"
        ),

        max_start_degree=(
            "start_degree",
            "max"
        ),

        max_end_degree=(
            "end_degree",
            "max"
        )
    )
    .reset_index()
)

corridor_topology["total_boundary_endpoints"] = (
    corridor_topology["boundary_start_count"] +
    corridor_topology["boundary_end_count"]
)

print("Corridors:", len(corridor_topology))

print("\nTotal boundary endpoints by corridor:")
print(
    corridor_topology["total_boundary_endpoints"]
    .value_counts()
    .sort_index()
)

Corridors: 2305

Total boundary endpoints by corridor:
total_boundary_endpoints
0    1951
1     312
2      37
3       2
4       1
6       1
8       1
Name: count, dtype: int64


In [178]:
print("\nCorridors with exactly 2 spatial boundary endpoints:")
print(
    (
        corridor_topology["total_boundary_endpoints"] == 2
    ).sum()
)

print("\nCorridors with exactly 1 spatial boundary endpoint:")
print(
    (
        corridor_topology["total_boundary_endpoints"] == 1
    ).sum()
)

print("\nCorridors with 0 spatial boundary endpoints:")
print(
    (
        corridor_topology["total_boundary_endpoints"] == 0
    ).sum()
)

print("\nCorridors with >2 spatial boundary endpoints:")
print(
    (
        corridor_topology["total_boundary_endpoints"] > 2
    ).sum()
)


Corridors with exactly 2 spatial boundary endpoints:
37

Corridors with exactly 1 spatial boundary endpoint:
312

Corridors with 0 spatial boundary endpoints:
1951

Corridors with >2 spatial boundary endpoints:
5


In [179]:
problem_corridors = corridor_topology[
    corridor_topology["total_boundary_endpoints"] != 2
].copy()

print(
    problem_corridors[
        [
            "fromjunction",
            "tojunction",
            "segments",
            "boundary_start_count",
            "boundary_end_count",
            "total_boundary_endpoints",
            "max_start_degree",
            "max_end_degree"
        ]
    ].sort_values(
        ["total_boundary_endpoints", "segments"]
    ).to_string(index=False)
)

                           fromjunction                                   tojunction  segments  boundary_start_count  boundary_end_count  total_boundary_endpoints  max_start_degree  max_end_degree
                        Agasod Junction                       Bina Malkhedi Junction         1                     0                   0                         0                 3               3
                          Agastiyampall                      Tiruturaipundi Junction         1                     0                   0                         0                 3               2
                            Agra Cantt.                          Idgah Agra Junction         1                     0                   0                         0                 4               3
                                Agthori                                    Changsari         1                     0                   0                         0                 2               3
               

In [180]:
high_boundary_corridors = corridor_topology[
    corridor_topology["total_boundary_endpoints"] > 2
].copy()

print(
    high_boundary_corridors[
        [
            "fromjunction",
            "tojunction",
            "segments",
            "boundary_start_count",
            "boundary_end_count",
            "total_boundary_endpoints",
            "min_start_degree",
            "min_end_degree",
            "max_start_degree",
            "max_end_degree"
        ]
    ].sort_values("total_boundary_endpoints", ascending=False)
    .to_string(index=False)
)

           fromjunction      tojunction  segments  boundary_start_count  boundary_end_count  total_boundary_endpoints  min_start_degree  min_end_degree  max_start_degree  max_end_degree
New Jalpaiguri Junction      Darjeeling        39                     5                   3                         8                 1               1                 3               3
     Kachhpura Junction Itarsi Junction       471                     3                   3                         6                 1               1                 3               3
        Cta Chitradurga       Tolahunse         2                     2                   2                         4                 1               1                 1               1
                   Dhar            Tihi        42                     1                   2                         3                 1               1                 2               2
        Kollam Junction         Punalur        34                     

In [181]:
corridor_geometry = (
    network_lines
    .groupby(["fromjunction", "tojunction"])["geometry"]
    .agg(lambda x: shapely.union_all(x.values))
    .reset_index()
)

print("Corridor geometries:", len(corridor_geometry))
print(corridor_geometry.geometry.geom_type.value_counts())

Corridor geometries: 2305
MultiLineString    2016
LineString          289
Name: count, dtype: int64


In [182]:
corridor_geometry["geometry_type"] = (
    corridor_geometry.geometry.geom_type
)

corridor_geometry["geometry_length"] = (
    corridor_geometry.geometry.length
)

print(
    corridor_geometry[
        ["geometry_type", "geometry_length"]
    ].describe(include="all")
)

          geometry_type  geometry_length
count              2305      2305.000000
unique                2              NaN
top     MultiLineString              NaN
freq               2016              NaN
mean                NaN         0.292942
std                 NaN         0.361106
min                 NaN         0.000089
25%                 NaN         0.037215
50%                 NaN         0.165667
75%                 NaN         0.408143
max                 NaN         3.221163


In [183]:
from shapely.geometry import Point

def extract_endpoints(geom):
    if geom.geom_type == "LineString":
        return [
            Point(geom.coords[0]),
            Point(geom.coords[-1])
        ]

    elif geom.geom_type == "MultiLineString":
        endpoints = []

        for line in geom.geoms:
            endpoints.append(Point(line.coords[0]))
            endpoints.append(Point(line.coords[-1]))

        return endpoints

    return []

corridor_geometry["spatial_endpoints"] = (
    corridor_geometry.geometry.apply(extract_endpoints)
)

corridor_geometry["endpoint_count"] = (
    corridor_geometry["spatial_endpoints"].str.len()
)

print(
    corridor_geometry["endpoint_count"]
    .value_counts()
    .sort_index()
)

endpoint_count
2       289
4        81
6       166
8        68
10       91
       ... 
1286      1
1306      1
1362      1
1388      1
1422      1
Name: count, Length: 281, dtype: int64


In [184]:
print("\nGeometry types:")
print(corridor_geometry["geometry_type"].value_counts())

print("\nEndpoint count summary:")
print(corridor_geometry["endpoint_count"].describe())

print("\nCorridors with endpoint count != 2:")
print(
    (corridor_geometry["endpoint_count"] != 2).sum()
)


Geometry types:
geometry_type
MultiLineString    2016
LineString          289
Name: count, dtype: int64

Endpoint count summary:
count    2305.000000
mean      102.883297
std       171.070884
min         2.000000
25%         8.000000
50%        38.000000
75%       120.000000
max      1422.000000
Name: endpoint_count, dtype: float64

Corridors with endpoint count != 2:
2016


In [185]:
# Create endpoint records from every valid network segment

start_points = network_lines[
    [
        "fromjunction",
        "tojunction",
        "start_lon",
        "start_lat"
    ]
].copy()

start_points["endpoint_role"] = "start"

start_points = start_points.rename(
    columns={
        "start_lon": "longitude",
        "start_lat": "latitude"
    }
)


end_points = network_lines[
    [
        "fromjunction",
        "tojunction",
        "end_lon",
        "end_lat"
    ]
].copy()

end_points["endpoint_role"] = "end"

end_points = end_points.rename(
    columns={
        "end_lon": "longitude",
        "end_lat": "latitude"
    }
)


corridor_endpoint_records = pd.concat(
    [start_points, end_points],
    ignore_index=True
)

# Spatial coordinate key
corridor_endpoint_records["endpoint_key"] = (
    corridor_endpoint_records["longitude"].round(6).astype(str)
    + "_"
    + corridor_endpoint_records["latitude"].round(6).astype(str)
)

print("Endpoint records:", len(corridor_endpoint_records))
print(
    "Unique spatial endpoints:",
    corridor_endpoint_records["endpoint_key"].nunique()
)

Endpoint records: 236878
Unique spatial endpoints: 117943


In [186]:
corridor_endpoint_locations = (
    corridor_endpoint_records
    .groupby(
        ["fromjunction", "tojunction"]
    )
    .agg(
        unique_spatial_locations=(
            "endpoint_key",
            "nunique"
        ),
        endpoint_records=(
            "endpoint_key",
            "size"
        )
    )
    .reset_index()
)

print(
    corridor_endpoint_locations[
        "unique_spatial_locations"
    ].describe()
)

print("\nDistribution:")
print(
    corridor_endpoint_locations[
        "unique_spatial_locations"
    ]
    .value_counts()
    .sort_index()
)

count    2305.000000
mean       52.421692
std        85.552814
min         2.000000
25%         5.000000
50%        20.000000
75%        61.000000
max       712.000000
Name: unique_spatial_locations, dtype: float64

Distribution:
unique_spatial_locations
2      289
3       72
4      172
5       67
6       94
      ... 
645      1
656      1
682      1
695      1
712      1
Name: count, Length: 286, dtype: int64


In [187]:
clean_endpoint_corridors = (
    corridor_endpoint_locations[
        corridor_endpoint_locations["unique_spatial_locations"] <= 4
    ]
    .sort_values(
        ["unique_spatial_locations", "endpoint_records"]
    )
)

print(
    clean_endpoint_corridors.head(30).to_string(
        index=False
    )
)

               fromjunction                              tojunction  unique_spatial_locations  endpoint_records
            Agasod Junction                  Bina Malkhedi Junction                         2                 2
              Agastiyampall                 Tiruturaipundi Junction                         2                 2
                Agra Cantt.                     Idgah Agra Junction                         2                 2
                    Agthori                               Changsari                         2                 2
         Aishbagh Junction                     Lucknow Junction NER                         2                 2
                   Ajaygarh                                Barethia                         2                 2
        Alambagh West Cabin                     Lucknow Charbagh NR                         2                 2
                  Alirajpur                                Khandala                         2           

In [188]:
complex_endpoint_corridors = (
    corridor_endpoint_locations
    .sort_values(
        "unique_spatial_locations",
        ascending=False
    )
)

print(
    complex_endpoint_corridors.head(20).to_string(
        index=False
    )
)

        fromjunction          tojunction  unique_spatial_locations  endpoint_records
    Madgaon Junction              Thokur                       712              1422
Khurda Road Junction    Naupada Junction                       695              1388
Dharmavaram Junction     Pakala Junction                       682              1362
Himmatnagar Junction     Jai Samand Road                       656              1306
         Jammu Tawi             Baramula                       645              1286
   Udhna Chord Cabin    Jalgaon Junction                       612              1222
    Mudkhed Junction  Majri Bridge Cabin                       611              1220
 Peddapalli Junction  Nizamabad Junction                       585              1168
      Botad Junction   Sabarmati D Cabin                       576              1150
Yerraguntla Junction    Nandyal Junction                       572              1142
       Guna Junction    Gwalior Junction                       55

In [189]:
# Station master coordinates
station_coords = cleaned_stations[
    cleaned_stations["spatially_usable"]
].copy()

station_coords = station_coords[
    [
        "station_code",
        "station_name",
        "longitude",
        "latitude"
    ]
].copy()

# Junction names from the network
junction_names = pd.unique(
    pd.concat([
        network_lines["fromjunction"],
        network_lines["tojunction"]
    ]).dropna()
)

junction_df = pd.DataFrame({
    "junction_name": junction_names
})

# Basic normalization used previously
def normalize_name(x):
    if pd.isna(x):
        return ""
    
    x = str(x).strip().upper()
    x = x.replace(".", "")
    x = x.replace(",", "")
    x = " ".join(x.split())
    return x

station_coords["normalized_name"] = (
    station_coords["station_name"].apply(normalize_name)
)

junction_df["normalized_name"] = (
    junction_df["junction_name"].apply(normalize_name)
)

# Map normalized station name -> station coordinate
station_name_map = (
    station_coords
    .drop_duplicates("normalized_name")
    .set_index("normalized_name")
)

junction_known = junction_df[
    junction_df["normalized_name"].isin(
        station_name_map.index
    )
].copy()

junction_known["station_code"] = (
    junction_known["normalized_name"]
    .map(station_name_map["station_code"])
)

junction_known["station_name"] = (
    junction_known["normalized_name"]
    .map(station_name_map["station_name"])
)

junction_known["longitude"] = (
    junction_known["normalized_name"]
    .map(station_name_map["longitude"])
)

junction_known["latitude"] = (
    junction_known["normalized_name"]
    .map(station_name_map["latitude"])
)

print("Known junctions:", len(junction_known))
print(junction_known.head(20).to_string(index=False))

Known junctions: 727
    junction_name   normalized_name station_code      station_name  longitude  latitude
            Kalka             KALKA          KLK             Kalka  76.931759 30.837667
    Mettupalayam       METTUPALAYAM          MTP      Mettupalayam  76.934533 11.299729
     Mettupalayam      METTUPALAYAM          MTP      Mettupalayam  76.934533 11.299729
 Coimbatore North  COIMBATORE NORTH          CBF  COIMBATORE NORTH  76.954304 11.019937
          Rupamau           RUPAMAU          RUM           RUPAMAU  81.325346 26.230209
          Sanawad           SANAWAD          SWD           Sanawad  76.074567 22.175810
         Takkolam          TAKKOLAM          TKO          TAKKOLAM  79.677055 13.023783
    Hindu College     HINDU COLLEGE           HC     HINDU COLLEGE  80.074903 13.118538
  Chennai Central   CHENNAI CENTRAL          MAS   CHENNAI CENTRAL  80.274856 13.084761
     Washermanpet      WASHERMANPET          WST      WASHERMANPET  80.283020 13.109223
 Vyasarpadi

In [190]:
from shapely.geometry import Point

known_points = gpd.GeoDataFrame(
    junction_known.copy(),
    geometry=[
        Point(xy)
        for xy in zip(
            junction_known["longitude"],
            junction_known["latitude"]
        )
    ],
    crs="EPSG:4326"
)

# Project to a metric CRS for distance calculations
known_points_m = known_points.to_crs("EPSG:3857")
network_m = network_lines.to_crs("EPSG:3857")

# Network geometry as a single spatial index source
network_union = shapely.union_all(
    network_m.geometry.array
)

known_points_m["distance_to_network_m"] = (
    known_points_m.geometry.distance(network_union)
)

print(
    known_points_m[
        "distance_to_network_m"
    ].describe()
)

count     727.000000
mean       86.696630
std       478.907271
min         0.005712
25%        13.423685
50%        30.611617
75%        54.133331
max      9343.638124
Name: distance_to_network_m, dtype: float64


In [191]:
print("\nDistance <= 100 m:")
print(
    (known_points_m["distance_to_network_m"] <= 100).sum()
)

print("\nDistance <= 500 m:")
print(
    (known_points_m["distance_to_network_m"] <= 500).sum()
)

print("\nDistance <= 1 km:")
print(
    (known_points_m["distance_to_network_m"] <= 1000).sum()
)

print("\nDistance <= 2 km:")
print(
    (known_points_m["distance_to_network_m"] <= 2000).sum()
)

print("\nDistance > 2 km:")
print(
    (known_points_m["distance_to_network_m"] > 2000).sum()
)


Distance <= 100 m:
696

Distance <= 500 m:
713

Distance <= 1 km:
717

Distance <= 2 km:
720

Distance > 2 km:
7


In [192]:
from shapely.geometry import Point

known_points = gpd.GeoDataFrame(
    junction_known.copy(),
    geometry=[
        Point(xy)
        for xy in zip(
            junction_known["longitude"],
            junction_known["latitude"]
        )
    ],
    crs="EPSG:4326"
)

known_points_m = known_points.to_crs("EPSG:3857")
network_m = network_lines.to_crs("EPSG:3857")

network_union = shapely.union_all(
    network_m.geometry.array
)

known_points_m["distance_to_network_m"] = (
    known_points_m.geometry.distance(network_union)
)

print(
    known_points_m["distance_to_network_m"].describe()
)

print("\nDistance <= 100 m:")
print(
    (known_points_m["distance_to_network_m"] <= 100).sum()
)

print("\nDistance <= 500 m:")
print(
    (known_points_m["distance_to_network_m"] <= 500).sum()
)

print("\nDistance <= 1 km:")
print(
    (known_points_m["distance_to_network_m"] <= 1000).sum()
)

print("\nDistance <= 2 km:")
print(
    (known_points_m["distance_to_network_m"] <= 2000).sum()
)

print("\nDistance > 2 km:")
print(
    (known_points_m["distance_to_network_m"] > 2000).sum()
)

count     727.000000
mean       86.696630
std       478.907271
min         0.005712
25%        13.423685
50%        30.611617
75%        54.133331
max      9343.638124
Name: distance_to_network_m, dtype: float64

Distance <= 100 m:
696

Distance <= 500 m:
713

Distance <= 1 km:
717

Distance <= 2 km:
720

Distance > 2 km:
7


In [193]:
print(
    known_points_m[
        [
            "junction_name",
            "station_code",
            "station_name",
            "distance_to_network_m"
        ]
    ]
    .sort_values(
        "distance_to_network_m",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

      junction_name station_code        station_name  distance_to_network_m
            Kathana         KTNA             Kathana            9343.638124
       Dullabcherra         DLCR        Dullabcherra            5148.700637
           Bamanhat          BXT            Bamanhat            3904.365326
         Bangriposi          BGY          Bangriposi            3733.626099
           Manuguru         MUGR            Manuguru            2866.576774
             Ramtek          RTK              Ramtek            2282.088885
             Kendri         KDRI              Kendri            2053.928443
            Gawnaha          GAH             Gawnaha            1662.368494
             Jhajha          JAJ              JHAJHA            1577.038022
      Kharwa Chanda         KRCD       KHARWA CHANDA            1398.765897
            Tantpur          TPO             Tantpur             874.175535
            Chandod          CDD             Chandod             837.251418
Vadtal Swami

In [194]:
unmatched_junctions = junction_df[
    ~junction_df["normalized_name"].isin(
        station_name_map.index
    )
].copy()

print("Unmatched junctions:", len(unmatched_junctions))
print(
    unmatched_junctions.head(30)
    .to_string(index=False)
)

Unmatched junctions: 1339
             junction_name            normalized_name
      Tirunelveli Junction       TIRUNELVELI JUNCTION
Thiruvananthapuram Central THIRUVANANTHAPURAM CENTRAL
      Virudunagar Junction       VIRUDUNAGAR JUNCTION
                   Edamann                    EDAMANN
           Bhagavathipuram            BHAGAVATHIPURAM
            Erode Junction             ERODE JUNCTION
         Pollachi Junction          POLLACHI JUNCTION
          Madurai Junction           MADURAI JUNCTION
       Coimbatore Junction        COIMBATORE JUNCTION
           Irugur Junction            IRUGUR JUNCTION
         Palakkad Junction          PALAKKAD JUNCTION
     Chengalpattu Junction      CHENGALPATTU JUNCTION
        Arakkonam Junction         ARAKKONAM JUNCTION
            Gudur Junction             GUDUR JUNCTION
                Anna Nagar                 ANNA NAGAR
     Basin Bridge Junction      BASIN BRIDGE JUNCTION
              Thomas Mount               THOMAS MOUNT
  

In [195]:
junction_from = (
    network_lines[
        ["fromjunction", "geometry"]
    ]
    .rename(
        columns={"fromjunction": "junction_name"}
    )
)

junction_to = (
    network_lines[
        ["tojunction", "geometry"]
    ]
    .rename(
        columns={"tojunction": "junction_name"}
    )
)

junction_segments = pd.concat(
    [junction_from, junction_to],
    ignore_index=True
)

junction_geometry = (
    junction_segments
    .groupby("junction_name")["geometry"]
    .agg(lambda x: shapely.union_all(x.values))
    .reset_index()
)

junction_geometry = gpd.GeoDataFrame(
    junction_geometry,
    geometry="geometry",
    crs=network_lines.crs
)

print("Junction geometries:", len(junction_geometry))
print(
    junction_geometry.geometry.geom_type.value_counts()
)

Junction geometries: 2066
MultiLineString    1923
LineString          143
Name: count, dtype: int64


In [196]:
junction_geometry["representative_point"] = (
    junction_geometry.geometry.representative_point()
)

junction_points = junction_geometry[
    ["junction_name", "representative_point"]
].copy()

junction_points = gpd.GeoDataFrame(
    junction_points,
    geometry="representative_point",
    crs=network_lines.crs
)

junction_points["normalized_name"] = (
    junction_points["junction_name"]
    .apply(normalize_name)
)

unmatched_junction_points = junction_points[
    junction_points["junction_name"].isin(
        unmatched_junctions["junction_name"]
    )
].copy()

print(
    "Unmatched junction spatial points:",
    len(unmatched_junction_points)
)

Unmatched junction spatial points: 1339


In [197]:
station_points_m = gpd.GeoDataFrame(
    station_coords.copy(),
    geometry=[
        Point(xy)
        for xy in zip(
            station_coords["longitude"],
            station_coords["latitude"]
        )
    ],
    crs="EPSG:4326"
).to_crs("EPSG:3857")

junction_points_m = (
    unmatched_junction_points
    .to_crs("EPSG:3857")
)

print("Stations:", len(station_points_m))
print(
    "Unmatched junction points:",
    len(junction_points_m)
)

Stations: 8697
Unmatched junction points: 1339


In [198]:
candidate_matches = gpd.sjoin_nearest(
    junction_points_m,
    station_points_m,
    how="left",
    distance_col="distance_m"
)

print("Candidate rows:", len(candidate_matches))

print(
    candidate_matches[
        [
            "junction_name",
            "station_code",
            "station_name",
            "distance_m"
        ]
    ]
    .head(30)
    .to_string(index=False)
)

Candidate rows: 1339
         junction_name station_code         station_name   distance_m
            Abhayapuri         GLPT        GOALPARA TOWN  1894.992344
           Abhayapuri           MZQ        MAJGAON ASSAM  2851.104973
       Abohar Junction          BFE         BAHAWAL BASI   288.334906
     Achhnera Junction         KHSN   KHERA SANDHAN HALT   960.886195
       Adipur Junction           AI               ADIPUR 22052.779685
         Adra Junction          IBL             INDRABIL  1196.475862
          Afk Angamali          AFK             ANGAMALI  3675.142796
       Agasod Junction          KOA              KARONDA  2087.360043
         Agastiyampall         VLKN           Velankanni 29628.397472
        Agastiyampalli         VLKN           Velankanni 40142.795196
           Ahiran Halt         GALE               GANKAR  2837.599093
     Ahmadpur Junction          BHP BOLPUR SHANTINIKETAN  1307.862567
    Ahmedabad Junction          KKF             KANKARIA  1055.424040

In [199]:
print(
    candidate_matches["distance_m"].describe()
)

print("\n<= 100 m:")
print(
    (candidate_matches["distance_m"] <= 100).sum()
)

print("\n<= 500 m:")
print(
    (candidate_matches["distance_m"] <= 500).sum()
)

print("\n<= 1 km:")
print(
    (candidate_matches["distance_m"] <= 1000).sum()
)

print("\n<= 2 km:")
print(
    (candidate_matches["distance_m"] <= 2000).sum()
)

count     1339.000000
mean      6139.891679
std      10482.812330
min         20.631652
25%       1044.835226
50%       2302.179372
75%       5338.744779
max      78749.021261
Name: distance_m, dtype: float64

<= 100 m:
26

<= 500 m:
167

<= 1 km:
321

<= 2 km:
581


In [200]:
# Create endpoint records for each segment within its corridor

local_start = network_lines[
    [
        "fromjunction",
        "tojunction",
        "start_key"
    ]
].copy()

local_start = local_start.rename(
    columns={"start_key": "endpoint_key"}
)

local_end = network_lines[
    [
        "fromjunction",
        "tojunction",
        "end_key"
    ]
].copy()

local_end = local_end.rename(
    columns={"end_key": "endpoint_key"}
)

local_endpoints = pd.concat(
    [local_start, local_end],
    ignore_index=True
)

# Count how many times each spatial endpoint occurs
# WITHIN EACH labelled corridor
local_endpoint_counts = (
    local_endpoints
    .groupby(
        [
            "fromjunction",
            "tojunction",
            "endpoint_key"
        ]
    )
    .size()
    .reset_index(
        name="local_endpoint_count"
    )
)

print(
    local_endpoint_counts[
        "local_endpoint_count"
    ].value_counts()
    .sort_index()
)

local_endpoint_count
1      4827
2    115965
3        39
4         1
Name: count, dtype: int64


In [201]:
local_boundary_points = local_endpoint_counts[
    local_endpoint_counts["local_endpoint_count"] == 1
].copy()

print(
    "Local boundary endpoint records:",
    len(local_boundary_points)
)

print(
    "Corridors represented:",
    local_boundary_points[
        [
            "fromjunction",
            "tojunction"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

Local boundary endpoint records: 4827
Corridors represented: 2304


In [202]:
local_boundary_counts = (
    local_boundary_points
    .groupby(
        [
            "fromjunction",
            "tojunction"
        ]
    )
    .size()
    .reset_index(
        name="local_boundary_count"
    )
)

print(
    local_boundary_counts[
        "local_boundary_count"
    ]
    .value_counts()
    .sort_index()
)

local_boundary_count
1        2
2     2210
3       15
4       62
5        1
6        8
8        2
10       2
11       1
12       1
Name: count, dtype: int64


In [203]:
print(
    "\nExactly 2 local boundary points:",
    (
        local_boundary_counts[
            "local_boundary_count"
        ] == 2
    ).sum()
)

print(
    "Exactly 1:",
    (
        local_boundary_counts[
            "local_boundary_count"
        ] == 1
    ).sum()
)

print(
    "More than 2:",
    (
        local_boundary_counts[
            "local_boundary_count"
        ] > 2
    ).sum()
)


Exactly 2 local boundary points: 2210
Exactly 1: 2
More than 2: 92


In [204]:
endpoint_coordinate_map = pd.concat([
    network_lines[
        ["start_key", "start_lon", "start_lat"]
    ].rename(
        columns={
            "start_key": "endpoint_key",
            "start_lon": "longitude",
            "start_lat": "latitude"
        }
    ),
    network_lines[
        ["end_key", "end_lon", "end_lat"]
    ].rename(
        columns={
            "end_key": "endpoint_key",
            "end_lon": "longitude",
            "end_lat": "latitude"
        }
    )
]).drop_duplicates("endpoint_key")

print(
    "Unique endpoint coordinates:",
    len(endpoint_coordinate_map)
)

Unique endpoint coordinates: 117943


In [205]:
local_boundary_points_geo = (
    local_boundary_points
    .merge(
        endpoint_coordinate_map,
        on="endpoint_key",
        how="left"
    )
)

print(
    local_boundary_points_geo[
        [
            "fromjunction",
            "tojunction",
            "endpoint_key",
            "longitude",
            "latitude"
        ]
    ].head(20).to_string(index=False)
)

     fromjunction             tojunction           endpoint_key  longitude  latitude
Abhanpur Junction               Dhamtari (81.550303, 20.714625)  81.550303 20.714625
Abhanpur Junction               Dhamtari (81.745741, 21.053455)  81.745741 21.053455
Abhanpur Junction                 Kendri (81.739381, 21.106433)  81.739381 21.106433
Abhanpur Junction                 Kendri (81.745741, 21.053455)  81.745741 21.053455
Abhanpur Junction                  Rajim (81.745741, 21.053455)  81.745741 21.053455
Abhanpur Junction                  Rajim (81.869378, 20.969768)  81.869378 20.969768
       Abhayapuri       Dudhnai Junction (90.642588, 26.356539)  90.642588 26.356539
       Abhayapuri       Dudhnai Junction  (90.76207, 26.001673)  90.762070 26.001673
  Abohar Junction      Bathinda Junction   (74.215961, 30.1406)  74.215961 30.140600
  Abohar Junction      Bathinda Junction (74.931767, 30.218941)  74.931767 30.218941
  Abohar Junction       Fazilka Junction (74.033228, 30.409104)  

In [206]:
clean_corridor_keys = (
    local_boundary_counts[
        local_boundary_counts["local_boundary_count"] == 2
    ][
        [
            "fromjunction",
            "tojunction"
        ]
    ]
)

clean_corridor_endpoints = (
    local_boundary_points_geo
    .merge(
        clean_corridor_keys,
        on=["fromjunction", "tojunction"],
        how="inner"
    )
)

print(
    "Clean corridor endpoint records:",
    len(clean_corridor_endpoints)
)

print(
    "Unique clean corridors:",
    clean_corridor_endpoints[
        [
            "fromjunction",
            "tojunction"
        ]
    ].drop_duplicates().shape[0]
)

Clean corridor endpoint records: 4420
Unique clean corridors: 2210


In [207]:
clean_corridor_endpoints["endpoint_number"] = (
    clean_corridor_endpoints
    .groupby(
        ["fromjunction", "tojunction"]
    )
    .cumcount() + 1
)

clean_corridor_endpoint_table = (
    clean_corridor_endpoints
    .pivot(
        index=["fromjunction", "tojunction"],
        columns="endpoint_number",
        values=["endpoint_key", "longitude", "latitude"]
    )
    .reset_index()
)

clean_corridor_endpoint_table.columns = [
    "_".join(
        [str(x) for x in col if str(x) != "nan"]
    )
    for col in clean_corridor_endpoint_table.columns
]

print(
    clean_corridor_endpoint_table.head(20)
    .to_string(index=False)
)

    fromjunction_              tojunction_         endpoint_key_1         endpoint_key_2 longitude_1 longitude_2 latitude_1 latitude_2
Abhanpur Junction                 Dhamtari (81.550303, 20.714625) (81.745741, 21.053455)   81.550303   81.745741  20.714625  21.053455
Abhanpur Junction                   Kendri (81.739381, 21.106433) (81.745741, 21.053455)   81.739381   81.745741  21.106433  21.053455
Abhanpur Junction                    Rajim (81.745741, 21.053455) (81.869378, 20.969768)   81.745741   81.869378  21.053455  20.969768
       Abhayapuri         Dudhnai Junction (90.642588, 26.356539)  (90.76207, 26.001673)   90.642588    90.76207  26.356539  26.001673
  Abohar Junction        Bathinda Junction   (74.215961, 30.1406) (74.931767, 30.218941)   74.215961   74.931767    30.1406  30.218941
  Abohar Junction         Fazilka Junction (74.033228, 30.409104)   (74.215961, 30.1406)   74.033228   74.215961  30.409104    30.1406
         Achalpur   Murtizapur Junction NG (77.359764, 

In [208]:
station_lookup = (
    station_coords[
        [
            "station_code",
            "station_name",
            "normalized_name",
            "longitude",
            "latitude"
        ]
    ]
    .drop_duplicates("normalized_name")
    .set_index("normalized_name")
)

print("Unique station names:", len(station_lookup))

Unique station names: 8620


In [209]:
validation_corridors = clean_corridor_endpoint_table.copy()

validation_corridors["from_normalized"] = (
    validation_corridors["fromjunction_"]
    .apply(normalize_name)
)

validation_corridors["to_normalized"] = (
    validation_corridors["tojunction_"]
    .apply(normalize_name)
)

validation_corridors["from_known"] = (
    validation_corridors["from_normalized"]
    .isin(station_lookup.index)
)

validation_corridors["to_known"] = (
    validation_corridors["to_normalized"]
    .isin(station_lookup.index)
)

both_known = validation_corridors[
    validation_corridors["from_known"] &
    validation_corridors["to_known"]
].copy()

print(
    "Clean corridors:",
    len(validation_corridors)
)

print(
    "Both endpoints known by name:",
    len(both_known)
)
# [MINIMAL FIX: Map expected station coordinates to the dataframe]
both_known['from_lon'] = both_known['from_normalized'].map(station_lookup['longitude'])
both_known['from_lat'] = both_known['from_normalized'].map(station_lookup['latitude'])
both_known['to_lon'] = both_known['to_normalized'].map(station_lookup['longitude'])
both_known['to_lat'] = both_known['to_normalized'].map(station_lookup['latitude'])


Clean corridors: 2210
Both endpoints known by name: 270


In [210]:
# Force all coordinate columns to numeric
distance_cols = [
    "from_lon",
    "from_lat",
    "to_lon",
    "to_lat",
    "longitude_1",
    "latitude_1",
    "longitude_2",
    "latitude_2"
]

for col in distance_cols:
    both_known[col] = pd.to_numeric(
        both_known[col],
        errors="coerce"
    )


def haversine_series(lon1, lat1, lon2, lat2):
    lon1 = np.radians(
        np.asarray(lon1, dtype=float)
    )
    lat1 = np.radians(
        np.asarray(lat1, dtype=float)
    )
    lon2 = np.radians(
        np.asarray(lon2, dtype=float)
    )
    lat2 = np.radians(
        np.asarray(lat2, dtype=float)
    )

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2
        +
        np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 6371.0088 * 2 * np.arcsin(
        np.sqrt(a)
    )


# Assignment A
both_known["assignment_a_km"] = (
    haversine_series(
        both_known["from_lon"],
        both_known["from_lat"],
        both_known["longitude_1"],
        both_known["latitude_1"]
    )
    +
    haversine_series(
        both_known["to_lon"],
        both_known["to_lat"],
        both_known["longitude_2"],
        both_known["latitude_2"]
    )
)


# Assignment B
both_known["assignment_b_km"] = (
    haversine_series(
        both_known["from_lon"],
        both_known["from_lat"],
        both_known["longitude_2"],
        both_known["latitude_2"]
    )
    +
    haversine_series(
        both_known["to_lon"],
        both_known["to_lat"],
        both_known["longitude_1"],
        both_known["latitude_1"]
    )
)


both_known["best_assignment"] = np.where(
    both_known["assignment_a_km"]
    <= both_known["assignment_b_km"],
    "A",
    "B"
)

both_known["best_total_km"] = (
    both_known[
        ["assignment_a_km", "assignment_b_km"]
    ].min(axis=1)
)

both_known["assignment_gap_km"] = (
    both_known[
        ["assignment_a_km", "assignment_b_km"]
    ].max(axis=1)
    -
    both_known["best_total_km"]
)

---

> **Status: Haversine Validation Section — Implemented but NOT Executed**
>
> The cells below define and apply a haversine-based coordinate-distance validation
> between the corridor endpoint coordinates (extracted from `Railway_Network_.geojson`)
> and the station master coordinates (`cleaned_stations.csv`).
>
> **The method is correctly implemented** based on the matching context above (270 corridors
> with both endpoints name-matched to the station master).
>
> **These cells were not executed in this notebook version.** No threshold results
> (e.g., ‘both endpoints ≤ 1 km’) have been verified from this notebook.
>
> The graph-design teammate should treat this section as a **starting point for future
> spatial validation**, not as a completed analysis.

---

In [211]:
print(
    both_known[
        [
            "assignment_a_km",
            "assignment_b_km",
            "best_total_km",
            "assignment_gap_km"
        ]
    ].describe()
)

print("\nAssignment A:")
print(
    (both_known["best_assignment"] == "A").sum()
)

print("\nAssignment B:")
print(
    (both_known["best_assignment"] == "B").sum()
)

print("\nBest total distance <= 0.1 km:")
print(
    (both_known["best_total_km"] <= 0.1).sum()
)

print("\nBest total distance <= 0.5 km:")
print(
    (both_known["best_total_km"] <= 0.5).sum()
)

print("\nBest total distance <= 1 km:")
print(
    (both_known["best_total_km"] <= 1).sum()
)

print("\nBest total distance <= 2 km:")
print(
    (both_known["best_total_km"] <= 2).sum()
)

       assignment_a_km  assignment_b_km  best_total_km  assignment_gap_km
count       270.000000       270.000000     270.000000         270.000000
mean         25.210624        37.276875       6.600999          49.285501
std          84.394786        76.534970      65.311126          57.482656
min           0.025469         0.048251       0.025469           0.008820
25%           0.459655         2.440907       0.320135          13.804407
50%           1.933339        17.479608       0.850288          31.100659
75%          14.839345        51.257563       2.157688          63.030943
max        1126.075435      1068.767809    1068.767809         323.986002

Assignment A:
184

Assignment B:
86

Best total distance <= 0.1 km:
10

Best total distance <= 0.5 km:
92

Best total distance <= 1 km:
148

Best total distance <= 2 km:
201


In [212]:
print(
    both_known[
        [
            "fromjunction_",
            "tojunction_",
            "assignment_a_km",
            "assignment_b_km",
            "best_assignment",
            "best_total_km",
            "assignment_gap_km"
        ]
    ]
    .sort_values("best_total_km")
    .head(30)
    .to_string(index=False)
)

   fromjunction_        tojunction_  assignment_a_km  assignment_b_km best_assignment  best_total_km  assignment_gap_km
         Rupamau               Jais         0.025469        39.304779               A       0.025469          39.279310
            Jais             Amethi         0.040875        62.209042               A       0.040875          62.168167
         Manabar         Maligura\n        27.952014         0.048251               B       0.048251          27.903762
      Malkapuram        Betamcherla         0.051685        42.697908               A       0.051685          42.646224
       Nichitpur          Tetulmari         0.055054        11.612664               A       0.055054          11.557610
        Barkhera              Budni         0.076512        29.642214               A       0.076512          29.565702
           Tomka          Baghuapal         0.083789        16.502162               A       0.083789          16.418373
    Darshannagar        GoshainGanj     

In [213]:
good_2km = both_known[
    both_known["best_total_km"] <= 2
].copy()

print("Good <= 2 km:", len(good_2km))

print(
    good_2km[
        [
            "fromjunction_",
            "tojunction_",
            "best_assignment",
            "best_total_km",
            "assignment_gap_km"
        ]
    ]
    .sort_values("best_total_km")
    .head(50)
    .to_string(index=False)
)

Good <= 2 km: 201
             fromjunction_          tojunction_ best_assignment  best_total_km  assignment_gap_km
                   Rupamau                 Jais               A       0.025469          39.279310
                      Jais               Amethi               A       0.040875          62.168167
                   Manabar           Maligura\n               B       0.048251          27.903762
                Malkapuram          Betamcherla               A       0.051685          42.646224
                 Nichitpur            Tetulmari               A       0.055054          11.557610
                  Barkhera                Budni               A       0.076512          29.565702
                     Tomka            Baghuapal               A       0.083789          16.418373
              Darshannagar          GoshainGanj               A       0.086613          53.491115
              Jarandeshwar               Satara               B       0.087039          14.024978
  

In [214]:
bad_2km = both_known[
    both_known["best_total_km"] > 2
].copy()

print("Bad > 2 km:", len(bad_2km))

print(
    bad_2km[
        [
            "fromjunction_",
            "tojunction_",
            "assignment_a_km",
            "assignment_b_km",
            "best_assignment",
            "best_total_km",
            "assignment_gap_km"
        ]
    ]
    .sort_values("best_total_km")
    .head(50)
    .to_string(index=False)
)

Bad > 2 km: 69
        fromjunction_             tojunction_  assignment_a_km  assignment_b_km best_assignment  best_total_km  assignment_gap_km
      Talcher Thermal            Talcher Road         3.764469         2.157013               B       2.157013           1.607457
            Vyasnagar          New West Cabin         2.157914         7.338331               A       2.157914           5.180417
    Gainsari Junction               Pachperwa         2.179535        20.249193               A       2.179535          18.069658
           Sagadapata             Chilikidara        19.160009         2.190000               B       2.190000          16.970009
               Gatora             Latia Cabin         2.198268        37.403515               A       2.198268          35.205247
     Nanpara Junction          Nepalganj Road         2.206111        37.330694               A       2.206111          35.124583
              Gawnaha           Bhikhna Thori         2.210028        22.04

In [215]:
print(
    bad_2km[
        [
            "fromjunction_",
            "tojunction_",
            "assignment_a_km",
            "assignment_b_km",
            "best_assignment",
            "best_total_km",
            "assignment_gap_km"
        ]
    ]
    .sort_values(
        "best_total_km",
        ascending=False
    )
    .head(30)
    .to_string(index=False)
)

        fromjunction_          tojunction_  assignment_a_km  assignment_b_km best_assignment  best_total_km  assignment_gap_km
             Algapur            Ramnathpur      1126.075435      1068.767809               B    1068.767809          57.307626
           Gevra Road          Pendra Road       103.166363        77.955534               B      77.955534          25.210829
              Dalgaon       New Jalpaiguri        68.972747        69.251725               A      68.972747           0.278977
           Badampahar   Aunlajori Junction        33.630649        47.909623               A      33.630649          14.278974
              Murarai Bonidanga Link Cabin        31.175127        55.022936               A      31.175127          23.847809
             Kasganj                  Etah        27.673193        27.231359               B      27.231359           0.441834
             Salagaon           Haridaspur        25.730046        27.487346               A      25.730046    

In [216]:
both_known["from_best_distance_km"] = np.where(
    both_known["best_assignment"] == "A",
    haversine_series(
        both_known["from_lon"],
        both_known["from_lat"],
        both_known["longitude_1"],
        both_known["latitude_1"]
    ),
    haversine_series(
        both_known["from_lon"],
        both_known["from_lat"],
        both_known["longitude_2"],
        both_known["latitude_2"]
    )
)

both_known["to_best_distance_km"] = np.where(
    both_known["best_assignment"] == "A",
    haversine_series(
        both_known["to_lon"],
        both_known["to_lat"],
        both_known["longitude_2"],
        both_known["latitude_2"]
    ),
    haversine_series(
        both_known["to_lon"],
        both_known["to_lat"],
        both_known["longitude_1"],
        both_known["latitude_1"]
    )
)

print(
    both_known[
        [
            "from_best_distance_km",
            "to_best_distance_km"
        ]
    ].describe()
)

       from_best_distance_km  to_best_distance_km
count             270.000000           270.000000
mean                1.284709             5.316291
std                 4.856358            65.176404
min                 0.007488             0.009167
25%                 0.087656             0.084025
50%                 0.327956             0.288303
75%                 0.936315             0.813411
max                67.982235          1068.408689


In [217]:
def distance_class(d):
    if d <= 0.1:
        return "<=100m"
    elif d <= 0.5:
        return "<=500m"
    elif d <= 1:
        return "<=1km"
    elif d <= 2:
        return "<=2km"
    else:
        return ">2km"

both_known["from_distance_class"] = (
    both_known["from_best_distance_km"]
    .apply(distance_class)
)

both_known["to_distance_class"] = (
    both_known["to_best_distance_km"]
    .apply(distance_class)
)

print("FROM endpoint:")
print(
    both_known["from_distance_class"]
    .value_counts()
)

print("\nTO endpoint:")
print(
    both_known["to_distance_class"]
    .value_counts()
)

FROM endpoint:
from_distance_class
<=500m    90
<=100m    71
<=1km     48
<=2km     31
>2km      30
Name: count, dtype: int64

TO endpoint:
to_distance_class
<=500m    91
<=100m    83
<=1km     41
>2km      28
<=2km     27
Name: count, dtype: int64


In [218]:
for threshold in [0.1, 0.5, 1, 2, 5]:
    
    count = (
        (both_known["from_best_distance_km"] <= threshold)
        &
        (both_known["to_best_distance_km"] <= threshold)
    ).sum()
    
    print(
        f"Both endpoints <= {threshold} km:",
        count
    )

Both endpoints <= 0.1 km: 31
Both endpoints <= 0.5 km: 114
Both endpoints <= 1 km: 172
Both endpoints <= 2 km: 219
Both endpoints <= 5 km: 249
